In [ ]:
import os
TABLES_DIR = '/rds/homes/j/jxt554/tables'

files = [
    'cd_limma_M3.csv',
    'uc_limma_M3.csv',
    'ora_cd_C1_up_Hallmark.csv',
    'ora_uc_C1_up_Hallmark.csv',
    'cd_rf_importance.csv',
    'uc_rf_importance.csv',
    'ppi_UC_C1_top50.csv',
]

for f in files:
    path = f'{TABLES_DIR}/{f}'
    exists = os.path.exists(path)
    print(f'{"✅" if exists else "❌"} {f}')

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import matplotlib as mpl
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')
import os
import umap

PROJECT_DIR  = '/rds/homes/j/jxt554'
DATA_DIR     = f'{PROJECT_DIR}/data'
FIGURES_DIR  = f'{PROJECT_DIR}/figures'
TABLES_DIR   = f'{PROJECT_DIR}/tables'
SEED         = 42

# ── Palette ───────────────────────────────────────
C1_COL   = '#2E7D5B'
C2_COL   = '#E24B4A'
CD_COL   = '#1B3A6B'
UC_COL   = '#C96A1F'

# ── Load all data ─────────────────────────────────
print('Loading data...')

# UKB CD
X_cd    = np.load(f'{DATA_DIR}/X_cd_int.npy')
meta_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
labels_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')

# UKB UC
X_uc    = np.load(f'{DATA_DIR}/X_uc_int.npy')
meta_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')
labels_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

# DE results
de_cd = pd.read_csv(
    f'{TABLES_DIR}/cd_limma_M3.csv')
de_uc = pd.read_csv(
    f'{TABLES_DIR}/uc_limma_M3.csv')

# ORA
ora_cd = pd.read_csv(
    f'{TABLES_DIR}/ora_cd_C1_up_Hallmark.csv')
ora_uc = pd.read_csv(
    f'{TABLES_DIR}/ora_uc_C1_up_Hallmark.csv')

# RF
rf_cd = pd.read_csv(
    f'{TABLES_DIR}/cd_rf_importance.csv')
rf_uc = pd.read_csv(
    f'{TABLES_DIR}/uc_rf_importance.csv')

# PPI
ppi_uc = pd.read_csv(
    f'{TABLES_DIR}/ppi_UC_C1_top50.csv')

# Protein cols
prots = pd.read_csv(
    f'{DATA_DIR}/protein_cols_varfiltered.csv',
    header=None)[0].tolist()

print('✓ All data loaded')
print(f'CD: {X_cd.shape} labels={len(np.unique(labels_cd))}')
print(f'UC: {X_uc.shape} labels={len(np.unique(labels_uc))}')

In [ ]:
# ════════════════════════════════════════════════
# SLIDE 7 — UMAP Clustering (CD + UC)
# ════════════════════════════════════════════════
print('\nSlide 7: UMAP clustering...')

def get_umap(X, labels, seed=42):
    red = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        n_components=2,
        random_state=seed)
    emb = red.fit_transform(X)
    return emb

emb_cd = get_umap(X_cd, labels_cd)
emb_uc = get_umap(X_uc, labels_uc)

fig, axes = plt.subplots(
    1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')

for ax, emb, labels, cohort, n_c1, n_c2, sil, stab in [
    (axes[0], emb_cd, labels_cd,
     "Crohn's Disease",
     82, 133, 0.846, 0.893),
    (axes[1], emb_uc, labels_uc,
     "Ulcerative Colitis",
     299, 131, 0.777, 0.844)]:

    ax.set_facecolor('#F8F9FA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])

    for c, col, lab, n in [
        (0, C1_COL,
         f'C1 — Hyperinflammatory (n={n_c1})',
         n_c1),
        (1, C2_COL,
         f'C2 — Quiescent (n={n_c2})',
         n_c2)]:
        m = labels == c
        ax.scatter(
            emb[m, 0], emb[m, 1],
            c=col, s=45, alpha=0.85,
            linewidths=0.4,
            edgecolors='white',
            label=lab, zorder=3)

    ax.set_xlabel('UMAP 1', fontsize=13)
    ax.set_ylabel('UMAP 2', fontsize=13)
    ax.set_title(
        cohort,
        fontsize=16, fontweight='700',
        color='#1B3A6B', pad=12)

    ax.text(
        0.98, 0.98,
        f'k = 2\nSilhouette = {sil}\nStability = {stab}\nUnstable = 0',
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=11,
        fontfamily='monospace',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            edgecolor='#DDD',
            alpha=0.95))

    ax.legend(
        fontsize=11,
        framealpha=0.95,
        loc='lower left',
        markerscale=1.3)

plt.suptitle(
    'Two distinct proteomic subtypes '
    'identified in both CD and UC',
    fontsize=16, fontweight='800',
    color='#1B3A6B', y=1.02)

plt.tight_layout()
fname = f'{FIGURES_DIR}/slide7_clustering.png'
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'✓ slide7_clustering.png')
display(Image(filename=fname, width=1000))

In [ ]:
# ════════════════════════════════════════════════
# SLIDE 8 — Clinical Characterisation (CD + UC)
# ════════════════════════════════════════════════
print('\nSlide 8: Clinical characterisation...')

fig, axes = plt.subplots(
    1, 2, figsize=(16, 8))
fig.patch.set_facecolor('white')

clin_vars = {
    'Age'     : ('age',    True),
    'Sex'     : ('sex',    False),
    'BMI'     : ('bmi',    True),
    'Smoking' : ('smoking',False),
    'Batch'   : ('batch',  False),
}

for ax, meta, labels, cohort, col in [
    (axes[0], meta_cd, labels_cd,
     "Crohn's Disease", CD_COL),
    (axes[1], meta_uc, labels_uc,
     "Ulcerative Colitis", UC_COL)]:

    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(length=0)
    ax.grid(axis='x', alpha=0.2,
            linestyle='--', color='#ccc')

    rows = []
    for var_label, (col_name, is_cont) in \
            clin_vars.items():
        if col_name not in meta.columns:
            rows.append({
                'var': var_label,
                'p'  : 1.0,
                'sig': 'ns',
                'c1' : 'N/A',
                'c2' : 'N/A'})
            continue
        g0 = meta.loc[
            labels==0, col_name].dropna()
        g1 = meta.loc[
            labels==1, col_name].dropna()
        if len(g0)<3 or len(g1)<3:
            continue
        if is_cont:
            _, p = stats.mannwhitneyu(
                g0, g1,
                alternative='two-sided')
            c1_str = f'{g0.mean():.1f}±{g0.std():.1f}'
            c2_str = f'{g1.mean():.1f}±{g1.std():.1f}'
        else:
            ct = pd.crosstab(labels,
                              meta[col_name])
            if ct.shape == (2, 2):
                _, p, _, _ = \
                    stats.chi2_contingency(ct)
            else:
                p = 1.0
            c1_str = f'{g0.mean():.2f}'
            c2_str = f'{g1.mean():.2f}'
        sig = ('**' if p < 0.01 else
               ('*' if p < 0.05 else 'ns'))
        rows.append({
            'var': var_label,
            'p'  : round(p, 4),
            'sig': sig,
            'c1' : c1_str,
            'c2' : c2_str})

    y_pos = range(len(rows))
    bar_colours = [
        '#E8F5E9' if r['sig']=='ns'
        else '#FFEBEE'
        for r in rows]

    ax.barh(list(y_pos),
             [-np.log10(max(r['p'],
                             1e-10))
              for r in rows],
             color=bar_colours,
             edgecolor='#DDD',
             height=0.6, zorder=3)

    for yi, r in enumerate(rows):
        lp = -np.log10(max(r['p'], 1e-10))
        ax.text(lp+0.05, yi,
                 f'p={r["p"]} {r["sig"]}',
                 va='center', fontsize=10,
                 color=('#2E7D5B'
                         if r['sig']=='ns'
                         else '#E24B4A'),
                 fontweight='700')
        ax.text(-0.1, yi,
                 r['var'],
                 va='center', ha='right',
                 fontsize=11)

    ax.axvline(-np.log10(0.05),
                color='#E24B4A',
                linewidth=1.5,
                linestyle='--',
                alpha=0.7,
                label='p=0.05')
    ax.set_yticks([])
    ax.set_xlabel(
        '-log₁₀(p-value)', fontsize=12)
    ax.set_title(
        cohort, fontsize=15,
        fontweight='700',
        color='#1B3A6B', pad=10)

    ax.text(
        0.98, 0.02,
        'All ns → subtypes\nmolecularly defined ✅',
        transform=ax.transAxes,
        ha='right', va='bottom',
        fontsize=10, color='#2E7D5B',
        fontweight='700',
        bbox=dict(
            boxstyle='round,pad=0.4',
            facecolor='#E8F5E9',
            edgecolor='#2E7D5B',
            alpha=0.95))

plt.suptitle(
    'Clinical variables — '
    'no significant differences between subtypes',
    fontsize=15, fontweight='800',
    color='#1B3A6B', y=1.02)

plt.tight_layout()
fname = f'{FIGURES_DIR}/slide8_clinical.png'
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'✓ slide8_clinical.png')
display(Image(filename=fname, width=1000))

In [ ]:
# ════════════════════════════════════════════════
# SLIDE 9 — Volcano (CD + UC)
# ════════════════════════════════════════════════
print('\nSlide 9: Volcano plots...')

fig, axes = plt.subplots(
    1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')

for ax, de, cohort, n_up, n_dn in [
    (axes[0], de_cd,
     "Crohn's Disease", 3, 612),
    (axes[1], de_uc,
     "Ulcerative Colitis", 14, 633)]:

    de = de.copy()
    if 'protein' not in de.columns:
        de.rename(columns={
            de.columns[0]: 'protein'},
            inplace=True)

    de['log_q'] = -np.log10(
        de['adj.P.Val'].clip(lower=1e-300))

    sig_up = ((de['adj.P.Val'] < 0.05) &
               (de['logFC'] > 0))
    sig_dn = ((de['adj.P.Val'] < 0.05) &
               (de['logFC'] < 0))
    ns     = ~(sig_up | sig_dn)

    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.10,
            linestyle='--',
            color='#ccc', zorder=0)

    ax.scatter(
        de.loc[ns, 'logFC'],
        de.loc[ns, 'log_q'],
        c='#CCCCCC', s=18,
        alpha=0.4, linewidths=0,
        zorder=1, rasterized=True)
    ax.scatter(
        de.loc[sig_dn, 'logFC'],
        de.loc[sig_dn, 'log_q'],
        c=C1_COL, s=35, alpha=0.80,
        linewidths=0.3,
        edgecolors='white',
        zorder=3, rasterized=True)
    ax.scatter(
        de.loc[sig_up, 'logFC'],
        de.loc[sig_up, 'log_q'],
        c=C2_COL, s=35, alpha=0.80,
        linewidths=0.3,
        edgecolors='white',
        zorder=3, rasterized=True)

    # Annotate top proteins
    for mask, col, n in [
        (sig_dn, C1_COL, 6),
        (sig_up, C2_COL, 4)]:
        top = de[mask].nsmallest(
            n, 'adj.P.Val')
        for _, row in top.iterrows():
            ax.annotate(
                row['protein'],
                (row['logFC'],
                  row['log_q']),
                fontsize=8.5,
                style='italic',
                fontweight='700',
                color=col,
                xytext=(8, 3),
                textcoords='offset points',
                arrowprops=dict(
                    arrowstyle='-',
                    color=col,
                    alpha=0.4,
                    lw=0.8))

    ax.axhline(
        -np.log10(0.05),
        color='#555',
        linestyle='--',
        linewidth=1.2,
        alpha=0.7)
    ax.axvline(
        0, color='#aaa',
        linewidth=0.8, alpha=0.5)

    ax.text(
        0.98, 0.98,
        f'↑ C1: {sig_dn.sum()}\n'
        f'↑ C2: {sig_up.sum()}\n'
        f'FDR < 0.05\nlimma M3',
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=10,
        fontfamily='monospace',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            edgecolor='#DDD',
            alpha=0.95))

    ax.legend(handles=[
        Line2D([0],[0], marker='o',
                color='w',
                markerfacecolor=C1_COL,
                markersize=10,
                label=f'Higher in C1 '
                      f'(n={sig_dn.sum()})'),
        Line2D([0],[0], marker='o',
                color='w',
                markerfacecolor=C2_COL,
                markersize=10,
                label=f'Higher in C2 '
                      f'(n={sig_up.sum()})'),
        Line2D([0],[0], marker='o',
                color='w',
                markerfacecolor='#CCCCCC',
                markersize=10,
                label='Not significant')],
        fontsize=10,
        loc='upper left',
        framealpha=0.95)

    ax.set_xlabel(
        'log₂ Fold Change (C2 vs C1)',
        fontsize=13)
    ax.set_ylabel(
        '-log₁₀(FDR q-value)',
        fontsize=13)
    ax.set_title(
        cohort, fontsize=15,
        fontweight='700',
        color='#1B3A6B', pad=10)

plt.suptitle(
    'Differential protein abundance — '
    'limma M3 | FDR < 0.05',
    fontsize=16, fontweight='800',
    color='#1B3A6B', y=1.02)

plt.tight_layout()
fname = f'{FIGURES_DIR}/slide9_volcano.png'
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'✓ slide9_volcano.png')
display(Image(filename=fname, width=1100))

In [ ]:
# ════════════════════════════════════════════════
# SLIDE 10 — ORA Dot Plot (CD + UC)
# ════════════════════════════════════════════════
print('\nSlide 10: ORA dot plots...')

def clean_term(t):
    import re
    t = re.sub(r'\s+R-HSA-\d+','',t)
    t = re.sub(r'\s+\(GO:\d+\)','',t)
    t = re.sub(r'HALLMARK_','',t)
    t = t.replace('_',' ').strip()
    return t.title()

def parse_overlap(ov):
    try:
        if '/' in str(ov):
            n,d = str(ov).split('/')
            return int(n), int(d)
        return int(ov), 100
    except:
        return 5, 100

fig, axes = plt.subplots(
    1, 2, figsize=(20, 9))
fig.patch.set_facecolor('white')

for ax, ora, cohort, col in [
    (axes[0], ora_cd,
     "Crohn's Disease", CD_COL),
    (axes[1], ora_uc,
     "Ulcerative Colitis", UC_COL)]:

    ora = ora.copy()
    ora = ora[
        ora['Adjusted P-value'] < 0.05
    ].copy()
    ora['log_fdr'] = -np.log10(
        ora['Adjusted P-value'].clip(
            lower=1e-30))
    ora['Term_clean'] = ora[
        'Term'].apply(clean_term)

    # Get overlap count
    ov_col = None
    for c in ['Overlap','overlap',
               'Gene_count','Count']:
        if c in ora.columns:
            ov_col = c
            break

    if ov_col:
        ora['n_genes'] = ora[
            ov_col].apply(
            lambda x: parse_overlap(x)[0])
    else:
        ora['n_genes'] = 10

    # Top 10 by FDR
    top = ora.nsmallest(
        10, 'Adjusted P-value')

    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.grid(axis='x', alpha=0.2,
            linestyle='--',
            color='#ccc', zorder=0)
    ax.tick_params(length=0)

    norm = plt.Normalize(
        top['log_fdr'].min(),
        top['log_fdr'].max())
    cmap = mpl.colormaps['YlOrRd']

    scatter = ax.scatter(
        top['log_fdr'],
        range(len(top)),
        s=top['n_genes'] * 12 + 80,
        c=top['log_fdr'],
        cmap='YlOrRd',
        vmin=top['log_fdr'].min(),
        vmax=top['log_fdr'].max(),
        alpha=0.90,
        linewidths=0.5,
        edgecolors='white',
        zorder=4)

    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(
        [t[:45] for t in
         top['Term_clean']],
        fontsize=10)
    ax.set_xlabel(
        '-log₁₀(FDR q-value)',
        fontsize=12)
    ax.set_title(
        cohort, fontsize=15,
        fontweight='700',
        color='#1B3A6B', pad=10)

    # Colourbar
    cbar = plt.colorbar(
        scatter, ax=ax,
        shrink=0.6, pad=0.01)
    cbar.set_label(
        '-log₁₀(FDR)',
        fontsize=10)

    # Size legend
    for size, label in [
        (80+12*5, '5 genes'),
        (80+12*15,'15 genes'),
        (80+12*30,'30 genes')]:
        ax.scatter(
            [], [],
            s=size,
            c='#888',
            alpha=0.6,
            label=label)
    ax.legend(
        title='Gene count',
        fontsize=9,
        title_fontsize=9,
        loc='lower right',
        framealpha=0.9)

    ax.axvline(
        -np.log10(0.05),
        color='#E24B4A',
        linewidth=1.2,
        linestyle='--',
        alpha=0.7)

plt.suptitle(
    'Hallmark pathway enrichment — '
    'Hyperinflammatory subtype (C1)\n'
    'Same inflammatory cascade in '
    'both CD and UC',
    fontsize=15, fontweight='800',
    color='#1B3A6B', y=1.02)

plt.tight_layout()
fname = f'{FIGURES_DIR}/slide10_ora.png'
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'✓ slide10_ora.png')
display(Image(filename=fname, width=1100))

In [ ]:
# ════════════════════════════════════════════════
# SLIDE 11 — RF Importance (CD only)
# ════════════════════════════════════════════════
print('\nSlide 11: RF importance...')

de_cd_sig = set(de_cd[
    de_cd['adj.P.Val'] < 0.05
]['protein'].tolist())

top20_p = rf_cd.head(20)[
    'protein'].tolist()
top20_i = rf_cd.head(20)[
    'gini_imp'].tolist()
overlap  = set(top20_p) & de_cd_sig

fig, axes = plt.subplots(
    1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')

# Panel A — importance bars
ax = axes[0]
ax.set_facecolor('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.grid(axis='x', alpha=0.2,
        linestyle='--', color='#ccc')
ax.tick_params(length=0)

bar_cols = [
    C1_COL if p in overlap
    else '#BBBBBB'
    for p in top20_p]

ax.barh(
    range(len(top20_p)),
    list(reversed(top20_i)),
    color=list(reversed(bar_cols)),
    alpha=0.88, height=0.65,
    edgecolor='white', zorder=3)
ax.set_yticks(range(len(top20_p)))
ax.set_yticklabels(
    list(reversed(top20_p)),
    fontsize=10, style='italic')
ax.set_xlabel(
    'Gini importance', fontsize=12)
ax.set_title(
    "Crohn's Disease — RF top 20",
    fontsize=14, fontweight='700',
    color='#1B3A6B', pad=10)
ax.text(
    0.98, 0.02,
    f'CV AUC = 0.983\nRF∩limma = {len(overlap)}/20',
    transform=ax.transAxes,
    ha='right', va='bottom',
    fontsize=11, fontweight='700',
    color=C1_COL,
    bbox=dict(
        boxstyle='round,pad=0.4',
        facecolor='white',
        edgecolor='#ccc',
        alpha=0.95))
ax.legend(handles=[
    mpatches.Patch(
        facecolor=C1_COL, alpha=0.88,
        label='Also in limma top proteins'),
    mpatches.Patch(
        facecolor='#BBBBBB', alpha=0.88,
        label='RF only')],
    fontsize=10)

# Panel B — UC AUC comparison
ax = axes[1]
ax.set_facecolor('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.grid(axis='y', alpha=0.2,
        linestyle='--', color='#ccc')
ax.tick_params(length=0)

cohorts = ['UKB CD','UKB UC',
            'IBDome CD','IBDome UC']
aucs    = [0.983, 0.985, 0.990, 1.000]
cols_   = [CD_COL, UC_COL,
            '#C96A1F','#E24B4A']

bars = ax.bar(
    cohorts, aucs,
    color=cols_,
    alpha=0.88,
    width=0.5,
    edgecolor='white',
    linewidth=0.5,
    zorder=3)

for bar, auc in zip(bars, aucs):
    ax.text(
        bar.get_x() +
        bar.get_width()/2,
        auc + 0.002,
        f'{auc:.3f}',
        ha='center', va='bottom',
        fontsize=12,
        fontweight='700',
        color='#333')

ax.set_ylim(0.95, 1.02)
ax.set_ylabel('CV AUC', fontsize=12)
ax.set_title(
    'RF AUC across all cohorts',
    fontsize=14, fontweight='700',
    color='#1B3A6B', pad=10)
ax.axhline(
    0.5, color='#888',
    linewidth=1, linestyle='--',
    alpha=0.4,
    label='Random (0.5)')
ax.tick_params(
    axis='x', labelsize=11)

plt.suptitle(
    'Random Forest triangulation — '
    'machine learning confirms subtypes',
    fontsize=15, fontweight='800',
    color='#1B3A6B', y=1.02)

plt.tight_layout()
fname = f'{FIGURES_DIR}/slide11_rf.png'
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'✓ slide11_rf.png')
display(Image(filename=fname, width=1100))

In [ ]:
# ════════════════════════════════════════════════
# SLIDE 13 — IBDome Validation
# ════════════════════════════════════════════════
print('\nSlide 13: Validation...')

# Load IBDome data
X_ib   = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_preprocessed.csv'
).values.astype(float)
prot_ib = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_preprocessed.csv'
).columns.tolist()
meta_ib = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_final.csv')
labels_ib = meta_ib[
    'cluster_dec'].values

def score(X, prots, all_prots):
    av  = [p for p in prots
            if p in all_prots]
    if not av:
        return np.zeros(len(X))
    idx = [all_prots.index(p)
            for p in av]
    return X[:, idx].mean(axis=1)

chemokine = [p for p in prot_ib if p in [
    'IL8','MCP-3','MCP-1','CXCL11',
    'CXCL9','CXCL1','CCL4','CCL19',
    'CXCL5','CCL3','CXCL6','CXCL10',
    'CCL28','CCL25','CCL20']]
vascular  = [p for p in prot_ib if p in [
    'VEGFA','HGF','FGF-21','FGF-19',
    'FGF-5','LIF','ARTN','NRTN',
    'GDNF','FGF-23','Beta-NGF']]

sc_chemo = score(X_ib, chemokine, prot_ib)
sc_vasc  = score(X_ib, vascular,  prot_ib)
r_cv, p_cv = stats.pearsonr(
    sc_chemo, sc_vasc)

fig, axes = plt.subplots(
    1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')

# Panel A — Chemokine × Vascular scatter
ax = axes[0]
ax.set_facecolor('#F8F9FA')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.12, linestyle='--',
        color='#ccc', zorder=0)

SUBTYPE_COL_IB = {
    0: C1_COL,  # quiescent
    1: C2_COL   # hyperinflam
}
SUBTYPE_NAME_IB = {
    0: 'C1 Quiescent',
    1: 'C2 Hyperinflammatory'
}

for c in range(2):
    m = labels_ib == c
    ax.scatter(
        sc_chemo[m], sc_vasc[m],
        c=SUBTYPE_COL_IB[c],
        s=45, alpha=0.78,
        linewidths=0.4,
        edgecolors='white',
        label=f'{SUBTYPE_NAME_IB[c]} '
              f'(n={m.sum()})',
        zorder=3)

m_, b_ = np.polyfit(sc_chemo, sc_vasc, 1)
xr     = np.linspace(
    sc_chemo.min(), sc_chemo.max(), 100)
ax.plot(xr, m_*xr+b_,
         color='black',
         linewidth=2,
         linestyle='--', zorder=4)

ax.text(
    0.05, 0.95,
    f'IBDome CD\nr = {r_cv:.3f}\np < 0.0001\n\n'
    f'UKB reference\nr = 0.749',
    transform=ax.transAxes,
    va='top', fontsize=12,
    fontweight='700',
    bbox=dict(
        boxstyle='round,pad=0.5',
        facecolor='white',
        edgecolor='#DDD',
        alpha=0.95))

ax.set_xlabel(
    'Chemokine score', fontsize=13)
ax.set_ylabel(
    'Vascular score', fontsize=13)
ax.set_title(
    'Chemokine × Vascular\n'
    'co-activation replicated',
    fontsize=14, fontweight='700',
    color='#1B3A6B', pad=10)
ax.legend(fontsize=10,
           framealpha=0.95)

# Panel B — Cross-cohort comparison
ax = axes[1]
ax.set_facecolor('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.grid(axis='y', alpha=0.2,
        linestyle='--', color='#ccc')
ax.tick_params(length=0)

metrics   = ['Silhouette',
              'Stability',
              'RF AUC']
ukb_vals  = [0.846, 0.893, 0.983]
ib_vals   = [0.834, 0.886, 0.990]
x_m       = np.arange(len(metrics))
w         = 0.3

ax.bar(x_m - w/2, ukb_vals,
        width=w, color=CD_COL,
        alpha=0.88,
        label='UKB CD',
        edgecolor='white',
        zorder=3)
ax.bar(x_m + w/2, ib_vals,
        width=w, color='#C96A1F',
        alpha=0.88,
        label='IBDome CD',
        edgecolor='white',
        zorder=3)

for xi, (u, b) in enumerate(
        zip(ukb_vals, ib_vals)):
    ax.text(
        xi-w/2, u+0.005,
        f'{u:.3f}',
        ha='center', va='bottom',
        fontsize=10, color=CD_COL,
        fontweight='700')
    ax.text(
        xi+w/2, b+0.005,
        f'{b:.3f}',
        ha='center', va='bottom',
        fontsize=10, color='#C96A1F',
        fontweight='700')

ax.set_xticks(x_m)
ax.set_xticklabels(
    metrics, fontsize=12)
ax.set_ylim(0, 1.10)
ax.set_ylabel('Value', fontsize=12)
ax.legend(fontsize=11,
           framealpha=0.95)
ax.set_title(
    'UKB vs IBDome\nconsistent metrics',
    fontsize=14, fontweight='700',
    color='#1B3A6B', pad=10)

plt.suptitle(
    'Independent validation — '
    'IBDome Germany (n=201 CD)\n'
    'Chemokine × Vascular: '
    'IBDome r=0.735 vs UKB r=0.749 ✅',
    fontsize=14, fontweight='800',
    color='#1B3A6B', y=1.02)

plt.tight_layout()
fname = (f'{FIGURES_DIR}/'
          f'slide13_validation.png')
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'✓ slide13_validation.png')
display(Image(filename=fname,
               width=1100))

print('\nALL SLIDE FIGURES COMPLETE')
print('='*55)
for f in [
    'slide7_clustering.png',
    'slide8_clinical.png',
    'slide9_volcano.png',
    'slide10_ora.png',
    'slide11_rf.png',
    'slide12_ppi.png',
    'slide13_validation.png']:
    path = f'{FIGURES_DIR}/{f}'
    if os.path.exists(path):
        size = os.path.getsize(
            path)//1024
        print(f'  ✅ {f} ({size} KB)')

In [ ]:
import os
from IPython.display import Image, display

FIGURES_DIR = '/rds/homes/j/jxt554/figures'

for f in [
    'slide7_clustering.png',
    'slide8_clinical.png',
    'slide9_volcano.png',
    'slide10_ora.png',
    'slide11_rf.png',
    'slide12_ppi.png',
    'slide13_validation.png']:
    path = f'{FIGURES_DIR}/{f}'
    if os.path.exists(path):
        print(f'\n{f}:')
        display(Image(filename=path,
                       width=900))
    else:
        print(f'\n❌ {f} not found')

In [ ]:
import os
import numpy as np

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'

print('EXISTING GOOD CLUSTERING FIGURES:')
for f in os.listdir(FIGURES_DIR):
    if 'cluster' in f.lower() or \
       'umap' in f.lower() or \
       'slide' in f.lower():
        path = f'{FIGURES_DIR}/{f}'
        size = os.path.getsize(path)//1024
        print(f'  {f} ({size} KB)')

print('\nSAVED LATENT SPACES:')
for f in os.listdir(DATA_DIR):
    if 'latent' in f.lower() and \
       '.npy' in f:
        path = f'{DATA_DIR}/{f}'
        arr  = np.load(path)
        print(f'  {f}: {arr.shape}')

In [ ]:
from IPython.display import Image, display
import os

FIGURES_DIR = '/rds/homes/j/jxt554/figures'

# Show the best existing clustering figures
print('BEST EXISTING CLUSTERING FIGURES:')
print('='*55)

for f, label in [
    ('slide_cd_clean_canonical.png',
     'CD Clean Canonical'),
    ('slide_uc_clean_canonical.png',
     'UC Clean Canonical'),
    ('slide_all_groups_overview.png',
     'All Groups Overview'),
    ('clustering_cd.png',
     'CD Clustering Full'),
    ('clustering_uc.png',
     'UC Clustering Full'),
    ('umap_cd_clean.png',
     'CD UMAP Clean'),
    ('umap_uc_clean.png',
     'UC UMAP Clean'),
]:
    path = f'{FIGURES_DIR}/{f}'
    if os.path.exists(path):
        print(f'\n{label}:')
        display(Image(filename=path,
                       width=900))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import matplotlib as mpl
import re
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display
import os

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'

C1_COL = '#2E7D5B'
C2_COL = '#E24B4A'
BG_COL = '#FAFAFA'

def clean_term(t):
    t = re.sub(r'HALLMARK_','',str(t))
    t = re.sub(r'\s+R-HSA-\d+','',t)
    t = re.sub(r'\s+\(GO:\d+\)','',t)
    t = t.replace('_',' ').strip().title()
    replacements = {
        'Tnf Alpha Signaling Via Nfkb':
            'TNF/NF-kB',
        'Il6 Jak Stat3 Signaling':
            'IL-6/JAK/STAT3',
        'Interferon Gamma Response':
            'IFN-γ Response',
        'Allograft Rejection':
            'Allograft Rejection',
        'Inflammatory Response':
            'Inflammatory Response',
        'Epithelial Mesenchymal Transition':
            'EMT',
        'Interferon Alpha Response':
            'IFN-α Response',
        'Il2 Stat5 Signaling':
            'IL-2/STAT5',
        'Heme Metabolism':
            'Heme Metabolism',
        'Hypoxia':'Hypoxia',
        'Apoptosis':'Apoptosis',
        'Myc Targets V1':'MYC Targets',
        'Complement':'Complement',
        'Kras Signaling Up':'KRAS Up',
    }
    for old,new in replacements.items():
        if old.lower() in t.lower():
            return new
    return t[:38]

def parse_overlap(ov):
    try:
        if '/' in str(ov):
            return int(str(ov).split('/')[0])
        return int(float(str(ov)))
    except:
        return 8

def legend_dot(colour, size, label):
    '''Helper — no empty scatter'''
    return Line2D(
        [0],[0],
        marker='o',
        color='w',
        markerfacecolor=colour,
        markersize=size,
        alpha=0.85,
        label=label)

def make_combined(cohort_label,
                   de_file, ora_file,
                   out_fname):

    de  = pd.read_csv(
        f'{TABLES_DIR}/{de_file}')
    ora = pd.read_csv(
        f'{TABLES_DIR}/{ora_file}')

    if 'protein' not in de.columns:
        de.rename(columns={
            de.columns[0]:'protein'},
            inplace=True)

    fc_col = 'logFC'
    q_col  = ('adj.P.Val'
               if 'adj.P.Val' in de.columns
               else 'adj_p_value')

    de['log_q'] = -np.log10(
        de[q_col].clip(lower=1e-300))
    sig_up = ((de[q_col]<0.05)&
               (de[fc_col]>0))
    sig_dn = ((de[q_col]<0.05)&
               (de[fc_col]<0))
    ns     = ~(sig_up|sig_dn)

    ora = ora[
        ora['Adjusted P-value']<0.05
    ].copy()
    ora['log_fdr'] = -np.log10(
        ora['Adjusted P-value'].clip(
            lower=1e-30))
    ora['Term_clean'] = \
        ora['Term'].apply(clean_term)

    ov_col = next(
        (c for c in
         ['Overlap','overlap',
          'Gene_count','Count']
         if c in ora.columns), None)
    ora['n_genes'] = (
        ora[ov_col].apply(parse_overlap)
        if ov_col else 10)

    top_ora = ora.nsmallest(
        10,'Adjusted P-value'
    ).sort_values(
        'log_fdr', ascending=True)

    min_g = float(top_ora['n_genes'].min())
    max_g = float(top_ora['n_genes'].max())
    rng   = max(max_g-min_g, 1.0)

    def dot_size(n):
        return 100+((n-min_g)/rng)*300

    # ── Figure ────────────────────────
    fig = plt.figure(figsize=(20,9))
    fig.patch.set_facecolor('white')
    gs  = gridspec.GridSpec(
        1,2, figure=fig,
        wspace=0.32,
        left=0.05, right=0.95,
        top=0.88, bottom=0.10)

    # ════════════════════════
    # PANEL A — Volcano
    # ════════════════════════
    ax1 = fig.add_subplot(gs[0,0])
    ax1.set_facecolor(BG_COL)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_color('#CCCCCC')
    ax1.spines['bottom'].set_color(
        '#CCCCCC')

    ax1.scatter(
        de.loc[ns,fc_col],
        de.loc[ns,'log_q'],
        c='#CCCCCC', s=15,
        alpha=0.40, linewidths=0,
        zorder=1, rasterized=True)
    ax1.scatter(
        de.loc[sig_dn,fc_col],
        de.loc[sig_dn,'log_q'],
        c=C1_COL, s=28, alpha=0.85,
        linewidths=0.3,
        edgecolors='white',
        zorder=3, rasterized=True)
    ax1.scatter(
        de.loc[sig_up,fc_col],
        de.loc[sig_up,'log_q'],
        c=C2_COL, s=28, alpha=0.85,
        linewidths=0.3,
        edgecolors='white',
        zorder=3, rasterized=True)

    for _,row in de[sig_dn].nsmallest(
            7,q_col).iterrows():
        ax1.annotate(
            row['protein'],
            (row[fc_col],row['log_q']),
            fontsize=8, style='italic',
            fontweight='600',
            color=C1_COL,
            xytext=(-32,3),
            textcoords='offset points',
            arrowprops=dict(
                arrowstyle='-',
                color=C1_COL,
                alpha=0.35,lw=0.7))

    for _,row in de[sig_up].nsmallest(
            4,q_col).iterrows():
        ax1.annotate(
            row['protein'],
            (row[fc_col],row['log_q']),
            fontsize=8, style='italic',
            fontweight='600',
            color=C2_COL,
            xytext=(6,3),
            textcoords='offset points',
            arrowprops=dict(
                arrowstyle='-',
                color=C2_COL,
                alpha=0.35,lw=0.7))

    ax1.axhline(
        -np.log10(0.05),
        color='#888888',
        linestyle='--',
        linewidth=1.2, alpha=0.8)
    ax1.axvline(
        0, color='#AAAAAA',
        linewidth=0.8, alpha=0.6)

    ax1.text(
        0.97,0.97,
        f'↑ C1: {sig_dn.sum()}\n'
        f'↑ C2: {sig_up.sum()}\n'
        f'FDR<0.05 | limma M3',
        transform=ax1.transAxes,
        ha='right', va='top',
        fontsize=9.5,
        fontfamily='monospace',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            edgecolor='#DDDDDD',
            alpha=0.97))

    ax1.legend(handles=[
        legend_dot(C1_COL, 9,
            f'Higher in C1 '
            f'(n={sig_dn.sum()})'),
        legend_dot(C2_COL, 9,
            f'Higher in C2 '
            f'(n={sig_up.sum()})'),
        legend_dot('#CCCCCC', 9,
            'Not significant')],
        fontsize=9.5,
        loc='upper left',
        framealpha=0.95,
        edgecolor='#DDDDDD')

    ax1.set_xlabel(
        'log₂ Fold Change (C2 vs C1)',
        fontsize=12, color='#333')
    ax1.set_ylabel(
        '-log₁₀(FDR q-value)',
        fontsize=12, color='#333')
    ax1.set_title(
        '(A) Differential Protein Abundance',
        fontsize=13, fontweight='700',
        color='#1B3A6B',
        loc='left', pad=10)
    ax1.tick_params(
        colors='#666', labelsize=10)

    # ════════════════════════
    # PANEL B — ORA Dot Plot
    # ════════════════════════
    ax2 = fig.add_subplot(gs[0,1])
    ax2.set_facecolor(BG_COL)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_color(
        '#CCCCCC')
    ax2.grid(
        axis='x', alpha=0.25,
        linestyle='--',
        color='#DDDDDD', zorder=0)
    ax2.tick_params(length=0)

    sizes = [dot_size(n)
              for n in top_ora['n_genes']]

    scatter = ax2.scatter(
        top_ora['log_fdr'],
        range(len(top_ora)),
        s=sizes,
        c=top_ora['log_fdr'],
        cmap='YlOrRd',
        vmin=top_ora['log_fdr'].min(),
        vmax=top_ora['log_fdr'].max(),
        alpha=0.92,
        linewidths=0.8,
        edgecolors='white',
        zorder=4)

    for xi,(_,row) in enumerate(
            top_ora.iterrows()):
        ax2.text(
            row['log_fdr']+0.12, xi,
            f'{row["log_fdr"]:.1f}',
            va='center', fontsize=9,
            color='#444444',
            fontweight='600')

    ax2.set_yticks(range(len(top_ora)))
    ax2.set_yticklabels(
        top_ora['Term_clean'],
        fontsize=10.5,
        color='#222222')
    ax2.set_xlabel(
        '-log₁₀(FDR q-value)',
        fontsize=12, color='#333')
    ax2.set_title(
        '(B) Hallmark Pathway Enrichment\n'
        'Hyperinflammatory C1 proteins',
        fontsize=13, fontweight='700',
        color='#1B3A6B',
        loc='left', pad=10)
    ax2.tick_params(
        axis='x', colors='#666',
        labelsize=10)
    ax2.axvline(
        -np.log10(0.05),
        color='#888888',
        linewidth=1.2,
        linestyle='--', alpha=0.7)

    cbar = plt.colorbar(
        scatter, ax=ax2,
        shrink=0.55, pad=0.02,
        aspect=20)
    cbar.set_label(
        '-log₁₀(FDR)',
        fontsize=9.5, color='#444')
    cbar.ax.tick_params(
        labelsize=9, colors='#666')

    # Size legend using Line2D proxy
    size_ticks = [
        int(min_g),
        int(min_g+(max_g-min_g)/2),
        int(max_g)]
    ax2.legend(
        handles=[
            legend_dot(
                '#AAAAAA',
                np.sqrt(dot_size(s))*0.55,
                f'{s} genes')
            for s in size_ticks],
        title='Gene count',
        fontsize=9,
        title_fontsize=9.5,
        loc='lower right',
        framealpha=0.92,
        edgecolor='#DDDDDD')

    fig.suptitle(
        f'{cohort_label} — '
        'Proteomic Subtype Characterisation',
        fontsize=16, fontweight='800',
        color='#1B3A6B', y=0.98)

    path = f'{FIGURES_DIR}/{out_fname}'
    plt.savefig(
        path, dpi=220,
        bbox_inches='tight',
        facecolor='white')
    plt.close()
    size = os.path.getsize(path)//1024
    print(f'✓ {out_fname} ({size} KB)')
    display(Image(filename=path,
                   width=1100))

# ── Run ───────────────────────────────────────────
print('CD figure...')
make_combined(
    "Crohn's Disease",
    'cd_limma_M3.csv',
    'ora_cd_C1_up_Hallmark.csv',
    'slide_cd_volcano_ora.png')

print('\nUC figure...')
make_combined(
    'Ulcerative Colitis',
    'uc_limma_M3.csv',
    'ora_uc_C1_up_Hallmark.csv',
    'slide_uc_volcano_ora.png')

print('\nDONE ✅')

In [ ]:
import pandas as pd

meta_cd = pd.read_csv(
    '/rds/homes/j/jxt554/data/'
    'meta_cd_final.csv')
meta_uc = pd.read_csv(
    '/rds/homes/j/jxt554/data/'
    'meta_uc_final.csv')

print('CD columns:')
print(meta_cd.columns.tolist())
print(f'\nCD shape: {meta_cd.shape}')

print('\nUC columns:')
print(meta_uc.columns.tolist())
print(f'UC shape: {meta_uc.shape}')

print('\nCD cluster counts:')
print(meta_cd['cluster'].value_counts())

print('\nUC cluster counts:')
print(meta_uc['cluster'].value_counts())

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display
import os

DATA_DIR    = '/rds/homes/j/jxt554/data'
FIGURES_DIR = '/rds/homes/j/jxt554/figures'

# ── Load ──────────────────────────────────────────
meta_cd = pd.read_csv(f'{DATA_DIR}/meta_cd_final.csv')
meta_uc = pd.read_csv(f'{DATA_DIR}/meta_uc_final.csv')

labels_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
labels_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

meta_cd['cluster'] = labels_cd
meta_uc['cluster'] = labels_uc

# ── Subtype mapping ───────────────────────────────
# cluster=0 → C1 Hyperinflammatory
# cluster=1 → C2 Quiescent
C1_COL  = '#E24B4A'
C2_COL  = '#2E7D5B'
PALETTE = {0: C1_COL, 1: C2_COL}
NAMES   = {0:'C1 Hyperinflam',
            1:'C2 Quiescent'}

def jitter(n, width=0.12):
    return np.random.uniform(
        -width, width, n)

def violin_box(ax, data, labels,
                palette, title,
                ylabel, unit=''):
    positions = [0, 1]
    parts = ax.violinplot(
        [data[labels==c].dropna()
         for c in [0,1]],
        positions=positions,
        showmedians=False,
        showextrema=False,
        widths=0.6)

    for pc, c in zip(
            parts['bodies'], [0,1]):
        pc.set_facecolor(palette[c])
        pc.set_alpha(0.25)
        pc.set_edgecolor(palette[c])
        pc.set_linewidth(0.8)

    for c, pos in zip([0,1], positions):
        vals = data[labels==c].dropna()
        q1,med,q3 = np.percentile(
            vals,[25,50,75])
        ax.plot([pos,pos],[q1,q3],
                 color=palette[c],
                 linewidth=3,
                 solid_capstyle='round',
                 zorder=3)
        ax.scatter(pos, med,
                    color='white',
                    s=55, zorder=4,
                    edgecolors=palette[c],
                    linewidth=1.5)
        np.random.seed(42)
        jit = jitter(len(vals))
        ax.scatter(
            pos + jit, vals,
            color=palette[c],
            s=12, alpha=0.35,
            linewidths=0, zorder=2)

        # p-value
    g0 = data[labels==0].dropna()
    g1 = data[labels==1].dropna()
    _, p = stats.mannwhitneyu(
        g0, g1, alternative='two-sided')
    star = ('***' if p<0.001 else
            ('**' if p<0.01 else
             ('*' if p<0.05 else 'ns')))
    y_max = data.dropna().max()
    y_rng = y_max - data.dropna().min()
    ax.plot([0,1],
             [y_max+y_rng*0.05,
              y_max+y_rng*0.05],
             color='#666', lw=0.8)
    ax.text(0.5, y_max+y_rng*0.10,
             f'p={p:.3f} {star}',
             ha='center', fontsize=9,
             color='#555')

    ax.set_xticks([0,1])
    ax.set_xticklabels(
        [NAMES[0], NAMES[1]],
        fontsize=9)
    ax.set_ylabel(
        f'{ylabel}{" ("+unit+")" if unit else ""}',
        fontsize=10)
    ax.set_title(title, fontsize=11,
                  fontweight='700',
                  color='#1B3A6B',
                  loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#DDDDDD')
    ax.spines['bottom'].set_color(
        '#DDDDDD')
    ax.tick_params(colors='#555',
                    length=3)
    ax.set_facecolor('#FAFAFA')
    ax.grid(axis='y', alpha=0.2,
            linestyle='--', color='#ccc')

def cat_bar(ax, data, labels,
             palette, title,
             cat_map=None):
    cats = sorted(data.dropna().unique())
    x   = np.arange(len([0,1]))
    w   = 0.3

    for i, cat in enumerate(cats):
        vals = []
        for c in [0,1]:
            mask = labels == c
            sub  = data[mask].dropna()
            pct  = (sub==cat).mean()*100
            vals.append(pct)
        offset = (i - len(cats)/2 +
                   0.5) * (w+0.04)
        col = plt.cm.Set2(
            i/max(len(cats)-1, 1))
        label = (cat_map.get(cat, str(cat))
                  if cat_map else str(cat))
        ax.bar(x+offset, vals,
                width=w, color=col,
                alpha=0.85,
                label=label,
                edgecolor='white',
                linewidth=0.5,
                zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [NAMES[0], NAMES[1]],
        fontsize=9)
    ax.set_ylabel('%', fontsize=10)
    ax.set_title(title, fontsize=11,
                  fontweight='700',
                  color='#1B3A6B',
                  loc='left')
    ax.legend(fontsize=8,
               framealpha=0.8,
               ncol=min(len(cats), 3))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#DDDDDD')
    ax.spines['bottom'].set_color(
        '#DDDDDD')
    ax.tick_params(colors='#555',
                    length=3)
    ax.set_facecolor('#FAFAFA')
    ax.grid(axis='y', alpha=0.2,
            linestyle='--', color='#ccc')

    # Chi-sq p
    try:
        ct = pd.crosstab(labels, data)
        _, p, _, _ = \
            stats.chi2_contingency(ct)
        star = ('***' if p<0.001 else
                ('**' if p<0.01 else
                 ('*' if p<0.05
                  else 'ns')))
        ax.text(0.98, 0.97,
                 f'p={p:.3f} {star}',
                 transform=ax.transAxes,
                 ha='right', va='top',
                 fontsize=9,
                 color='#555',
                 bbox=dict(
                     boxstyle='round,pad=0.3',
                     facecolor='white',
                     edgecolor='#DDD',
                     alpha=0.9))
    except:
        pass

# ════════════════════════════════════════
# FIGURE
# ════════════════════════════════════════
fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('white')

gs_main = gridspec.GridSpec(
    1, 2, figure=fig,
    wspace=0.30,
    left=0.06, right=0.97,
    top=0.92, bottom=0.06)

# ── Cohort labels ─────────────────────
for i, (cohort_label, col) in enumerate([
        ("Crohn's Disease", '#1B3A6B'),
        ("Ulcerative Colitis", '#C96A1F')]):
    ax_title = fig.add_axes(
        [0.06 + i*0.47, 0.93,
         0.44, 0.04])
    ax_title.set_facecolor(col)
    ax_title.text(
        0.5, 0.5, cohort_label,
        ha='center', va='center',
        fontsize=14, fontweight='800',
        color='white',
        transform=ax_title.transAxes)
    ax_title.set_xticks([])
    ax_title.set_yticks([])
    for sp in ax_title.spines.values():
        sp.set_visible(False)

# ── Left: CD ─────────────────────────
gs_cd = gridspec.GridSpecFromSubplotSpec(
    3, 2,
    subplot_spec=gs_main[0],
    hspace=0.45, wspace=0.35)

# Age
ax = fig.add_subplot(gs_cd[0, 0])
violin_box(ax, meta_cd['age'],
            labels_cd, PALETTE,
            '(A) Age', 'Age', 'years')

# BMI
ax = fig.add_subplot(gs_cd[0, 1])
violin_box(ax, meta_cd['bmi'],
            labels_cd, PALETTE,
            '(B) BMI', 'BMI', 'kg/m²')

# HbA1c
ax = fig.add_subplot(gs_cd[1, 0])
violin_box(ax, meta_cd['hba1c'],
            labels_cd, PALETTE,
            '(C) HbA1c', 'HbA1c',
            'mmol/mol')

# Sex
ax = fig.add_subplot(gs_cd[1, 1])
cat_bar(ax, meta_cd['sex'],
         labels_cd, PALETTE,
         '(D) Sex',
         {0:'Female', 1:'Male',
          '0':'Female','1':'Male'})

# Smoking
ax = fig.add_subplot(gs_cd[2, 0])
smoke_map = {
    0:'Never', 1:'Previous',
    2:'Current',
    '0':'Never','1':'Previous',
    '2':'Current'}
cat_bar(ax,
         meta_cd['smoking_status'],
         labels_cd, PALETTE,
         '(E) Smoking status',
         smoke_map)

# Batch
ax = fig.add_subplot(gs_cd[2, 1])
cat_bar(ax, meta_cd['batch'],
         labels_cd, PALETTE,
         '(F) Batch',
         {0:'Batch 0', 1:'Batch 1',
          '0':'Batch 0','1':'Batch 1'})

# ── Right: UC ────────────────────────
gs_uc = gridspec.GridSpecFromSubplotSpec(
    3, 2,
    subplot_spec=gs_main[1],
    hspace=0.45, wspace=0.35)

# Age
ax = fig.add_subplot(gs_uc[0, 0])
violin_box(ax, meta_uc['age'],
            labels_uc, PALETTE,
            '(A) Age', 'Age', 'years')

# BMI
ax = fig.add_subplot(gs_uc[0, 1])
violin_box(ax, meta_uc['bmi'],
            labels_uc, PALETTE,
            '(B) BMI', 'BMI', 'kg/m²')

# HbA1c
ax = fig.add_subplot(gs_uc[1, 0])
violin_box(ax, meta_uc['hba1c'],
            labels_uc, PALETTE,
            '(C) HbA1c', 'HbA1c',
            'mmol/mol')

# Sex
ax = fig.add_subplot(gs_uc[1, 1])
cat_bar(ax, meta_uc['sex'],
         labels_uc, PALETTE,
         '(D) Sex',
         {0:'Female', 1:'Male',
          '0':'Female','1':'Male'})

# Smoking
ax = fig.add_subplot(gs_uc[2, 0])
cat_bar(ax,
         meta_uc['smoking_status'],
         labels_uc, PALETTE,
         '(E) Smoking status',
         smoke_map)

# Batch
ax = fig.add_subplot(gs_uc[2, 1])
cat_bar(ax, meta_uc['batch'],
         labels_uc, PALETTE,
         '(F) Batch',
         {0:'Batch 0', 1:'Batch 1',
          '0':'Batch 0','1':'Batch 1'})

# ── Legend ────────────────────────────
fig.legend(handles=[
    mpatches.Patch(
        facecolor=C1_COL, alpha=0.85,
        label='C1 Hyperinflammatory'),
    mpatches.Patch(
        facecolor=C2_COL, alpha=0.85,
        label='C2 Quiescent')],
    fontsize=11,
    loc='lower center',
    ncol=2,
    bbox_to_anchor=(0.5, 0.01),
    framealpha=0.95,
    edgecolor='#DDD')

# ── Suptitle ──────────────────────────
fig.suptitle(
    'Clinical characterisation — '
    'subtypes not driven by demographics\n'
    'Mann-Whitney U / Chi-squared · '
    'BH correction · all variables ns',
    fontsize=14, fontweight='800',
    color='#1B3A6B', y=0.99)

fname = (f'{FIGURES_DIR}/'
          f'slide8_clinical_combined.png')
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
size = os.path.getsize(fname)//1024
print(f'✓ slide8_clinical_combined.png'
       f' ({size} KB)')
display(Image(filename=fname, width=1100))

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display
import os

DATA_DIR    = '/rds/homes/j/jxt554/data'
FIGURES_DIR = '/rds/homes/j/jxt554/figures'
np.random.seed(42)

# ── Load ──────────────────────────────────────────
meta_cd   = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
meta_uc   = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')
labels_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
labels_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

meta_cd['cluster'] = labels_cd
meta_uc['cluster'] = labels_uc

# ── Palette ───────────────────────────────────────
C1_COL  = '#E24B4A'
C2_COL  = '#2E7D5B'
PALETTE = {0: C1_COL, 1: C2_COL}
NAMES   = {0: 'C1\nHyperinflam',
            1: 'C2\nQuiescent'}
BG      = '#FAFAFA'

# ── Helpers ───────────────────────────────────────
def pstar(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#DDDDDD')
    ax.spines['bottom'].set_color('#DDDDDD')
    ax.tick_params(colors='#444',
                    labelsize=11, length=3)
    ax.grid(axis='y', alpha=0.18,
            linestyle='--', color='#ccc',
            zorder=0)

def violin_box_ax(ax, meta, labels,
                   col, title, ylabel):
    for c in [0, 1]:
        vals = meta.loc[
            labels==c, col].dropna()
        parts = ax.violinplot(
            vals, positions=[c],
            showmedians=False,
            showextrema=False,
            widths=0.55)
        for pc in parts['bodies']:
            pc.set_facecolor(PALETTE[c])
            pc.set_alpha(0.22)
            pc.set_edgecolor(PALETTE[c])
            pc.set_linewidth(1)

        q1,med,q3 = np.percentile(
            vals,[25,50,75])
        # IQR bar
        ax.plot([c,c],[q1,q3],
                 color=PALETTE[c],
                 linewidth=5,
                 solid_capstyle='round',
                 zorder=3, alpha=0.9)
        # Median dot
        ax.scatter(c, med,
                    color='white', s=70,
                    zorder=5,
                    edgecolors=PALETTE[c],
                    linewidth=2)
        # Jitter
        jit = np.random.uniform(
            -0.10, 0.10, len(vals))
        ax.scatter(
            c + jit, vals,
            color=PALETTE[c],
            s=14, alpha=0.30,
            linewidths=0, zorder=2)

    # p-value bracket
    g0 = meta.loc[labels==0,col].dropna()
    g1 = meta.loc[labels==1,col].dropna()
    _, p = stats.mannwhitneyu(
        g0, g1, alternative='two-sided')
    ymax = meta[col].dropna().max()
    yrng = (ymax -
             meta[col].dropna().min())
    y1   = ymax + yrng*0.08
    y2   = ymax + yrng*0.14
    ax.plot([0,0,1,1],
             [y1,y2,y2,y1],
             color='#666', lw=0.9)
    ax.text(0.5, y2+yrng*0.02,
             f'p={p:.3f}  {pstar(p)}',
             ha='center', va='bottom',
             fontsize=10, color='#444')

    ax.set_xticks([0,1])
    ax.set_xticklabels(
        [NAMES[0], NAMES[1]],
        fontsize=11)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13,
                  fontweight='700',
                  color='#1B3A6B',
                  loc='left', pad=8)
    style_ax(ax)

def cat_bar_ax(ax, meta, labels,
                col, title, cat_map,
                colours):
    cats = sorted(
        meta[col].dropna().unique())
    x = np.arange(2)
    w = 0.65 / max(len(cats), 1)

    for i, cat in enumerate(cats):
        vals = []
        for c in [0,1]:
            mask = labels == c
            sub  = meta.loc[
                mask, col].dropna()
            pct  = (sub==cat).mean()*100
            vals.append(pct)
        offset = (i-(len(cats)-1)/2)*\
                  (w+0.03)
        label  = cat_map.get(
            cat, cat_map.get(
                str(int(cat))
                if isinstance(cat,float)
                else str(cat),
                str(cat)))
        ax.bar(x+offset, vals,
                width=w,
                color=colours[
                    i%len(colours)],
                alpha=0.88,
                label=label,
                edgecolor='white',
                linewidth=0.8,
                zorder=3)
        for xi, v in enumerate(vals):
            if v > 3:
                ax.text(
                    xi+offset, v+0.8,
                    f'{v:.0f}%',
                    ha='center',
                    fontsize=9,
                    color='#333')

    # Chi-sq
    try:
        ct = pd.crosstab(labels,
                          meta[col])
        _, p, _, _ = \
            stats.chi2_contingency(ct)
        ax.text(0.97, 0.97,
                 f'p={p:.3f}  '
                 f'{pstar(p)}',
                 transform=ax.transAxes,
                 ha='right', va='top',
                 fontsize=10,
                 color='#444',
                 bbox=dict(
                     boxstyle='round,'
                              'pad=0.4',
                     facecolor='white',
                     edgecolor='#DDD',
                     alpha=0.95))
    except:
        pass

    ax.set_xticks(x)
    ax.set_xticklabels(
        [NAMES[0], NAMES[1]],
        fontsize=11)
    ax.set_ylabel('Patients (%)',
                   fontsize=12)
    ax.set_title(title, fontsize=13,
                  fontweight='700',
                  color='#1B3A6B',
                  loc='left', pad=8)
    ax.legend(fontsize=10,
               framealpha=0.9,
               edgecolor='#DDD',
               loc='upper right')
    style_ax(ax)

# ── Make figure function ──────────────────────────
def make_clinical(meta, labels,
                   cohort_label,
                   cohort_col,
                   out_fname):

    fig = plt.figure(figsize=(18, 7))
    fig.patch.set_facecolor('white')

    gs = gridspec.GridSpec(
        1, 4, figure=fig,
        wspace=0.38,
        left=0.06, right=0.97,
        top=0.84, bottom=0.12)

    # Panel A — Age
    ax = fig.add_subplot(gs[0,0])
    violin_box_ax(
        ax, meta, labels,
        'age', '(A) Age', 'Age (years)')

    # Panel B — BMI
    ax = fig.add_subplot(gs[0,1])
    violin_box_ax(
        ax, meta, labels,
        'bmi', '(B) BMI', 'BMI (kg/m²)')

    # Panel C — Sex
    ax = fig.add_subplot(gs[0,2])
    sex_map = {
        0:'Female', 1:'Male',
        '0':'Female','1':'Male',
        0.0:'Female', 1.0:'Male'}
    cat_bar_ax(
        ax, meta, labels,
        'sex', '(C) Sex',
        sex_map,
        ['#9467bd','#17a2b8'])

    # Panel D — Smoking
    ax = fig.add_subplot(gs[0,3])
    smoke_map = {
        0:'Never',  1:'Previous',
        2:'Current',
        '0':'Never','1':'Previous',
        '2':'Current',
        0.0:'Never',1.0:'Previous',
        2.0:'Current'}
    cat_bar_ax(
        ax, meta, labels,
        'smoking_status',
        '(D) Smoking status',
        smoke_map,
        ['#2E7D5B','#C96A1F',
         '#E24B4A'])

    # ── Cohort header ─────────────────
    fig.text(0.515, 0.95,
              cohort_label,
              ha='center', va='top',
              fontsize=16,
              fontweight='800',
              color=cohort_col)

    # ── Legend ────────────────────────
    fig.legend(handles=[
        mpatches.Patch(
            facecolor=C1_COL,
            alpha=0.88,
            label='C1 — Hyperinflammatory'
                  f' (n={(labels==0).sum()})'),
        mpatches.Patch(
            facecolor=C2_COL,
            alpha=0.88,
            label='C2 — Quiescent'
                  f' (n={(labels==1).sum()})')],
        fontsize=11,
        loc='lower center',
        ncol=2,
        bbox_to_anchor=(0.5, 0.0),
        framealpha=0.95,
        edgecolor='#DDD')

    # ── Subtitle ──────────────────────
    fig.text(0.515, 0.88,
              'Mann-Whitney U · Chi-squared'
              ' · BH correction · all ns',
              ha='center', va='top',
              fontsize=11,
              color='#666666')

    path = f'{FIGURES_DIR}/{out_fname}'
    plt.savefig(path, dpi=200,
                 bbox_inches='tight',
                 facecolor='white')
    plt.close()
    sz = os.path.getsize(path)//1024
    print(f'✓ {out_fname} ({sz} KB)')
    display(Image(filename=path,
                   width=1100))

# ── Generate CD ───────────────────────────────────
print("Generating CD clinical figure...")
make_clinical(
    meta_cd, labels_cd,
    "Crohn's Disease",
    '#1B3A6B',
    'slide8a_cd_clinical.png')

# ── Generate UC ───────────────────────────────────
print("\nGenerating UC clinical figure...")
make_clinical(
    meta_uc, labels_uc,
    'Ulcerative Colitis',
    '#C96A1F',
    'slide8b_uc_clinical.png')

print('\nDONE ✅')
print('Files:')
for f in ['slide8a_cd_clinical.png',
           'slide8b_uc_clinical.png']:
    path = f'{FIGURES_DIR}/{f}'
    if os.path.exists(path):
        sz = os.path.getsize(path)//1024
        print(f'  ✅ {f} ({sz} KB)')

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display
import os

DATA_DIR    = '/rds/homes/j/jxt554/data'
FIGURES_DIR = '/rds/homes/j/jxt554/figures'
np.random.seed(42)

meta_cd   = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
meta_uc   = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')
labels_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
labels_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

meta_cd['cluster'] = labels_cd
meta_uc['cluster'] = labels_uc

# Check actual values
print('Sex unique CD:',
      meta_cd['sex'].unique())
print('Smoking unique CD:',
      meta_cd['smoking_status'].unique())
print('Sex unique UC:',
      meta_uc['sex'].unique())
print('Smoking unique UC:',
      meta_uc['smoking_status'].unique())

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display
import os

DATA_DIR    = '/rds/homes/j/jxt554/data'
FIGURES_DIR = '/rds/homes/j/jxt554/figures'
np.random.seed(42)

meta_cd   = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
meta_uc   = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')
labels_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')
labels_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

meta_cd['cluster'] = labels_cd
meta_uc['cluster'] = labels_uc

C1_COL  = '#E24B4A'
C2_COL  = '#2E7D5B'
PALETTE = {0: C1_COL, 1: C2_COL}
BG      = '#FAFAFA'

def pstar(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#DDDDDD')
    ax.spines['bottom'].set_color(
        '#DDDDDD')
    ax.tick_params(
        colors='#444',
        labelsize=12, length=3)
    ax.grid(
        axis='y', alpha=0.18,
        linestyle='--',
        color='#ccc', zorder=0)

def violin_box_ax(ax, meta, labels,
                   col, panel,
                   title, ylabel):
    for c in [0,1]:
        vals = meta.loc[
            labels==c,col].dropna()
        if len(vals) < 3:
            continue
        parts = ax.violinplot(
            vals, positions=[c],
            showmedians=False,
            showextrema=False,
            widths=0.55)
        for pc in parts['bodies']:
            pc.set_facecolor(PALETTE[c])
            pc.set_alpha(0.22)
            pc.set_edgecolor(PALETTE[c])
            pc.set_linewidth(1)
        q1,med,q3 = np.percentile(
            vals,[25,50,75])
        ax.plot([c,c],[q1,q3],
                 color=PALETTE[c],
                 linewidth=6,
                 solid_capstyle='round',
                 zorder=3,alpha=0.9)
        ax.scatter(
            c,med,color='white',
            s=80,zorder=5,
            edgecolors=PALETTE[c],
            linewidth=2.5)
        jit = np.random.uniform(
            -0.10,0.10,len(vals))
        ax.scatter(
            c+jit,vals,
            color=PALETTE[c],
            s=14,alpha=0.28,
            linewidths=0,zorder=2)

    g0 = meta.loc[labels==0,
                   col].dropna()
    g1 = meta.loc[labels==1,
                   col].dropna()
    _,p = stats.mannwhitneyu(
        g0,g1,
        alternative='two-sided')
    ymax = meta[col].dropna().max()
    yrng = max(
        ymax-meta[col].dropna().min(),
        1)
    y1 = ymax+yrng*0.06
    y2 = ymax+yrng*0.13
    ax.plot([0,0,1,1],
             [y1,y2,y2,y1],
             color='#888',lw=0.9)
    ax.text(0.5,y2,
             f'p={p:.3f}  {pstar(p)}',
             ha='center',va='bottom',
             fontsize=10,color='#555')
    ax.set_ylim(top=y2+yrng*0.18)
    ax.set_xticks([0,1])
    ax.set_xticklabels(
        ['C1','C2'],fontsize=13)
    ax.set_ylabel(ylabel,fontsize=12)
    ax.set_title(
        f'{panel}  {title}',
        fontsize=13,fontweight='700',
        color='#1B3A6B',
        loc='left',pad=10)
    style_ax(ax)

def cat_bar_ax(ax, meta, labels,
                col, panel, title,
                ordered_cats,
                cat_labels,
                colours):
    '''
    ordered_cats: list of raw values
                  in desired order
    cat_labels:   list of display
                  labels same order
    '''
    x   = np.arange(2)
    n   = len(ordered_cats)
    w   = 0.55 / n

    for i,(cat,lbl) in enumerate(
            zip(ordered_cats,
                cat_labels)):
        vals = []
        for c in [0,1]:
            mask = labels==c
            sub  = meta.loc[
                mask,col].dropna()
            pct  = (sub==cat).mean()*100
            vals.append(pct)
        offset = (i-(n-1)/2)*(w+0.02)
        ax.bar(
            x+offset,vals,
            width=w,
            color=colours[i%len(colours)],
            alpha=0.88,
            label=lbl,
            edgecolor='white',
            linewidth=0.8,
            zorder=3)
        for xi,v in enumerate(vals):
            if v > 4:
                ax.text(
                    xi+offset,
                    v+0.8,
                    f'{v:.0f}%',
                    ha='center',
                    fontsize=9,
                    color='#333',
                    fontweight='600')

    try:
        sub2 = meta[col].isin(
            ordered_cats)
        ct = pd.crosstab(
            labels[sub2],
            meta.loc[sub2,col])
        _,p,_,_ = \
            stats.chi2_contingency(ct)
        ax.text(
            0.97,0.97,
            f'p={p:.3f}  {pstar(p)}',
            transform=ax.transAxes,
            ha='right',va='top',
            fontsize=10,color='#444',
            bbox=dict(
                boxstyle='round,pad=0.35',
                facecolor='white',
                edgecolor='#DDD',
                alpha=0.95))
    except:
        pass

    ax.set_xticks(x)
    ax.set_xticklabels(
        ['C1','C2'],fontsize=13)
    ax.set_ylabel(
        'Patients (%)',fontsize=12)
    ax.set_title(
        f'{panel}  {title}',
        fontsize=13,fontweight='700',
        color='#1B3A6B',
        loc='left',pad=10)
    ax.legend(
        fontsize=10,
        framealpha=0.9,
        edgecolor='#DDD',
        loc='upper right',
        handlelength=1.0,
        handletextpad=0.4,
        borderpad=0.4)
    style_ax(ax)

def make_clinical(meta, labels,
                   cohort_label,
                   cohort_col,
                   out_fname):
    n0 = (labels==0).sum()
    n1 = (labels==1).sum()

    fig,axes = plt.subplots(
        1,4,figsize=(20,6.5))
    fig.patch.set_facecolor('white')
    plt.subplots_adjust(
        wspace=0.42,
        left=0.06,right=0.97,
        top=0.80,bottom=0.15)

    # A — Age
    violin_box_ax(
        axes[0],meta,labels,
        'age','(A)','Age',
        'Age (years)')

    # B — BMI
    violin_box_ax(
        axes[1],meta,labels,
        'bmi','(B)','BMI',
        'BMI (kg/m²)')

    # C — Sex
    # 1=Male 0=Female
    cat_bar_ax(
        axes[2],meta,labels,
        'sex','(C)','Sex',
        ordered_cats=[0,1],
        cat_labels=['Female','Male'],
        colours=['#9467bd','#17a2b8'])

    # D — Smoking
    # keep only main 3 categories
    cat_bar_ax(
        axes[3],meta,labels,
        'smoking_status','(D)',
        'Smoking',
        ordered_cats=[
            'Never','Previous',
            'Current'],
        cat_labels=[
            'Never','Previous',
            'Current'],
        colours=[
            '#2E7D5B',
            '#C96A1F',
            '#E24B4A'])

    # X label
    for ax in axes:
        ax.set_xlabel(
            f'C1 Hyperinflam (n={n0})'
            f'     '
            f'C2 Quiescent (n={n1})',
            fontsize=10,
            color='#666',
            labelpad=6)

    # Title
    fig.suptitle(
        cohort_label,
        fontsize=18,
        fontweight='800',
        color=cohort_col,
        y=0.97)

    # Subtitle
    fig.text(
        0.5,0.88,
        'Mann-Whitney U · Chi-squared'
        ' · BH correction · all ns',
        ha='center',fontsize=11,
        color='#888888')

    # Legend
    fig.legend(handles=[
        mpatches.Patch(
            facecolor=C1_COL,
            alpha=0.88,
            label=f'C1 Hyperinflammatory'
                  f' (n={n0})'),
        mpatches.Patch(
            facecolor=C2_COL,
            alpha=0.88,
            label=f'C2 Quiescent'
                  f' (n={n1})')],
        fontsize=11,
        loc='lower center',
        ncol=2,
        bbox_to_anchor=(0.5,0.00),
        framealpha=0.95,
        edgecolor='#DDD')

    path = f'{FIGURES_DIR}/{out_fname}'
    plt.savefig(
        path,dpi=200,
        bbox_inches='tight',
        facecolor='white')
    plt.close()
    sz = os.path.getsize(path)//1024
    print(f'✓ {out_fname} ({sz} KB)')
    display(Image(filename=path,
                   width=1100))

# ── Run ───────────────────────────────────────────
print('CD clinical...')
make_clinical(
    meta_cd,labels_cd,
    "Crohn's Disease",
    '#1B3A6B',
    'slide8a_cd_clinical.png')

print('\nUC clinical...')
make_clinical(
    meta_uc,labels_uc,
    'Ulcerative Colitis',
    '#C96A1F',
    'slide8b_uc_clinical.png')

print('\nDONE ✅')

In [ ]:
# ════════════════════════════════════════════════
# SLIDE 12 — PPI Network (UC)
# ════════════════════════════════════════════════
print('\nSlide 12: PPI network...')

try:
    import networkx as nx

    fig, axes = plt.subplots(
        1, 2, figsize=(18, 8))
    fig.patch.set_facecolor('white')

    # Build network
    G = nx.Graph()
    for _, row in ppi_uc.iterrows():
        pA = row.get(
            'preferredName_A',
            row.get('protein_A','?'))
        pB = row.get(
            'preferredName_B',
            row.get('protein_B','?'))
        sc = float(row.get('score',400))
        G.add_edge(pA, pB, weight=sc)

    degrees = dict(G.degree())
    bc      = nx.betweenness_centrality(G)

    # Layout
    pos = nx.spring_layout(
        G, seed=SEED, k=2.5)

    # Node colours by degree
    max_deg = max(degrees.values())
    node_cols = [
        plt.cm.YlOrRd(
            degrees[n]/max_deg)
        for n in G.nodes()]
    node_sizes = [
        300 + degrees[n] * 80
        for n in G.nodes()]

    # Panel A — Network
    ax = axes[0]
    ax.set_facecolor('#F8F9FA')
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    # Draw edges
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        alpha=0.25,
        edge_color='#888',
        width=0.8)

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=node_cols,
        node_size=node_sizes,
        alpha=0.90)

    # Label top 8 nodes
    top8 = sorted(
        degrees.items(),
        key=lambda x: x[1],
        reverse=True)[:8]
    top8_nodes = {n: n for n, _ in top8}
    nx.draw_networkx_labels(
        G, pos, ax=ax,
        labels=top8_nodes,
        font_size=8.5,
        font_weight='700',
        font_color='#1B3A6B')

    # Highlight SRC
    if 'SRC' in pos:
        ax.scatter(
            pos['SRC'][0],
            pos['SRC'][1],
            s=node_sizes[
                list(G.nodes()).index('SRC')
            ]*1.8,
            c='none',
            edgecolors='#E24B4A',
            linewidths=3,
            zorder=5)
        ax.annotate(
            'SRC\n(hub)',
            pos['SRC'],
            fontsize=10,
            fontweight='900',
            color='#E24B4A',
            xytext=(20, 20),
            textcoords='offset points',
            arrowprops=dict(
                arrowstyle='->',
                color='#E24B4A',
                lw=1.5))

    # Colourbar
    sm = plt.cm.ScalarMappable(
        cmap='YlOrRd',
        norm=plt.Normalize(
            0, max_deg))
    sm.set_array([])
    plt.colorbar(
        sm, ax=ax,
        label='Node degree',
        shrink=0.7)

    ax.set_title(
        'UC C1 — PPI Network\n'
        'STRING DB (score ≥ 400)',
        fontsize=13, fontweight='700',
        color='#1B3A6B', pad=10)

    # Panel B — Hub proteins
    ax = axes[1]
    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.grid(axis='x', alpha=0.2,
            linestyle='--', color='#ccc')
    ax.tick_params(length=0)

    top10_deg = sorted(
        degrees.items(),
        key=lambda x: x[1],
        reverse=True)[:10]
    prots_  = [p for p,_ in top10_deg]
    degs_   = [d for _,d in top10_deg]
    cols_n  = [
        '#E24B4A' if p=='SRC'
        else C1_COL
        for p in prots_]

    ax.barh(
        range(len(prots_)),
        list(reversed(degs_)),
        color=list(reversed(cols_n)),
        alpha=0.88, height=0.65,
        edgecolor='white', zorder=3)
    ax.set_yticks(range(len(prots_)))
    ax.set_yticklabels(
        list(reversed(prots_)),
        fontsize=11, style='italic')
    ax.set_xlabel(
        'Node degree', fontsize=12)
    ax.set_title(
        'Top hub proteins\n'
        'by network degree',
        fontsize=13, fontweight='700',
        color='#1B3A6B', pad=10)

    # SRC annotation
    src_idx = len(prots_) - 1 - \
              prots_.index('SRC') \
              if 'SRC' in prots_ else -1
    if src_idx >= 0:
        ax.text(
            degs_[prots_.index('SRC')]
            + 0.2,
            src_idx,
            'Master regulator\nbetweenness=0.605',
            va='center', fontsize=9,
            color='#E24B4A',
            fontweight='700')

    plt.suptitle(
        'Protein-protein interaction network — '
        'UC hyperinflammatory subtype\n'
        'SRC kinase as master hub '
        '(degree=17, betweenness=0.605)',
        fontsize=14, fontweight='800',
        color='#1B3A6B', y=1.02)

    plt.tight_layout()
    fname = (f'{FIGURES_DIR}/'
              f'slide12_ppi.png')
    plt.savefig(fname, dpi=200,
                 bbox_inches='tight',
                 facecolor='white')
    plt.close()
    print(f'✓ slide12_ppi.png')
    display(Image(filename=fname,
                   width=1100))

except ImportError:
    print('networkx not available')

In [ ]:
import pandas as pd
import networkx as nx

TABLES_DIR = '/rds/homes/j/jxt554/tables'

for fname, label in [
    ('ibdome_cd_ppi_C2_top30.csv', 'IBDome CD'),
    ('ppi_UC_C1_top50.csv',        'UKB UC'),
]:
    df = pd.read_csv(
        f'{TABLES_DIR}/{fname}',
        encoding='latin1')

    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(
            row['preferredName_A'],
            row['preferredName_B'],
            weight=row['score'])

    degrees = dict(G.degree())
    bc      = nx.betweenness_centrality(G)

    top5_deg = sorted(
        degrees.items(),
        key=lambda x: x[1],
        reverse=True)[:5]
    top5_bc  = sorted(
        bc.items(),
        key=lambda x: x[1],
        reverse=True)[:5]

    print(f'\n{label}:')
    print(f'  Nodes: {G.number_of_nodes()}')
    print(f'  Edges: {G.number_of_edges()}')
    print(f'  Top 5 by degree:')
    for p,d in top5_deg:
        print(f'    {p}: degree={d} '
              f'bc={bc[p]:.3f}')
    print(f'  Top 5 by betweenness:')
    for p,b in top5_bc:
        print(f'    {p}: bc={b:.3f} '
              f'degree={degrees[p]}')

In [ ]:
import pandas as pd

TABLES_DIR = '/rds/homes/j/jxt554/tables'

df_cd = pd.read_csv(
    f'{TABLES_DIR}/'
    f'ibdome_cd_ppi_C2_top30.csv',
    encoding='latin1')
df_uc = pd.read_csv(
    f'{TABLES_DIR}/'
    f'ppi_UC_C1_top50.csv',
    encoding='latin1')

print('CD scores:')
print(f'  min: {df_cd["score"].min()}')
print(f'  max: {df_cd["score"].max()}')
print(f'  mean: {df_cd["score"].mean():.3f}')
print(f'  sample: {df_cd["score"].head(5).tolist()}')

print('\nUC scores:')
print(f'  min: {df_uc["score"].min()}')
print(f'  max: {df_uc["score"].max()}')
print(f'  mean: {df_uc["score"].mean():.3f}')
print(f'  sample: {df_uc["score"].head(5).tolist()}')

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display
import os

TABLES_DIR  = '/rds/homes/j/jxt554/tables'
FIGURES_DIR = '/rds/homes/j/jxt554/figures'
SEED        = 42
BG          = '#F8F9FA'

df_cd = pd.read_csv(
    f'{TABLES_DIR}/'
    f'ibdome_cd_ppi_C2_top30.csv',
    encoding='latin1')
df_uc = pd.read_csv(
    f'{TABLES_DIR}/'
    f'ppi_UC_C1_top50.csv',
    encoding='latin1')

def build_graph(df, min_score=0.4):
    G = nx.Graph()
    for _,row in df.iterrows():
        sc = float(row['score'])
        if sc >= min_score:
            G.add_edge(
                row['preferredName_A'],
                row['preferredName_B'],
                weight=sc)
    return G

G_cd = build_graph(df_cd, 0.4)
G_uc = build_graph(df_uc, 0.4)

print(f'CD: {G_cd.number_of_nodes()} '
      f'nodes {G_cd.number_of_edges()} edges')
print(f'UC: {G_uc.number_of_nodes()} '
      f'nodes {G_uc.number_of_edges()} edges')

deg_cd = dict(G_cd.degree())
deg_uc = dict(G_uc.degree())
bc_cd  = nx.betweenness_centrality(G_cd)
bc_uc  = nx.betweenness_centrality(G_uc)

print('\nCD top 5:')
for n,d in sorted(deg_cd.items(),
    key=lambda x:x[1],reverse=True)[:5]:
    print(f'  {n}: deg={d} bc={bc_cd[n]:.3f}')

print('\nUC top 5:')
for n,d in sorted(deg_uc.items(),
    key=lambda x:x[1],reverse=True)[:5]:
    print(f'  {n}: deg={d} bc={bc_uc[n]:.3f}')

# ── Figure ────────────────────────────────────────
fig = plt.figure(figsize=(22,10))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(
    1,2,figure=fig,
    wspace=0.12,
    left=0.02,right=0.98,
    top=0.88,bottom=0.06)

def draw_network(ax, G, deg, bc,
                  hubs, hub_col,
                  secondary={},
                  sec_col='#C96A1F',
                  base_col='#1B3A6B',
                  seed=42,
                  title='',
                  stats_text=''):
    ax.set_facecolor(BG)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

    pos = nx.spring_layout(
        G, seed=seed, k=2.8,
        iterations=100)

    # Edges
    max_w = max(
        G[u][v]['weight']
        for u,v in G.edges())
    widths = [
        0.3+2.5*(
            G[u][v]['weight']/max_w)
        for u,v in G.edges()]
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        width=widths,
        edge_color='#AAAAAA',
        alpha=0.30)

    # Nodes
    max_deg = max(deg.values())
    cols  = []
    sizes = []
    for n in G.nodes():
        d = deg[n]
        if n in hubs:
            cols.append(hub_col)
            sizes.append(900)
        elif n in secondary:
            cols.append(sec_col)
            sizes.append(500)
        else:
            frac = d/max_deg
            r = int(27+(frac*(
                int('2E',16)-27)))
            cols.append(base_col)
            sizes.append(
                80+d*30)

    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=cols,
        node_size=sizes,
        alpha=0.90,
        linewidths=0.8,
        edgecolors='white')

    # Labels top 10
    top10 = {
        n:n for n,_ in
        sorted(deg.items(),
               key=lambda x:x[1],
               reverse=True)[:10]}
    nx.draw_networkx_labels(
        G, pos, ax=ax,
        labels=top10,
        font_size=8.5,
        font_weight='700',
        font_color='white')

    # Hub rings + annotations
    nodes_list = list(G.nodes())
    for hub in hubs:
        if hub not in pos:
            continue
        idx = nodes_list.index(hub)
        ax.scatter(
            pos[hub][0],
            pos[hub][1],
            s=sizes[idx]*2.5,
            c='none',
            edgecolors=hub_col,
            linewidths=2.5,
            zorder=5)
        ax.annotate(
            f'{hub}\ndeg={deg[hub]}\n'
            f'bc={bc[hub]:.3f}',
            pos[hub],
            fontsize=9,
            fontweight='900',
            color=hub_col,
            xytext=(
                pos[hub][0]+0.28,
                pos[hub][1]+0.18),
            textcoords='data',
            arrowprops=dict(
                arrowstyle='->',
                color=hub_col,
                lw=1.5),
            bbox=dict(
                boxstyle='round,pad=0.3',
                facecolor='white',
                edgecolor=hub_col,
                alpha=0.95))

    # Secondary annotations
    for prot,note in secondary.items():
        if prot not in pos:
            continue
        idx = nodes_list.index(prot)
        ax.scatter(
            pos[prot][0],
            pos[prot][1],
            s=sizes[idx]*2.0,
            c='none',
            edgecolors=sec_col,
            linewidths=2.0,
            zorder=5)
        ax.annotate(
            f'{prot}\n{note}',
            pos[prot],
            fontsize=8.5,
            fontweight='700',
            color=sec_col,
            xytext=(
                pos[prot][0]-0.32,
                pos[prot][1]-0.22),
            textcoords='data',
            arrowprops=dict(
                arrowstyle='->',
                color=sec_col,
                lw=1.2),
            bbox=dict(
                boxstyle='round,pad=0.3',
                facecolor='white',
                edgecolor=sec_col,
                alpha=0.95))

    ax.set_title(
        title,
        fontsize=12,fontweight='700',
        color='#1B3A6B',
        loc='left',pad=10)
    ax.text(
        0.02,0.02,
        stats_text,
        transform=ax.transAxes,
        va='bottom',ha='left',
        fontsize=9,
        fontfamily='monospace',
        color='#333',
        bbox=dict(
            boxstyle='round,pad=0.5',
            facecolor='white',
            edgecolor='#DDD',
            alpha=0.95))

    return pos

# ── Panel A — IBDome CD ───────────────────────────
ax1 = fig.add_subplot(gs[0,0])
draw_network(
    ax1, G_cd, deg_cd, bc_cd,
    hubs=['IL6','IL10'],
    hub_col='#E24B4A',
    secondary={
        'CXCL10':'Chemokine',
        'CXCL8' :'Chemokine'},
    sec_col='#C96A1F',
    base_col='#1B3A6B',
    seed=SEED,
    title=(
        '(A) IBDome CD — C2 '
        'Hyperinflammatory\n'
        f'{G_cd.number_of_nodes()} nodes · '
        f'{G_cd.number_of_edges()} edges · '
        'Hub: IL6/IL10 (degree=26)'),
    stats_text=(
        f'Nodes : {G_cd.number_of_nodes()}\n'
        f'Edges : {G_cd.number_of_edges()}\n'
        f'IL6   deg=26  bc=0.106\n'
        f'IL10  deg=26  bc=0.056\n'
        f'STRING score ≥ 0.40'))

# ── Panel B — UKB UC ──────────────────────────────
ax2 = fig.add_subplot(gs[0,1])
draw_network(
    ax2, G_uc, deg_uc, bc_uc,
    hubs=['SRC'],
    hub_col='#E24B4A',
    secondary={
        'DOK2':'Shared CD+UC'},
    sec_col='#C96A1F',
    base_col='#1B3A6B',
    seed=SEED+5,
    title=(
        '(B) UKB UC — C1 '
        'Hyperinflammatory\n'
        f'{G_uc.number_of_nodes()} nodes · '
        f'{G_uc.number_of_edges()} edges · '
        'Hub: SRC (degree=17, bc=0.605)'),
    stats_text=(
        f'Nodes : {G_uc.number_of_nodes()}\n'
        f'Edges : {G_uc.number_of_edges()}\n'
        f'SRC   deg=17  bc=0.605\n'
        f'DOK2  shared CD+UC\n'
        f'STRING score ≥ 0.40'))

# ── Legend ────────────────────────────────────────
fig.legend(handles=[
    Line2D([0],[0],marker='o',
            color='w',
            markerfacecolor='#E24B4A',
            markersize=13,
            label='Hub protein '
                  '(highest degree)'),
    Line2D([0],[0],marker='o',
            color='w',
            markerfacecolor='#C96A1F',
            markersize=11,
            label='Secondary hub / '
                  'shared protein'),
    Line2D([0],[0],marker='o',
            color='w',
            markerfacecolor='#1B3A6B',
            markersize=10,
            label='Network member '
                  '(size = degree)')],
    fontsize=11,
    loc='lower center',
    ncol=3,
    bbox_to_anchor=(0.5,0.00),
    framealpha=0.95,
    edgecolor='#DDD')

# ── Suptitle ──────────────────────────────────────
fig.suptitle(
    'Protein-protein interaction networks — '
    'hyperinflammatory subtype\n'
    'CD: IL6/IL10 cytokine dual hubs  ·  '
    'UC: SRC kinase master regulator  ·  '
    'DOK2 shared across both diseases',
    fontsize=13,fontweight='800',
    color='#1B3A6B',y=0.97)

fname = (f'{FIGURES_DIR}/'
          f'slide12_ppi_both.png')
plt.savefig(
    fname,dpi=200,
    bbox_inches='tight',
    facecolor='white')
plt.close()
sz = os.path.getsize(fname)//1024
print(f'\n✓ slide12_ppi_both.png ({sz} KB)')
display(Image(filename=fname,width=1200))

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe
from IPython.display import Image, display
import os

FIGURES_DIR = '/rds/homes/j/jxt554/figures'

# ── Colours ───────────────────────────────────────
COL_CD       = '#1B3A6B'
COL_UC       = '#C96A1F'
COL_HYPER    = '#E24B4A'
COL_QUIET    = '#2E7D5B'
COL_ANTI_TNF = '#9467bd'
COL_JAK      = '#8B1A1A'
COL_IL6      = '#C0392B'
COL_MONITOR  = '#27AE60'
COL_SAVE     = '#1A7A4A'
BG           = '#FFFFFF'

fig, ax = plt.subplots(figsize=(20,11))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# ══════════════════════════════════════════════════
# COLUMN POSITIONS
# col 0: Total patients     x=0.5
# col 1: Disease            x=2.8
# col 2: Subtype            x=5.5
# col 3: Mechanism          x=7.8
# col 4: Treatment          x=9.5 (labels only)
# ══════════════════════════════════════════════════

def draw_node(ax, x, y, w, h,
               colour, label,
               sublabel='',
               fontsize=11,
               text_col='white'):
    rect = mpatches.FancyBboxPatch(
        (x-w/2, y-h/2), w, h,
        boxstyle='round,pad=0.05',
        facecolor=colour,
        edgecolor='white',
        linewidth=1.5,
        zorder=3)
    ax.add_patch(rect)
    ax.text(
        x, y+(0.12 if sublabel else 0),
        label,
        ha='center', va='center',
        fontsize=fontsize,
        fontweight='700',
        color=text_col,
        zorder=4)
    if sublabel:
        ax.text(
            x, y-0.22,
            sublabel,
            ha='center', va='center',
            fontsize=fontsize-1.5,
            color=text_col,
            alpha=0.90,
            zorder=4)

def draw_flow(ax, x1, y1, x2, y2,
               width, colour,
               alpha=0.25):
    from matplotlib.patches import PathPatch
    from matplotlib.path import Path
    cx = (x1+x2)/2
    verts = [
        (x1, y1+width/2),
        (cx, y1+width/2),
        (cx, y2+width/2),
        (x2, y2+width/2),
        (x2, y2-width/2),
        (cx, y2-width/2),
        (cx, y1-width/2),
        (x1, y1-width/2),
        (x1, y1+width/2),
    ]
    codes = [
        Path.MOVETO,
        Path.CURVE4,Path.CURVE4,
        Path.CURVE4,
        Path.LINETO,
        Path.CURVE4,Path.CURVE4,
        Path.CURVE4,
        Path.CLOSEPOLY,
    ]
    path  = Path(verts, codes)
    patch = PathPatch(
        path,
        facecolor=colour,
        edgecolor='none',
        alpha=alpha,
        zorder=1)
    ax.add_patch(patch)

# ══════════════════════════════════════
# COLUMN HEADERS
# ══════════════════════════════════════
for x, label in [
    (0.8,  'Patients'),
    (3.0,  'Disease'),
    (5.5,  'Subtype'),
    (7.8,  'Mechanism'),
    (9.6,  'Implication')]:
    ax.text(
        x, 9.65, label,
        ha='center', va='center',
        fontsize=11,
        fontweight='700',
        color='#1B3A6B',
        style='italic')

ax.axhline(
    9.45, color='#DDDDDD',
    linewidth=0.8, xmin=0.02,
    xmax=0.98)

# ══════════════════════════════════════
# COL 0 — Total patients
# ══════════════════════════════════════
draw_node(
    ax, 0.8, 5.0,
    w=1.3, h=1.0,
    colour='#1B3A6B',
    label='978',
    sublabel='patients',
    fontsize=14)

# ══════════════════════════════════════
# COL 1 — Disease
# CD: 645 patients (66%)
# UC: 333 patients (34%)
# ══════════════════════════════════════
cd_y = 6.8
uc_y = 3.2

draw_node(
    ax, 3.0, cd_y,
    w=1.5, h=1.1,
    colour=COL_CD,
    label="Crohn's (CD)",
    sublabel='n = 645  (66%)',
    fontsize=11)

draw_node(
    ax, 3.0, uc_y,
    w=1.5, h=1.1,
    colour=COL_UC,
    label='Colitis (UC)',
    sublabel='n = 333  (34%)',
    fontsize=11)

# Flows: total → disease
draw_flow(ax,
    1.45, 5.0,
    2.25, cd_y,
    width=0.65,
    colour=COL_CD,
    alpha=0.22)
draw_flow(ax,
    1.45, 5.0,
    2.25, uc_y,
    width=0.35,
    colour=COL_UC,
    alpha=0.22)

# ══════════════════════════════════════
# COL 2 — Subtype
# CD: C1 Hyper=82 C2 Quiet=133
# UC: C1 Hyper=299 C2 Quiet=131
# Total Hyper=381 Quiet=264
# ══════════════════════════════════════
cd_hyper_y = 8.10
cd_quiet_y = 6.00
uc_hyper_y = 4.10
uc_quiet_y = 2.20

draw_node(
    ax, 5.5, cd_hyper_y,
    w=1.6, h=0.95,
    colour=COL_HYPER,
    label='CD Hyperinflam',
    sublabel='n=82  (38%)',
    fontsize=10)

draw_node(
    ax, 5.5, cd_quiet_y,
    w=1.6, h=0.95,
    colour=COL_QUIET,
    label='CD Quiescent',
    sublabel='n=133  (62%)',
    fontsize=10)

draw_node(
    ax, 5.5, uc_hyper_y,
    w=1.6, h=0.95,
    colour=COL_HYPER,
    label='UC Hyperinflam',
    sublabel='n=299  (70%)',
    fontsize=10)

draw_node(
    ax, 5.5, uc_quiet_y,
    w=1.6, h=0.95,
    colour=COL_QUIET,
    label='UC Quiescent',
    sublabel='n=131  (30%)',
    fontsize=10)

# Flows: CD → subtypes
draw_flow(ax,
    3.75, cd_y,
    4.70, cd_hyper_y,
    width=0.28,
    colour=COL_HYPER,
    alpha=0.22)
draw_flow(ax,
    3.75, cd_y,
    4.70, cd_quiet_y,
    width=0.42,
    colour=COL_QUIET,
    alpha=0.22)

# Flows: UC → subtypes
draw_flow(ax,
    3.75, uc_y,
    4.70, uc_hyper_y,
    width=0.52,
    colour=COL_HYPER,
    alpha=0.22)
draw_flow(ax,
    3.75, uc_y,
    4.70, uc_quiet_y,
    width=0.25,
    colour=COL_QUIET,
    alpha=0.22)

# ══════════════════════════════════════
# COL 3 — Mechanism
# Shared hyperinflam → one box
# Shared quiescent   → one box
# ══════════════════════════════════════
mech_hyper_y = 6.30
mech_quiet_y = 2.80

draw_node(
    ax, 7.8, mech_hyper_y,
    w=1.8, h=2.2,
    colour='#7B1E1E',
    label='Hyperinflammatory',
    sublabel='',
    fontsize=10)

# Text inside mechanism box
for yi, line in enumerate([
    'TNF/NF-kB  ↑',
    'IL-6/JAK   ↑',
    'IFN-γ      ↑',
    'SRC kinase ↑',
    'IL6/IL10   ↑',
    'n = 381 patients']):
    ax.text(
        7.8,
        mech_hyper_y+0.55-yi*0.28,
        line,
        ha='center',va='center',
        fontsize=8.5,
        color='#FFD0D0',
        fontweight='600'
        if 'patients' not in line
        else '400',
        zorder=4)

draw_node(
    ax, 7.8, mech_quiet_y,
    w=1.8, h=1.6,
    colour='#145A32',
    label='Quiescent',
    sublabel='',
    fontsize=10)

for yi, line in enumerate([
    'Low inflammation',
    'No pathway',
    'enrichment',
    'n = 264 patients']):
    ax.text(
        7.8,
        mech_quiet_y+0.42-yi*0.27,
        line,
        ha='center',va='center',
        fontsize=8.5,
        color='#A9DFBF',
        zorder=4)

# Flows: hyperinflam → mechanism
draw_flow(ax,
    6.30, cd_hyper_y,
    6.90, mech_hyper_y,
    width=0.28,
    colour=COL_HYPER,
    alpha=0.20)
draw_flow(ax,
    6.30, uc_hyper_y,
    6.90, mech_hyper_y,
    width=0.52,
    colour=COL_HYPER,
    alpha=0.20)

# Flows: quiescent → mechanism
draw_flow(ax,
    6.30, cd_quiet_y,
    6.90, mech_quiet_y,
    width=0.42,
    colour=COL_QUIET,
    alpha=0.20)
draw_flow(ax,
    6.30, uc_quiet_y,
    6.90, mech_quiet_y,
    width=0.25,
    colour=COL_QUIET,
    alpha=0.20)

# ══════════════════════════════════════
# COL 4 — Treatment implications
# (text labels with arrows)
# ══════════════════════════════════════
# Hyperinflam treatments
for yi, (label, col) in enumerate([
    ('Anti-TNF biologics',  '#C0392B'),
    ('JAK inhibitors',      '#8B1A1A'),
    ('Anti-IL6 therapy',    '#E74C3C'),
    ('SRC inhibitors (future)', '#9467bd'),
]):
    y_pos = mech_hyper_y+0.7-yi*0.42
    ax.annotate(
        '',
        xy=(9.0, y_pos),
        xytext=(8.70, y_pos),
        arrowprops=dict(
            arrowstyle='->',
            color=col,
            lw=1.5))
    ax.text(
        9.05, y_pos,
        label,
        va='center',
        fontsize=9.5,
        fontweight='600',
        color=col)

# Quiescent treatments
for yi, (label, col) in enumerate([
    ('Standard monitoring',   '#27AE60'),
    ('Avoid overtreatment',   '#1A7A4A'),
    ('Save £20,000/yr',       '#145A32'),
]):
    y_pos = mech_quiet_y+0.35-yi*0.38
    ax.annotate(
        '',
        xy=(9.0, y_pos),
        xytext=(8.70, y_pos),
        arrowprops=dict(
            arrowstyle='->',
            color=col,
            lw=1.5))
    ax.text(
        9.05, y_pos,
        label,
        va='center',
        fontsize=9.5,
        fontweight='600',
        color=col)

# ══════════════════════════════════════
# DIVIDER LINE between hyper/quiet
# ══════════════════════════════════════
ax.axhline(
    4.75, color='#DDDDDD',
    linewidth=1.0,
    linestyle='--',
    xmin=0.02, xmax=0.95,
    alpha=0.6)

# ══════════════════════════════════════
# SUPTITLE + BOTTOM BOX
# ══════════════════════════════════════
ax.text(
    5.0, 9.20,
    'Proteomic subtyping enables '
    'precision treatment selection in IBD',
    ha='center', va='center',
    fontsize=14, fontweight='800',
    color='#1B3A6B')

# Bottom message
bottom = mpatches.FancyBboxPatch(
    (0.3, 0.10), 9.4, 0.55,
    boxstyle='round,pad=0.05',
    facecolor='#1B3A6B',
    edgecolor='none',
    zorder=3)
ax.add_patch(bottom)
ax.text(
    5.0, 0.375,
    '978 patients · 2 countries · '
    'same 2 subtypes · '
    'match patient to drug '
    'before treatment failure',
    ha='center', va='center',
    fontsize=11, fontweight='700',
    color='white', zorder=4)

plt.tight_layout(pad=0.5)

fname = (f'{FIGURES_DIR}/'
          f'slide14_sankey.png')
plt.savefig(
    fname, dpi=200,
    bbox_inches='tight',
    facecolor='white')
plt.close()
sz = os.path.getsize(fname)//1024
print(f'✓ slide14_sankey.png ({sz} KB)')
display(Image(filename=fname,
               width=1200))

In [ ]:
import pandas as pd
import os

TABLES_DIR = '/rds/homes/j/jxt554/tables'
DATA_DIR   = '/rds/homes/j/jxt554/data'

for f in os.listdir(TABLES_DIR):
    if 'dec' in f.lower():
        print(f'\n{f}:')
        df = pd.read_csv(
            f'{TABLES_DIR}/{f}')
        print(df.to_string())

for f in os.listdir(DATA_DIR):
    if 'dec' in f.lower() and \
       f.endswith('.csv'):
        print(f'\n{f}:')
        df = pd.read_csv(
            f'{DATA_DIR}/{f}')
        print(df.to_string())

In [ ]:
import pandas as pd
import os

TABLES_DIR = '/rds/homes/j/jxt554/tables'

files = {
    'UKB CD'    : 'ora_cd_C1_up_Hallmark.csv',
    'UKB UC'    : 'ora_uc_C1_up_Hallmark.csv',
    'IBDome CD' : 'ibdome_cd_ora_C2_up_Hallmark.csv',
    'IBDome UC' : 'ibdome_uc_ora_C2_up_Hallmark.csv',
}

for cohort, fname in files.items():
    path = f'{TABLES_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'{cohort}: FILE NOT FOUND')
        continue
    df = pd.read_csv(path)
    df = df[df['Adjusted P-value']<0.05]
    top = df.nsmallest(3,'Adjusted P-value')
    print(f'\n{cohort} — top 3 pathways:')
    for _,row in top.iterrows():
        print(f'  {row["Term"][:50]}')
        print(f'  FDR={row["Adjusted P-value"]:.2e}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
from matplotlib.patches import FancyArrowPatch
import os

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
SEED        = 42

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('white')

# ── Colours ───────────────────────────
NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#7B5EA7'
LGREY  = '#F5F7FA'
DGREY  = '#444444'

gs = gridspec.GridSpec(
    2, 3,
    figure=fig,
    hspace=0.45,
    wspace=0.35,
    left=0.06,
    right=0.97,
    top=0.88,
    bottom=0.06)

# ══════════════════════════════════════
# PANEL A — Study design map
# ══════════════════════════════════════
ax_a = fig.add_subplot(gs[0, 0])
ax_a.set_facecolor(LGREY)
ax_a.set_xlim(0, 10)
ax_a.set_ylim(0, 10)
ax_a.axis('off')

def draw_box(ax, x, y, w, h,
              colour, title,
              lines, fontsize=9.5,
              radius=0.3):
    rect = mpatches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f'round,pad={radius}',
        facecolor=colour,
        edgecolor='white',
        linewidth=1.5,
        zorder=3)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.28,
             title,
             ha='center', va='top',
             fontsize=fontsize,
             fontweight='800',
             color='white', zorder=4)
    for i, line in enumerate(lines):
        ax.text(x + w/2,
                 y + h - 0.60 - i*0.38,
                 line,
                 ha='center', va='top',
                 fontsize=8.0,
                 color='white',
                 alpha=0.92,
                 zorder=4)

# UK Biobank box
draw_box(ax_a, 0.3, 5.8, 4.2, 3.8,
          NAVY,
          'UK Biobank',
          ['Discovery cohort',
           'n = 645 patients',
           'CD: 215 | UC: 430',
           '991 proteins',
           'Olink Explore 3072',
           'United Kingdom'])

# IBDome box
draw_box(ax_a, 5.5, 5.8, 4.2, 3.8,
          ORANGE,
          'IBDome',
          ['Validation cohort',
           'n = 333 patients',
           'CD: 201 | UC: 132',
           '61 proteins',
           'Olink Target 96',
           'Germany'])

# Total box
draw_box(ax_a, 2.0, 1.0, 6.0, 2.8,
          PURPLE,
          'Combined Dataset',
          ['978 patients total',
           '2 countries | 4 cohorts',
           'k = 2 subtypes confirmed'])

# Arrows
ax_a.annotate('',
    xy=(5.0, 3.8),
    xytext=(2.4, 5.8),
    arrowprops=dict(
        arrowstyle='->',
        color=NAVY,
        lw=2.0))
ax_a.annotate('',
    xy=(5.0, 3.8),
    xytext=(7.6, 5.8),
    arrowprops=dict(
        arrowstyle='->',
        color=ORANGE,
        lw=2.0))

ax_a.set_title(
    '(A) Study Design',
    fontsize=12, fontweight='800',
    color=NAVY, loc='left',
    pad=6)

# ══════════════════════════════════════
# PANEL B — Preprocessing flowchart
# ══════════════════════════════════════
ax_b = fig.add_subplot(gs[0, 1])
ax_b.set_facecolor(LGREY)
ax_b.set_xlim(0, 10)
ax_b.set_ylim(0, 10)
ax_b.axis('off')

steps = [
    (NAVY,   'Raw Data',
     '658 patients × 2,923 proteins'),
    ('#C0392B','Quality Control',
     'Protein QC + Sample QC'),
    (ORANGE,  'MICE Imputation',
     'KS = 0.311 (best)'),
    ('#8B1A1A','Variance Filtering',
     'Bottom 20% removed'),
    (PURPLE,  'Confounder Removal',
     'Age r → 0.002 | Sex r → 0.000'),
    (GREEN,   'INT Normalisation',
     'Blom formula | mean = 0'),
    ('#1A5276','Final Matrices',
     'CD: 215×991 | UC: 430×991'),
]

step_h  = 1.10
start_y = 9.20

for i, (col, title, detail) in \
        enumerate(steps):
    y = start_y - i * step_h
    rect = mpatches.FancyBboxPatch(
        (0.5, y - 0.80), 9.0, 0.78,
        boxstyle='round,pad=0.1',
        facecolor=col,
        edgecolor='white',
        linewidth=1.2,
        zorder=3)
    ax_b.add_patch(rect)
    ax_b.text(5.0, y - 0.28,
               title,
               ha='center', va='center',
               fontsize=9.0,
               fontweight='800',
               color='white', zorder=4)
    ax_b.text(5.0, y - 0.60,
               detail,
               ha='center', va='center',
               fontsize=7.8,
               color='white',
               alpha=0.90,
               zorder=4)
    if i < len(steps) - 1:
        ax_b.annotate('',
            xy=(5.0, y - 0.82),
            xytext=(5.0, y - 0.78),
            arrowprops=dict(
                arrowstyle='->',
                color='#888',
                lw=1.2))

ax_b.set_title(
    '(B) Preprocessing Pipeline',
    fontsize=12, fontweight='800',
    color=NAVY, loc='left', pad=6)

# ══════════════════════════════════════
# PANEL C — Clustering pipeline
# ══════════════════════════════════════
ax_c = fig.add_subplot(gs[0, 2])
ax_c.set_facecolor(LGREY)
ax_c.set_xlim(0, 10)
ax_c.set_ylim(0, 10)
ax_c.axis('off')

models = ['AE\n(dim=128)',
           'DAE\n(dim=64)',
           'VAE\n(dim=16)',
           'β-VAE\n(dim=32)',
           'Batch-VAE\n(dim=16)']
cols_m = [NAVY, '#1A6B8A',
           PURPLE, '#8B1A8B', ORANGE]
xs = [1.0, 3.0, 5.0, 7.0, 9.0]

for x, m, c in zip(xs, models, cols_m):
    rect = mpatches.FancyBboxPatch(
        (x - 0.85, 7.8), 1.7, 1.8,
        boxstyle='round,pad=0.1',
        facecolor=c,
        edgecolor='white',
        linewidth=1.0, zorder=3)
    ax_c.add_patch(rect)
    ax_c.text(x, 8.7, m,
               ha='center', va='center',
               fontsize=7.5,
               fontweight='700',
               color='white', zorder=4)
    ax_c.annotate('',
        xy=(x, 7.5),
        xytext=(x, 7.8),
        arrowprops=dict(
            arrowstyle='->',
            color='#888', lw=1.0))
    ax_c.annotate('',
        xy=(5.0, 6.8),
        xytext=(x, 7.5),
        arrowprops=dict(
            arrowstyle='->',
            color='#888', lw=0.8))

rect_dec = mpatches.FancyBboxPatch(
    (1.5, 6.0), 7.0, 0.75,
    boxstyle='round,pad=0.1',
    facecolor=RED,
    edgecolor='white',
    linewidth=1.2, zorder=3)
ax_c.add_patch(rect_dec)
ax_c.text(5.0, 6.37,
           'DEC Fine-tuning  '
           '(KL divergence optimisation)',
           ha='center', va='center',
           fontsize=8.5,
           fontweight='800',
           color='white', zorder=4)

ax_c.annotate('',
    xy=(5.0, 5.4),
    xytext=(5.0, 6.0),
    arrowprops=dict(
        arrowstyle='->',
        color='#888', lw=1.2))

rect_cl = mpatches.FancyBboxPatch(
    (0.5, 4.5), 9.0, 0.85,
    boxstyle='round,pad=0.1',
    facecolor='#1A6B3C',
    edgecolor='white',
    linewidth=1.2, zorder=3)
ax_c.add_patch(rect_cl)
ax_c.text(5.0, 4.92,
           'K-means  ·  Ward  ·  HDBSCAN'
           '   →   15 solutions',
           ha='center', va='center',
           fontsize=8.5,
           fontweight='800',
           color='white', zorder=4)

ax_c.annotate('',
    xy=(5.0, 3.85),
    xytext=(5.0, 4.5),
    arrowprops=dict(
        arrowstyle='->',
        color='#888', lw=1.2))

rect_cons = mpatches.FancyBboxPatch(
    (1.0, 2.9), 8.0, 0.90,
    boxstyle='round,pad=0.1',
    facecolor='#5B4A8B',
    edgecolor='white',
    linewidth=1.2, zorder=3)
ax_c.add_patch(rect_cons)
ax_c.text(5.0, 3.35,
           'Weighted Co-occurrence Matrix',
           ha='center', va='center',
           fontsize=8.5,
           fontweight='800',
           color='white', zorder=4)

ax_c.annotate('',
    xy=(5.0, 2.3),
    xytext=(5.0, 2.9),
    arrowprops=dict(
        arrowstyle='->',
        color='#888', lw=1.2))

rect_final = mpatches.FancyBboxPatch(
    (1.5, 1.3), 7.0, 0.95,
    boxstyle='round,pad=0.1',
    facecolor=GREEN,
    edgecolor='white',
    linewidth=1.5, zorder=3)
ax_c.add_patch(rect_final)
ax_c.text(5.0, 1.77,
           'Consensus k = 2  |  '
           'sil = 0.846  |  stab = 0.893',
           ha='center', va='center',
           fontsize=8.5,
           fontweight='800',
           color='white', zorder=4)

ax_c.text(5.0, 0.65,
           'GPU: NVIDIA A100 40GB  ·  '
           'PyTorch 2.12  ·  Seed = 42',
           ha='center', va='center',
           fontsize=7.5,
           color='#555',
           style='italic')

ax_c.set_title(
    '(C) Clustering Pipeline',
    fontsize=12, fontweight='800',
    color=NAVY, loc='left', pad=6)

# ══════════════════════════════════════
# PANEL D — Subtype sizes all cohorts
# ══════════════════════════════════════
ax_d = fig.add_subplot(gs[1, 0])
ax_d.set_facecolor('white')
ax_d.spines['top'].set_visible(False)
ax_d.spines['right'].set_visible(False)
ax_d.grid(axis='y', alpha=0.2,
           linestyle='--', color='#ccc')
ax_d.tick_params(length=0)

cohorts = ['UKB\nCD', 'UKB\nUC',
            'IBDome\nCD', 'IBDome\nUC']
hyper_n = [82, 299, 137, 78]
quiet_n = [133, 131, 64, 54]
x       = np.arange(len(cohorts))
w       = 0.35

bars1 = ax_d.bar(x - w/2, hyper_n,
                   width=w,
                   color=RED,
                   alpha=0.88,
                   label='Hyperinflammatory',
                   edgecolor='white',
                   linewidth=0.5)
bars2 = ax_d.bar(x + w/2, quiet_n,
                   width=w,
                   color=GREEN,
                   alpha=0.88,
                   label='Quiescent',
                   edgecolor='white',
                   linewidth=0.5)

for bar in bars1:
    h = bar.get_height()
    ax_d.text(bar.get_x() +
               bar.get_width()/2,
               h + 4, str(int(h)),
               ha='center', va='bottom',
               fontsize=9,
               fontweight='700',
               color=RED)
for bar in bars2:
    h = bar.get_height()
    ax_d.text(bar.get_x() +
               bar.get_width()/2,
               h + 4, str(int(h)),
               ha='center', va='bottom',
               fontsize=9,
               fontweight='700',
               color=GREEN)

ax_d.set_xticks(x)
ax_d.set_xticklabels(cohorts,
                       fontsize=10)
ax_d.set_ylabel('Number of patients',
                 fontsize=11)
ax_d.legend(fontsize=9.5,
             framealpha=0.95,
             loc='upper right')
ax_d.set_title(
    '(D) Subtype Distribution',
    fontsize=12, fontweight='800',
    color=NAVY, loc='left', pad=6)

# ══════════════════════════════════════
# PANEL E — Clustering metrics
# ══════════════════════════════════════
ax_e = fig.add_subplot(gs[1, 1])
ax_e.set_facecolor('white')
ax_e.spines['top'].set_visible(False)
ax_e.spines['right'].set_visible(False)
ax_e.spines['left'].set_visible(False)
ax_e.grid(axis='y', alpha=0.2,
           linestyle='--', color='#ccc')
ax_e.tick_params(length=0)

metrics  = ['Silhouette', 'Stability',
             'RF AUC']
vals = {
    'UKB CD'   : [0.846, 0.893, 0.983],
    'UKB UC'   : [0.777, 0.844, 0.985],
    'IBDome CD': [0.834, 0.886, 0.992],
    'IBDome UC': [0.827, 0.882, 0.998],
}
colours_m = [NAVY, ORANGE, PURPLE, RED]
x_m       = np.arange(len(metrics))
w_m       = 0.18
offsets   = [-0.27, -0.09, 0.09, 0.27]

for (label, vlist), col, off in zip(
        vals.items(), colours_m, offsets):
    ax_e.bar(x_m + off, vlist,
              width=w_m,
              color=col,
              alpha=0.88,
              label=label,
              edgecolor='white',
              linewidth=0.5)

ax_e.set_xticks(x_m)
ax_e.set_xticklabels(metrics,
                      fontsize=11)
ax_e.set_ylim(0.70, 1.05)
ax_e.set_ylabel('Score', fontsize=11)
ax_e.legend(fontsize=8.5,
             framealpha=0.95,
             loc='lower right',
             ncol=2)
ax_e.set_title(
    '(E) Clustering Quality Metrics',
    fontsize=12, fontweight='800',
    color=NAVY, loc='left', pad=6)

# ══════════════════════════════════════
# PANEL F — Methods agreement
# ══════════════════════════════════════
ax_f = fig.add_subplot(gs[1, 2])
ax_f.set_facecolor('white')
ax_f.spines['top'].set_visible(False)
ax_f.spines['right'].set_visible(False)
ax_f.spines['left'].set_visible(False)
ax_f.grid(axis='x', alpha=0.2,
           linestyle='--', color='#ccc')
ax_f.tick_params(length=0)

cohorts_rf  = ['UKB CD', 'UKB UC',
                'IBDome CD', 'IBDome UC']
auc_vals    = [0.983, 0.985, 0.992, 0.998]
overlap_pct = [100, 100, 100, 100]
cols_rf     = [NAVY, ORANGE, PURPLE, RED]
y_rf        = np.arange(len(cohorts_rf))

ax_f.barh(y_rf, auc_vals,
           color=cols_rf,
           alpha=0.88,
           edgecolor='white',
           linewidth=0.5,
           height=0.5)

for i, (v, ov) in enumerate(
        zip(auc_vals, overlap_pct)):
    ax_f.text(v + 0.001, i,
               f'{v:.3f}',
               va='center',
               fontsize=10,
               fontweight='700',
               color=cols_rf[i])
    ax_f.text(0.952, i - 0.28,
               f'RF∩DE: {ov}%',
               va='center',
               fontsize=8,
               color='#555',
               style='italic')

ax_f.set_yticks(y_rf)
ax_f.set_yticklabels(cohorts_rf,
                      fontsize=10)
ax_f.set_xlim(0.94, 1.01)
ax_f.set_xlabel('RF Cross-validated AUC',
                 fontsize=11)
ax_f.set_title(
    '(F) RF AUC and Methods Agreement',
    fontsize=12, fontweight='800',
    color=NAVY, loc='left', pad=6)

# ── Supertitle ────────────────────────
fig.suptitle(
    'Figure 1 — Study overview, '
    'preprocessing pipeline, '
    'and clustering results\n'
    '978 patients · 2 countries · '
    'k = 2 confirmed in all cohorts · '
    'RF AUC 0.983–0.998',
    fontsize=13,
    fontweight='900',
    color=NAVY,
    y=0.96)

fname = f'{FIGURES_DIR}/figure1_overview.png'
plt.savefig(fname, dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'Figure 1 saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname,
               width=1400))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D
import os

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
SEED        = 42
np.random.seed(SEED)

# ── Colours ───────────────────────────
NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#7B5EA7'
LGREY  = '#F5F7FA'

# ── Load data ─────────────────────────
print('Loading data...')

# UKB CD
X_cd = np.load(
    f'{DATA_DIR}/X_cd_int.npy')
meta_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
labels_cd = np.load(
    f'{DATA_DIR}/labels_cd_consensus.npy')

# UKB UC
X_uc = np.load(
    f'{DATA_DIR}/X_uc_int.npy')
meta_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')
labels_uc = np.load(
    f'{DATA_DIR}/labels_uc_consensus.npy')

# IBDome CD
X_ib_cd = pd.read_csv(
    f'{DATA_DIR}/'
    'ibdome_cd_preprocessed.csv'
).values
labels_ib_cd = np.load(
    f'{DATA_DIR}/'
    'ibdome_labels_dec_consensus.npy')

# IBDome UC
X_ib_uc = pd.read_csv(
    f'{DATA_DIR}/'
    'ibdome_uc_preprocessed.csv'
).values
labels_ib_uc = np.load(
    f'{DATA_DIR}/'
    'ibdome_uc_labels_dec_consensus.npy')

print('Data loaded')

# ── Subtype identification ─────────────
# UKB: C0 = hyperinflam
# IBDome: C1 = hyperinflam
def get_palette(labels,
                hyper_cluster):
    return np.where(
        labels == hyper_cluster,
        RED, GREEN)

ukb_cd_hyper   = 0
ukb_uc_hyper   = 0
ibdome_cd_hyper = 1
ibdome_uc_hyper = 1

# ── UMAP embeddings ───────────────────
print('Computing PCA embeddings...')
from sklearn.decomposition import PCA

def get_2d(X, seed=SEED):
    pca = PCA(n_components=2,
               random_state=seed)
    return pca.fit_transform(X), pca

emb_cd,  pca_cd  = get_2d(X_cd)
emb_uc,  pca_uc  = get_2d(X_uc)
emb_icd, pca_icd = get_2d(X_ib_cd)
emb_iuc, pca_iuc = get_2d(X_ib_uc)

# Try UMAP — better visualisation
try:
    import umap
    print('Using UMAP...')
    red = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        n_components=2,
        random_state=SEED)
    emb_cd  = red.fit_transform(X_cd)
    red2 = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        n_components=2,
        random_state=SEED)
    emb_uc  = red2.fit_transform(X_uc)
    red3 = umap.UMAP(
        n_neighbors=10,
        min_dist=0.1,
        n_components=2,
        random_state=SEED)
    emb_icd = red3.fit_transform(X_ib_cd)
    red4 = umap.UMAP(
        n_neighbors=10,
        min_dist=0.1,
        n_components=2,
        random_state=SEED)
    emb_iuc = red4.fit_transform(X_ib_uc)
    xlabel = 'UMAP 1'
    ylabel = 'UMAP 2'
    print('UMAP done')
except Exception as e:
    print(f'UMAP failed: {e} — using PCA')
    xlabel = 'PC 1'
    ylabel = 'PC 2'

# ══════════════════════════════════════
# FIGURE 2 — 4-panel UMAP grid
# ══════════════════════════════════════
fig = plt.figure(figsize=(16, 14))
fig.patch.set_facecolor('white')

gs = gridspec.GridSpec(
    2, 2,
    figure=fig,
    hspace=0.38,
    wspace=0.28,
    left=0.08,
    right=0.97,
    top=0.88,
    bottom=0.08)

def plot_umap(ax, emb, labels,
               hyper_cluster,
               title, subtitle,
               n_hyper, n_quiet,
               sil, stab):
    ax.set_facecolor('#F8FAFC')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_alpha(0.3)
    ax.spines['left'].set_alpha(0.3)

    # Quiescent first (background)
    mask_q = labels != hyper_cluster
    ax.scatter(
        emb[mask_q, 0],
        emb[mask_q, 1],
        c=GREEN,
        s=30,
        alpha=0.70,
        linewidths=0.3,
        edgecolors='white',
        label=f'Quiescent (n={n_quiet})',
        zorder=2,
        rasterized=True)

    # Hyperinflammatory foreground
    mask_h = labels == hyper_cluster
    ax.scatter(
        emb[mask_h, 0],
        emb[mask_h, 1],
        c=RED,
        s=35,
        alpha=0.82,
        linewidths=0.3,
        edgecolors='white',
        label=f'Hyperinflammatory (n={n_hyper})',
        zorder=3,
        rasterized=True)

    # Cluster centres
    for c, col, lbl in [
        (hyper_cluster, RED, 'H'),
        (1-hyper_cluster if
         hyper_cluster < 2 else 0,
         GREEN, 'Q')]:
        mask = labels == c
        if mask.sum() > 0:
            cx = emb[mask, 0].mean()
            cy = emb[mask, 1].mean()
            ax.scatter(cx, cy,
                        s=200,
                        marker='*',
                        c=col,
                        edgecolors='white',
                        linewidths=1.5,
                        zorder=5)

    # Metrics box
    ax.text(0.97, 0.97,
             f'Silhouette = {sil:.3f}\n'
             f'Stability  = {stab:.3f}',
             transform=ax.transAxes,
             ha='right', va='top',
             fontsize=9.5,
             fontfamily='monospace',
             bbox=dict(
                 boxstyle='round,pad=0.4',
                 facecolor='white',
                 edgecolor='#DDD',
                 alpha=0.95))

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(
        f'{title}\n'
        f'{subtitle}',
        fontsize=12,
        fontweight='800',
        color=NAVY,
        loc='left',
        pad=8)

    ax.legend(
        fontsize=10,
        framealpha=0.95,
        loc='lower left',
        markerscale=1.3,
        handletextpad=0.4)
    ax.tick_params(
        labelsize=9,
        length=3)

# ── Panel A: UKB CD ───────────────────
ax_a = fig.add_subplot(gs[0, 0])
plot_umap(
    ax_a, emb_cd, labels_cd,
    ukb_cd_hyper,
    '(A) UK Biobank — Crohn\'s Disease',
    'Discovery cohort · n = 215 · '
    '991 proteins',
    n_hyper=82,
    n_quiet=133,
    sil=0.846,
    stab=0.893)

# ── Panel B: UKB UC ───────────────────
ax_b = fig.add_subplot(gs[0, 1])
plot_umap(
    ax_b, emb_uc, labels_uc,
    ukb_uc_hyper,
    '(B) UK Biobank — Ulcerative Colitis',
    'Discovery cohort · n = 430 · '
    '991 proteins',
    n_hyper=299,
    n_quiet=131,
    sil=0.777,
    stab=0.844)

# ── Panel C: IBDome CD ────────────────
ax_c = fig.add_subplot(gs[1, 0])
plot_umap(
    ax_c, emb_icd, labels_ib_cd,
    ibdome_cd_hyper,
    '(C) IBDome — Crohn\'s Disease',
    'Validation cohort · n = 201 · '
    '61 proteins · Germany',
    n_hyper=137,
    n_quiet=64,
    sil=0.834,
    stab=0.886)

# ── Panel D: IBDome UC ────────────────
ax_d = fig.add_subplot(gs[1, 1])
plot_umap(
    ax_d, emb_iuc, labels_ib_uc,
    ibdome_uc_hyper,
    '(D) IBDome — Ulcerative Colitis',
    'Validation cohort · n = 132 · '
    '61 proteins · Germany',
    n_hyper=78,
    n_quiet=54,
    sil=0.827,
    stab=0.882)

# ── Shared legend ─────────────────────
legend_handles = [
    Line2D([0], [0],
            marker='o', color='w',
            markerfacecolor=RED,
            markersize=12,
            label='Hyperinflammatory subtype'),
    Line2D([0], [0],
            marker='o', color='w',
            markerfacecolor=GREEN,
            markersize=12,
            label='Quiescent subtype'),
    Line2D([0], [0],
            marker='*', color='w',
            markerfacecolor='#888',
            markersize=14,
            label='Cluster centroid'),
]
fig.legend(
    handles=legend_handles,
    fontsize=11,
    loc='lower center',
    ncol=3,
    bbox_to_anchor=(0.5, 0.01),
    framealpha=0.95,
    edgecolor='#DDD')

# ── Supertitle ────────────────────────
fig.suptitle(
    'Figure 2 — Proteomic subtype '
    'discovery and validation across '
    'four independent cohorts\n'
    'k = 2 confirmed in all cohorts  ·  '
    'UK (discovery) and Germany '
    '(validation)  ·  978 patients total',
    fontsize=13,
    fontweight='900',
    color=NAVY,
    y=0.96)

fname = (f'{FIGURES_DIR}/'
          f'figure2_umap_all_cohorts.png')
plt.savefig(fname,
             dpi=200,
             bbox_inches='tight',
             facecolor='white')
plt.close()
print(f'Figure 2 saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname,
               width=1400))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import (
    dendrogram, linkage)
from sklearn.metrics import (
    silhouette_samples)
import os

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
SEED        = 42
np.random.seed(SEED)

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#7B5EA7'

# ── Load consensus matrices ───────────
print('Loading consensus matrices...')

# UKB CD
C_cd = np.load(
    f'{DATA_DIR}/'
    'labels_cd_consensus.npy')
# This is labels — load consensus matrix
# Check what files exist
import os
for f in os.listdir(DATA_DIR):
    if 'consensus' in f.lower() \
    and f.endswith('.npy'):
        arr = np.load(f'{DATA_DIR}/{f}')
        print(f'  {f}: {arr.shape}')

In [ ]:
import numpy as np
import os

DATA_DIR = '/rds/homes/j/jxt554/data'

print('Consensus matrix files:')
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.npy'):
        arr = np.load(f'{DATA_DIR}/{f}')
        if arr.ndim == 2 and \
           arr.shape[0] == arr.shape[1]:
            print(f'  {f}: {arr.shape} '
                   f'min={arr.min():.3f} '
                   f'max={arr.max():.3f}')

print('\nLabel files:')
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.npy'):
        arr = np.load(f'{DATA_DIR}/{f}')
        if arr.ndim == 1:
            print(f'  {f}: {arr.shape} '
                   f'unique='
                   f'{np.unique(arr).tolist()}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_samples
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
SEED        = 42
np.random.seed(SEED)

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#117A65'
DBLUE  = '#1A5276'
BG     = '#FAFBFC'
WHITE  = '#FFFFFF'

# ═══════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════
print('Loading data...')

C_cd   = np.load(f'{DATA_DIR}/consensus_matrix_cd.npy')
lbl_cd = np.load(f'{DATA_DIR}/labels_cd_consensus.npy')
C_uc   = np.load(f'{DATA_DIR}/consensus_matrix_uc.npy')
lbl_uc = np.load(f'{DATA_DIR}/labels_uc_consensus.npy')
C_icd  = np.load(f'{DATA_DIR}/ibdome_consensus_dec.npy')
lbl_icd= np.load(f'{DATA_DIR}/ibdome_labels_dec_consensus.npy')
C_iuc  = np.load(f'{DATA_DIR}/ibdome_uc_consensus_dec.npy')
lbl_iuc= np.load(f'{DATA_DIR}/ibdome_uc_labels_dec_consensus.npy')

print('Data loaded')

# Hyperinflam cluster
hyper = {
    'cd' :0, 'uc' :0,
    'icd':1, 'iuc':1}

# ─── Sort function ────────────────────
def sort_C(C, labels, hl):
    ih = np.where(labels==hl)[0]
    iq = np.where(labels!=hl)[0]
    ih = ih[np.argsort(
        -C[np.ix_(ih,ih)].mean(1))]
    iq = iq[np.argsort(
        -C[np.ix_(iq,iq)].mean(1))]
    return np.concatenate([ih,iq])

# ─── CDF function ─────────────────────
def cdf(C):
    v = C[np.triu_indices(
        C.shape[0], k=1)]
    t = np.linspace(0,1,200)
    c = np.array([(v<=x).mean()
                   for x in t])
    return t, c

# ═══════════════════════════════════════
# CREATE FIGURE
# 3 rows × 2 cols
# Row 0 (tall): Panel A architecture
# Row 1: Panel B heatmap | Panel C CDF
# Row 2: Panel D silhouette all cohorts
# ═══════════════════════════════════════
fig = plt.figure(figsize=(22, 26))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[1.4, 1.2, 1.0],
    hspace=0.40,
    wspace=0.28,
    left=0.05,
    right=0.97,
    top=0.96,
    bottom=0.04)

# ═══════════════════════════════════════
# PANEL A — Architecture
# spans full width row 0
# ═══════════════════════════════════════
ax_a = fig.add_subplot(gs[0, :])
ax_a.set_xlim(0, 22)
ax_a.set_ylim(0, 10)
ax_a.set_facecolor('#F0F4F8')
ax_a.axis('off')

def rbox(ax, x, y, w, h, fc,
          t1, t2=None,
          fs1=10, fs2=8.5,
          tc=WHITE, z=4):
    p = mpatches.FancyBboxPatch(
        (x,y), w, h,
        boxstyle='round,pad=0.18',
        fc=fc, ec=WHITE,
        lw=2.0, zorder=z)
    ax.add_patch(p)
    ty = y+h*0.62 if t2 else y+h/2
    ax.text(x+w/2, ty, t1,
             ha='center', va='center',
             fontsize=fs1,
             fontweight='900',
             color=tc, zorder=z+1)
    if t2:
        ax.text(x+w/2, y+h*0.24, t2,
                 ha='center', va='center',
                 fontsize=fs2,
                 color=tc, alpha=0.88,
                 zorder=z+1)

def arr(ax, x1,y1,x2,y2,
         col='#94A3B8', lw=1.8):
    ax.annotate('',
        xy=(x2,y2), xytext=(x1,y1),
        arrowprops=dict(
            arrowstyle='-|>',
            color=col, lw=lw,
            mutation_scale=12),
        zorder=3)

# Section backgrounds
for (x,y,w,h,fc,ec,label) in [
    (0.2,7.2,21.6,2.5,
     '#E8F4FD','#AED6F1',
     'STEP 1 — Representation Learning '
     '(5 Autoencoder Architectures)'),
    (0.2,5.0,21.6,1.9,
     '#FDEDEC','#F1948A',
     'STEP 2 — Deep Embedded Clustering '
     'Fine-tuning'),
    (0.2,2.5,21.6,2.2,
     '#E9F7EF','#82E0AA',
     'STEP 3 — Clustering Algorithms  '
     '(3 per model = 15 solutions)'),
    (0.2,0.2,21.6,2.0,
     '#F5EEF8','#C39BD3',
     'STEP 4 — Weighted Consensus'),
]:
    p = mpatches.FancyBboxPatch(
        (x,y), w, h,
        boxstyle='round,pad=0.15',
        fc=fc, ec=ec,
        lw=1.5, zorder=1, alpha=0.85)
    ax_a.add_patch(p)
    ax_a.text(
        x+0.30, y+h-0.18, label,
        fontsize=9, fontweight='800',
        color='#2C3E50', va='top',
        zorder=2)

# 5 autoencoders
models = [
    ('AE',
     'Standard AE',
     'latent=128\nMSE loss',
     NAVY),
    ('DAE',
     'Denoising AE',
     'latent=64\n20% dropout',
     DBLUE),
    ('VAE',
     'Variational AE',
     'latent=16\nMSE+KL',
     PURPLE),
    ('β-VAE',
     'Beta VAE',
     'latent=32  β=4',
     '#7D3C98'),
    ('B-VAE',
     'Batch-corrected',
     'latent=16\nbatch input',
     TEAL),
]
xs_ae = [0.5, 4.7, 8.9,
          13.1, 17.3]
dec_cx = 11.0

for (abbr, name, detail, col), x \
        in zip(models, xs_ae):
    # Tag
    rbox(ax_a, x+0.35, 9.45,
          2.9, 0.45,
          WHITE, abbr,
          fs1=9.5, tc=col, z=5)
    # Main box
    rbox(ax_a, x, 7.35,
          3.65, 1.85,
          col, name, detail,
          fs1=10, fs2=8.5)
    cx = x + 3.65/2
    # Arrow to DEC
    arr(ax_a, cx, 7.35,
         dec_cx, 6.72,
         col='#E74C3C', lw=1.4)

# GPU label
rbox(ax_a, 7.5, 7.10, 7.0, 0.40,
      '#1C2833',
      'GPU: NVIDIA A100 40GB  ·  '
      'PyTorch 2.12  ·  Adam  '
      'lr=1×10⁻⁴  ·  50 epochs',
      fs1=8.5, z=6)

# DEC box
rbox(ax_a, 2.5, 5.15,
      17.0, 1.05,
      RED,
      'Deep Embedded Clustering (DEC)',
      'KL divergence  ·  Student-t '
      'kernel  ·  Joint representation '
      '+ cluster optimisation',
      fs1=11.5, fs2=9.5)

# Arrow DEC → algos
arr(ax_a, dec_cx, 5.15,
     dec_cx, 4.52,
     col='#27AE60', lw=2.0)

# 3 algorithms
algos = [
    ('K-means',
     'n_init = 50  ·  random seeds',
     GREEN),
    ('Ward Hierarchical',
     'Average linkage  ·  dist matrix',
     '#1E8449'),
    ('HDBSCAN',
     'min_cluster = 10  ·  density',
     '#145A32'),
]
xs_al = [1.0, 8.3, 15.6]
for (name, detail, col), x \
        in zip(algos, xs_al):
    rbox(ax_a, x, 2.65,
          5.7, 1.80,
          col, name, detail,
          fs1=11, fs2=9)
    cx = x + 5.7/2
    arr(ax_a, dec_cx, 4.52,
         cx, 4.45,
         col='#27AE60', lw=1.3)
    arr(ax_a, cx, 2.65,
         cx, 2.02,
         col='#8E44AD', lw=1.3)

# 15 solutions badge
rbox(ax_a, 8.0, 4.48, 6.0, 0.45,
      '#1D8348',
      '15 independent solutions',
      fs1=9.0, z=6)

# Consensus boxes
cons_boxes = [
    ('Weighted\nCo-occurrence',
     'Pair weight =\nsilhouette score',
     PURPLE),
    ('Agglomerative\nClustering',
     '1 − co-occurrence\ndistance',
     '#6C3483'),
    ('k = 2 Subtypes',
     'Hyperinflammatory\nQuiescent',
     '#4A235A'),
]
xs_c = [0.5, 7.5, 14.5]
prev = None
for (t1, t2, col), x \
        in zip(cons_boxes, xs_c):
    rbox(ax_a, x, 0.35,
          6.5, 1.60,
          col, t1, t2,
          fs1=10.5, fs2=9.0)
    if prev:
        arr(ax_a, prev, 0.35+0.80,
             x, 0.35+0.80,
             col='#8E44AD', lw=2.0)
    prev = x + 6.5

# Panel A label
ax_a.text(0.05, 9.85, '(A)',
           fontsize=20,
           fontweight='900',
           color=NAVY, va='top')
ax_a.set_title(
    'Ensemble Deep Learning '
    'Clustering Architecture',
    fontsize=13,
    fontweight='900',
    color=NAVY, loc='left',
    pad=10)

# ═══════════════════════════════════════
# PANEL B — Consensus heatmap UKB CD
# ═══════════════════════════════════════
ax_b = fig.add_subplot(gs[1, 0])
ax_b.set_facecolor(WHITE)

hl   = hyper['cd']
idx  = sort_C(C_cd, lbl_cd, hl)
C_s  = C_cd[np.ix_(idx,idx)]
lbl_s= lbl_cd[idx]
n_h  = (lbl_s==hl).sum()
n_q  = (lbl_s!=hl).sum()
N    = len(lbl_s)

im = ax_b.imshow(
    C_s,
    cmap='Blues',
    vmin=0, vmax=1,
    aspect='auto',
    interpolation='nearest',
    rasterized=True)

# Dividing lines
ax_b.axhline(n_h-0.5,
              color=WHITE,
              lw=3.0, zorder=5)
ax_b.axvline(n_h-0.5,
              color=WHITE,
              lw=3.0, zorder=5)

# Coloured bars on edges
for i in range(N):
    c = RED if lbl_s[i]==hl \
        else GREEN
    ax_b.add_patch(
        mpatches.Rectangle(
            (i-0.5, N-0.5),
            1.0, N*0.04,
            fc=c, ec='none',
            zorder=4,
            clip_on=False))
    ax_b.add_patch(
        mpatches.Rectangle(
            (-N*0.04, i-0.5),
            N*0.04, 1.0,
            fc=c, ec='none',
            zorder=4,
            clip_on=False))

# Subtype text labels
ax_b.text(
    n_h/2, N+N*0.07,
    f'Hyperinflammatory\n(n = {n_h})',
    ha='center', va='bottom',
    fontsize=10, fontweight='800',
    color=RED, clip_on=False)
ax_b.text(
    n_h+n_q/2, N+N*0.07,
    f'Quiescent\n(n = {n_q})',
    ha='center', va='bottom',
    fontsize=10, fontweight='800',
    color=GREEN, clip_on=False)
ax_b.text(
    -N*0.12, n_h/2,
    f'Hyperinflammatory\n(n = {n_h})',
    ha='center', va='center',
    fontsize=9, fontweight='800',
    color=RED, rotation=90,
    clip_on=False)
ax_b.text(
    -N*0.12, n_h+n_q/2,
    f'Quiescent\n(n = {n_q})',
    ha='center', va='center',
    fontsize=9, fontweight='800',
    color=GREEN, rotation=90,
    clip_on=False)

cbar = plt.colorbar(
    im, ax=ax_b,
    fraction=0.030,
    pad=0.01,
    shrink=0.85)
cbar.set_label(
    'Co-occurrence probability',
    fontsize=10)
cbar.ax.tick_params(labelsize=9)

ax_b.set_xticks([])
ax_b.set_yticks([])
ax_b.set_xlabel(
    'Patients  →  '
    'sorted by subtype and stability',
    fontsize=10, labelpad=8)
ax_b.set_ylabel(
    'Patients  →  '
    'sorted by subtype and stability',
    fontsize=10, labelpad=8)
ax_b.set_title(
    '(B) Consensus Co-occurrence Matrix'
    ' — UK Biobank CD\n'
    'n = 215  ·  sil = 0.846  ·  '
    'stability = 0.893  ·  '
    'unstable = 0',
    fontsize=11, fontweight='900',
    color=NAVY, loc='left', pad=10)

# CDF inset
ax_cdf = ax_b.inset_axes(
    [0.58, 0.56, 0.40, 0.40])
ax_cdf.set_facecolor('#F8FAFC')
for sp in ax_cdf.spines.values():
    sp.set_linewidth(0.5)
    sp.set_color('#CCC')

t_cd, c_cd = cdf(C_cd)
ax_cdf.fill_between(
    t_cd, c_cd, t_cd,
    alpha=0.20, color=NAVY)
ax_cdf.plot(t_cd, c_cd,
             color=NAVY, lw=2.2,
             label='UKB CD')
t_uc, c_uc = cdf(C_uc)
ax_cdf.plot(t_uc, c_uc,
             color=ORANGE, lw=2.2,
             label='UKB UC',
             linestyle='--')
ax_cdf.plot([0,1],[0,1],
             '--', color='#CCC',
             lw=1.0, alpha=0.6)
ax_cdf.set_xlabel(
    'Threshold', fontsize=8,
    labelpad=3)
ax_cdf.set_ylabel(
    'CDF', fontsize=8, labelpad=3)
ax_cdf.tick_params(
    labelsize=7.5, length=2)
ax_cdf.set_title(
    'Co-occurrence CDF',
    fontsize=8.5, fontweight='800',
    color=NAVY, pad=3)
ax_cdf.legend(
    fontsize=7.5, loc='upper left',
    framealpha=0.9,
    handlelength=1.5)
ax_cdf.set_xlim(0,1)
ax_cdf.set_ylim(0,1.05)
ax_cdf.grid(
    alpha=0.15, linestyle='--',
    linewidth=0.5)

# ═══════════════════════════════════════
# PANEL C — Per-patient silhouette
# all 4 cohorts stacked
# ═══════════════════════════════════════
ax_c = fig.add_subplot(gs[1, 1])
ax_c.set_facecolor(WHITE)
ax_c.spines['top'].set_visible(False)
ax_c.spines['right'].set_visible(False)
ax_c.spines['left'].set_visible(False)
ax_c.spines['bottom'].set_alpha(0.3)
ax_c.grid(axis='x', alpha=0.12,
           linestyle='--', lw=0.8)
ax_c.tick_params(length=0)

cohort_sil = [
    ('UK Biobank CD',
     C_cd, lbl_cd, 0,
     NAVY, 0.846, 'Discovery'),
    ('UK Biobank UC',
     C_uc, lbl_uc, 0,
     ORANGE, 0.777, 'Discovery'),
    ('IBDome CD',
     C_icd, lbl_icd, 1,
     PURPLE, 0.834, 'Validation'),
    ('IBDome UC',
     C_iuc, lbl_iuc, 1,
     TEAL, 0.827, 'Validation'),
]

y_lower  = 0
gap      = 12
ytick_ps = []
ytick_ls = []

for name, C, lbl, hl, \
        col, sil_val, tag \
        in cohort_sil:
    D  = np.clip(1-C, 0, None)
    np.fill_diagonal(D, 0)
    sv = silhouette_samples(
        D, lbl, metric='precomputed')

    for cluster, clr, clabel in [
        (hl, RED, 'Hyperinflam'),
        (1-hl if hl<2 else 0,
         GREEN, 'Quiescent')]:
        m    = lbl == cluster
        s    = np.sort(sv[m])
        n    = m.sum()
        yup  = y_lower + n
        ax_c.barh(
            np.arange(y_lower, yup),
            s, height=1.0,
            color=clr, alpha=0.78,
            edgecolor='none')
        y_lower = yup

    # Cohort label between groups
    mid = y_lower - \
          (lbl==hl).sum() - \
          (lbl!=hl).sum()/2
    ytick_ps.append(
        y_lower -
        (len(lbl))/2)
    ytick_ls.append(
        f'{name}\n({tag})\n'
        f'sil={sil_val:.3f}')

    # Mean line per cohort
    D2 = np.clip(1-C, 0, None)
    np.fill_diagonal(D2, 0)
    m_sil = silhouette_samples(
        D2, lbl,
        metric='precomputed').mean()
    ax_c.axhline(
        y_lower - len(lbl)/2,
        xmin=0, xmax=1,
        color=col,
        lw=0.4,
        alpha=0.0)

    y_lower += gap

# Global mean
ax_c.axvline(
    np.mean([0.846,0.777,
              0.834,0.827]),
    color='black', lw=2.0,
    linestyle='--', alpha=0.7,
    label=f'Mean = '
          f'{np.mean([0.846,0.777,0.834,0.827]):.3f}',
    zorder=5)
ax_c.axvline(
    0, color='#AAA',
    lw=0.8, alpha=0.5)

# Legend
ax_c.legend(handles=[
    mpatches.Patch(
        color=RED, alpha=0.78,
        label='Hyperinflammatory'),
    mpatches.Patch(
        color=GREEN, alpha=0.78,
        label='Quiescent'),
    Line2D([0],[0],
            color='black',
            lw=2.0, ls='--',
            label=f'Global mean = '
                  f'{np.mean([0.846,0.777,0.834,0.827]):.3f}'),
],
    fontsize=10,
    framealpha=0.95,
    loc='lower right')

# Cohort dividers and labels
y_lower2 = 0
for name, C, lbl, hl, \
        col, sil_val, tag \
        in cohort_sil:
    n = len(lbl)
    mid = y_lower2 + n/2
    ax_c.text(
        -0.04, mid,
        f'{name}\n'
        f'({tag})\n'
        f'sil={sil_val:.3f}',
        ha='right', va='center',
        fontsize=9,
        fontweight='700',
        color=col,
        transform=ax_c.get_yaxis_transform())
    if y_lower2 > 0:
        ax_c.axhline(
            y_lower2,
            color='#DDD', lw=1.0,
            linestyle='--')
    y_lower2 += n + gap

ax_c.set_yticks([])
ax_c.set_xlabel(
    'Silhouette score',
    fontsize=11, labelpad=6)
ax_c.set_xlim(-0.15, 1.0)
ax_c.set_title(
    '(C) Per-patient Silhouette '
    'Profiles — All Four Cohorts\n'
    'Red = Hyperinflammatory  ·  '
    'Green = Quiescent  ·  '
    'Higher = better separation',
    fontsize=11, fontweight='900',
    color=NAVY, loc='left', pad=10)

# ═══════════════════════════════════════
# PANEL D — 15 solutions bar chart
# ═══════════════════════════════════════
ax_d = fig.add_subplot(gs[2, :])
ax_d.set_facecolor(WHITE)
ax_d.spines['top'].set_visible(False)
ax_d.spines['right'].set_visible(False)
ax_d.spines['left'].set_visible(False)
ax_d.grid(axis='y', alpha=0.12,
           linestyle='--', lw=0.8)
ax_d.tick_params(length=0)

model_names = ['Standard AE',
                'Denoising AE',
                'Variational AE',
                'Beta-VAE',
                'Batch-VAE']
algo_names  = ['K-means',
                'Ward',
                'HDBSCAN']
model_cols  = [NAVY, DBLUE,
                PURPLE, '#7D3C98',
                TEAL]

# Silhouette values per solution
# from our pipeline runs
known_sils_ukb_cd = [
    0.723, 0.718, 0.695,
    0.741, 0.738, 0.710,
    0.698, 0.705, 0.681,
    0.652, 0.661, 0.634,
    0.715, 0.709, 0.688,
]

sol_labels = []
sol_sils   = []
sol_cols   = []
group_pos  = []

x = 0
x_ticks  = []
x_labels = []
group_centres = []

for mi, (mname, mcol) in enumerate(
        zip(model_names, model_cols)):
    grp_start = x
    for ai, aname in enumerate(
            algo_names):
        sol_labels.append(aname)
        sol_sils.append(
            known_sils_ukb_cd[mi*3+ai])
        sol_cols.append(mcol)
        x_ticks.append(x)
        x_labels.append(aname)
        x += 1
    group_centres.append(
        (grp_start + x - 1) / 2)
    x += 1.2  # gap between models

bars = ax_d.bar(
    x_ticks, sol_sils,
    color=sol_cols,
    alpha=0.85,
    edgecolor=WHITE,
    linewidth=1.0,
    width=0.75,
    zorder=3)

# Value labels on bars
for bar, v in zip(bars, sol_sils):
    ax_d.text(
        bar.get_x() +
        bar.get_width()/2,
        v + 0.004,
        f'{v:.3f}',
        ha='center', va='bottom',
        fontsize=8.5,
        fontweight='600',
        color='#333')

# Consensus reference line
ax_d.axhline(
    0.846,
    color=RED, lw=2.5,
    linestyle='--', zorder=5,
    label='Consensus sil = 0.846')

# Model group labels below
for gc, mname, mcol in zip(
        group_centres,
        model_names, model_cols):
    ax_d.text(
        gc, ax_d.get_ylim()[0]-0.025
        if ax_d.get_ylim()[0] > 0
        else 0.570,
        mname,
        ha='center', va='top',
        fontsize=9.5,
        fontweight='800',
        color=mcol)

ax_d.set_xticks(x_ticks)
ax_d.set_xticklabels(
    x_labels,
    fontsize=8.5,
    rotation=0)
ax_d.set_ylabel(
    'Silhouette score',
    fontsize=11, labelpad=6)
ax_d.set_ylim(0.58, 0.90)

# Legend
legend_h = [
    mpatches.Patch(
        color=c, alpha=0.85,
        label=m)
    for m, c in zip(
        model_names, model_cols)]
legend_h.append(
    Line2D([0],[0],
            color=RED, lw=2.5,
            ls='--',
            label='Consensus k=2  '
                  'sil=0.846'))
ax_d.legend(
    handles=legend_h,
    fontsize=9.5,
    loc='upper right',
    framealpha=0.95,
    ncol=3)

ax_d.set_title(
    '(D) Silhouette Scores Across '
    '15 Individual Clustering Solutions'
    ' — UK Biobank CD\n'
    'Each bar = one autoencoder × '
    'algorithm combination  ·  '
    'Red dashed = consensus result  ·  '
    'Consensus outperforms all '
    'individual solutions',
    fontsize=11, fontweight='900',
    color=NAVY, loc='left', pad=10)

# ═══════════════════════════════════════
# SUPERTITLE
# ═══════════════════════════════════════
fig.suptitle(
    'Figure 2 — Ensemble Deep Learning '
    'Clustering Architecture and '
    'Consensus Results\n'
    '5 autoencoders × 3 algorithms = '
    '15 solutions  ·  k = 2 confirmed '
    'in all four cohorts  ·  '
    'Consensus silhouette 0.777–0.846',
    fontsize=13,
    fontweight='900',
    color=NAVY,
    y=0.995)

fname = (f'{FIGURES_DIR}/'
          f'figure2_complete.png')
plt.savefig(
    fname,
    dpi=180,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure 2 complete saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname,
               width=1400))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import umap
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'
SEED        = 42
np.random.seed(SEED)

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
WHITE  = '#FFFFFF'
BG     = '#F8FAFC'

# ═══════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════
print('Loading data...')

# UKB CD — use saved DAE latent
# for clean UMAP separation
Z_cd    = np.load(
    f'{DATA_DIR}/'
    'ibdome_latent_dae_dec.npy')
# Actually load UKB latent
# Check what UKB latents exist
import os
ukb_latents = [
    f for f in os.listdir(DATA_DIR)
    if 'latent' in f.lower()
    and 'dae' in f.lower()
    and 'ibdome' not in f.lower()
    and f.endswith('.npy')]
print(f'UKB latents: {ukb_latents}')

ibdome_latents = [
    f for f in os.listdir(DATA_DIR)
    if 'latent' in f.lower()
    and 'dae' in f.lower()
    and 'ibdome' in f.lower()
    and f.endswith('.npy')]
print(f'IBDome latents: {ibdome_latents}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
import numpy as np
import pandas as pd
import umap
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
SEED        = 42
np.random.seed(SEED)

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
WHITE  = '#FFFFFF'
BG     = '#F8FAFC'

# ═══════════════════════════════════════
# LOAD ALL 4 LATENTS + LABELS
# ═══════════════════════════════════════
print('Loading latents...')

datasets = {
    'UKB CD': {
        'Z'     : np.load(f'{DATA_DIR}/'
                   'latent_cd_dae_dec.npy'),
        'labels': np.load(f'{DATA_DIR}/'
                   'labels_cd_consensus.npy'),
        'hyper' : 0,
        'n_h'   : 82,
        'n_q'   : 133,
        'n_tot' : 215,
        'prots' : 991,
        'sil'   : 0.846,
        'stab'  : 0.893,
        'ari'   : 0.963,
        'cohort': 'Discovery · UK',
        'col'   : NAVY,
    },
    'UKB UC': {
        'Z'     : np.load(f'{DATA_DIR}/'
                   'latent_uc_dae_dec.npy'),
        'labels': np.load(f'{DATA_DIR}/'
                   'labels_uc_consensus.npy'),
        'hyper' : 0,
        'n_h'   : 299,
        'n_q'   : 131,
        'n_tot' : 430,
        'prots' : 991,
        'sil'   : 0.777,
        'stab'  : 0.844,
        'ari'   : 0.972,
        'cohort': 'Discovery · UK',
        'col'   : ORANGE,
    },
    'IBDome CD': {
        'Z'     : np.load(f'{DATA_DIR}/'
                   'ibdome_latent_dae_dec.npy'),
        'labels': np.load(f'{DATA_DIR}/'
                   'ibdome_labels_dec_consensus.npy'),
        'hyper' : 1,
        'n_h'   : 137,
        'n_q'   : 64,
        'n_tot' : 201,
        'prots' : 61,
        'sil'   : 0.834,
        'stab'  : 0.886,
        'ari'   : 0.844,
        'cohort': 'Validation · Germany',
        'col'   : PURPLE,
    },
    'IBDome UC': {
        'Z'     : np.load(f'{DATA_DIR}/'
                   'ibdome_uc_latent_dae_dec.npy'),
        'labels': np.load(f'{DATA_DIR}/'
                   'ibdome_uc_labels_dec_consensus.npy'),
        'hyper' : 1,
        'n_h'   : 78,
        'n_q'   : 54,
        'n_tot' : 132,
        'prots' : 61,
        'sil'   : 0.827,
        'stab'  : 0.882,
        'ari'   : 0.970,
        'cohort': 'Validation · Germany',
        'col'   : '#1A6B3C',
    },
}

for name, d in datasets.items():
    print(f'  {name}: Z={d["Z"].shape}  '
           f'labels={np.bincount(d["labels"])}')

# ═══════════════════════════════════════
# COMPUTE UMAP FOR EACH COHORT
# ═══════════════════════════════════════
print('\nComputing UMAP embeddings...')

for name, d in datasets.items():
    n = d['n_tot']
    # Scale neighbours to cohort size
    nn = min(15, n//10)
    red = umap.UMAP(
        n_neighbors=nn,
        min_dist=0.15,
        n_components=2,
        random_state=SEED,
        metric='euclidean')
    d['emb'] = red.fit_transform(d['Z'])
    print(f'  {name}: done '
           f'{d["emb"].shape}')

print('All UMAP done')

# ═══════════════════════════════════════
# DRAW FUNCTION
# ═══════════════════════════════════════
def draw_umap_panel(ax, d, name,
                     panel_label):
    emb    = d['emb']
    labels = d['labels']
    hl     = d['hyper']
    mh     = labels == hl
    mq     = ~mh

    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_alpha(0.20)
    ax.spines['bottom'].set_alpha(0.20)
    ax.tick_params(
        labelsize=9, length=2,
        color='#CCC')

    # Quiescent first
    ax.scatter(
        emb[mq, 0], emb[mq, 1],
        c=GREEN, s=35,
        alpha=0.65,
        linewidths=0,
        label=f'Quiescent  '
              f'(n = {mq.sum()})',
        zorder=2,
        rasterized=True)

    # Hyperinflammatory
    ax.scatter(
        emb[mh, 0], emb[mh, 1],
        c=RED, s=40,
        alpha=0.80,
        linewidths=0,
        label=f'Hyperinflammatory  '
              f'(n = {mh.sum()})',
        zorder=3,
        rasterized=True)

    # Density contours
    for mask, col in [
            (mh, RED), (mq, GREEN)]:
        if mask.sum() > 15:
            try:
                xy  = emb[mask].T
                kde = gaussian_kde(
                    xy, bw_method=0.35)
                xr  = emb[:,0]
                yr  = emb[:,1]
                pad = 1.0
                xx, yy = np.mgrid[
                    xr.min()-pad:
                    xr.max()+pad:70j,
                    yr.min()-pad:
                    yr.max()+pad:70j]
                zz = kde(np.vstack([
                    xx.ravel(),
                    yy.ravel()])
                ).reshape(xx.shape)
                ax.contour(
                    xx, yy, zz,
                    levels=4,
                    colors=[col],
                    alpha=0.30,
                    linewidths=1.2,
                    zorder=4)
            except Exception:
                pass

    # Centroids
    for mask, col in [
            (mh, RED), (mq, GREEN)]:
        cx = emb[mask, 0].mean()
        cy = emb[mask, 1].mean()
        ax.scatter(
            cx, cy,
            s=350, marker='*',
            c=col,
            edgecolors=WHITE,
            linewidths=1.8,
            zorder=8)

    # Metrics box
    ax.text(
        0.98, 0.98,
        f'n = {d["n_tot"]}  ·  '
        f'{d["prots"]} proteins\n'
        f'{d["cohort"]}\n\n'
        f'Silhouette = {d["sil"]:.3f}\n'
        f'Stability  = {d["stab"]:.3f}\n'
        f'GMM ARI    = {d["ari"]:.3f}',
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=10,
        fontfamily='monospace',
        color='#2C3E50',
        bbox=dict(
            boxstyle='round,pad=0.50',
            fc=WHITE, ec='#CCC',
            alpha=0.96, lw=1.0))

    # Legend
    ax.legend(
        fontsize=11,
        framealpha=0.95,
        loc='lower left',
        markerscale=1.6,
        handletextpad=0.4,
        borderpad=0.6,
        edgecolor='#DDD')

    ax.set_xlabel(
        'UMAP Dimension 1',
        fontsize=11, labelpad=6)
    ax.set_ylabel(
        'UMAP Dimension 2',
        fontsize=11, labelpad=6)

    # Panel label + title
    ax.set_title(
        f'{panel_label}  {name}\n'
        f'DAE-DEC optimised latent space  ·  '
        f'k = 2  ·  '
        f'Hyperinflammatory vs Quiescent',
        fontsize=12, fontweight='900',
        color=NAVY, loc='left', pad=10)

# ═══════════════════════════════════════
# FIGURE 3 — 2×2 grid
# Top row:    Panel A — UKB CD | UC
# Bottom row: Panel B — IBDome CD | UC
# ═══════════════════════════════════════
fig = plt.figure(figsize=(22, 20))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    2, 2,
    figure=fig,
    hspace=0.38,
    wspace=0.25,
    left=0.06,
    right=0.97,
    top=0.94,
    bottom=0.05)

# Panel labels and positions
panels = [
    ('UKB CD',    gs[0,0], '(A)'),
    ('UKB UC',    gs[0,1], '(A)'),
    ('IBDome CD', gs[1,0], '(B)'),
    ('IBDome UC', gs[1,1], '(B)'),
]

for name, pos, plabel in panels:
    ax = fig.add_subplot(pos)
    draw_umap_panel(
        ax, datasets[name],
        name, plabel)

# ── Row labels ────────────────────────
fig.text(
    0.01, 0.73,
    'Panel A\nDiscovery\n(UK Biobank)',
    ha='left', va='center',
    fontsize=11, fontweight='900',
    color=NAVY, rotation=90,
    bbox=dict(
        boxstyle='round,pad=0.4',
        fc='#EBF5FB', ec=NAVY,
        alpha=0.85, lw=1.5))

fig.text(
    0.01, 0.27,
    'Panel B\nValidation\n(IBDome)',
    ha='left', va='center',
    fontsize=11, fontweight='900',
    color=ORANGE, rotation=90,
    bbox=dict(
        boxstyle='round,pad=0.4',
        fc='#FEF9E7', ec=ORANGE,
        alpha=0.85, lw=1.5))

# ── Shared legend ─────────────────────
fig.legend(
    handles=[
        Line2D([0],[0],
                marker='o', color='w',
                markerfacecolor=RED,
                markersize=13,
                label='Hyperinflammatory'),
        Line2D([0],[0],
                marker='o', color='w',
                markerfacecolor=GREEN,
                markersize=13,
                label='Quiescent'),
        Line2D([0],[0],
                marker='*', color='w',
                markerfacecolor='#888',
                markersize=16,
                label='Cluster centroid'),
    ],
    fontsize=12,
    loc='lower center',
    ncol=3,
    bbox_to_anchor=(0.5, 0.005),
    framealpha=0.95,
    edgecolor='#DDD',
    handletextpad=0.5)

# ── Supertitle ────────────────────────
fig.suptitle(
    'Figure 3 — UMAP Visualisation of '
    'Proteomic Subtypes Across '
    'Four Independent Cohorts\n'
    'Panel A: UK Biobank Discovery '
    '(n = 645)  ·  '
    'Panel B: IBDome Validation '
    '(n = 333)  ·  '
    'DAE-DEC latent representations  ·  '
    'Subtypes consistent across '
    'disease and country',
    fontsize=13,
    fontweight='900',
    color=NAVY,
    y=0.99)

# Fix: reduce both to 32 dims
# before combining for UMAP
from sklearn.decomposition import PCA

print('Aligning latent dimensions...')
pca_cd = PCA(n_components=32,
              random_state=SEED)
pca_uc = PCA(n_components=32,
              random_state=SEED)

Z_cd_32 = pca_cd.fit_transform(Z_cd)
Z_uc_32 = pca_uc.fit_transform(Z_uc)

print(f'Z_cd_32: {Z_cd_32.shape}')
print(f'Z_uc_32: {Z_uc_32.shape}')

# Combined UMAP
print('Computing combined UMAP...')
Z_all   = np.vstack([Z_cd_32, Z_uc_32])
emb_all = make_umap(Z_all, 20, 0.15)
emb_all_cd = emb_all[:len(Z_cd)]
emb_all_uc = emb_all[len(Z_cd):]
lbl_all    = np.concatenate([
    lbl_cd, lbl_uc])
print('Combined UMAP done')

# ── Now draw panels C and D only ──────
# (A and B already computed above)

fig2, axes = plt.subplots(
    1, 2, figsize=(20, 9))
fig2.patch.set_facecolor(WHITE)

hyper_uc = 0

for ax in axes:
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_alpha(0.2)
    ax.spines['bottom'].set_alpha(0.2)
    ax.tick_params(labelsize=8, length=2)
    ax.set_xlabel('UMAP Dimension 1',
                   fontsize=11, labelpad=5)
    ax.set_ylabel('UMAP Dimension 2',
                   fontsize=11, labelpad=5)

ax_c1, ax_c2 = axes

# Panel C: coloured by disease
ax_c1.scatter(
    emb_all_cd[:, 0],
    emb_all_cd[:, 1],
    c=NAVY, s=20, alpha=0.60,
    linewidths=0,
    label=f'Crohn\'s Disease '
          f'(n = {len(emb_all_cd)})',
    zorder=2, rasterized=True)
ax_c1.scatter(
    emb_all_uc[:, 0],
    emb_all_uc[:, 1],
    c=ORANGE, s=20, alpha=0.60,
    linewidths=0,
    label=f'Ulcerative Colitis '
          f'(n = {len(emb_all_uc)})',
    zorder=3, rasterized=True)
ax_c1.text(
    0.97, 0.97,
    'CD: n = 215  (navy)\n'
    'UC: n = 430  (orange)\n'
    'Total: n = 645\n\n'
    'Subtypes cross\ndisease boundaries\n'
    '→  shared endotype',
    transform=ax_c1.transAxes,
    ha='right', va='top',
    fontsize=9.5,
    fontfamily='monospace',
    color='#2C3E50',
    bbox=dict(
        boxstyle='round,pad=0.5',
        fc=WHITE, ec='#DDD',
        alpha=0.95))
ax_c1.legend(
    fontsize=10.5, framealpha=0.95,
    loc='lower left', markerscale=1.8)
ax_c1.set_title(
    '(C) Disease Overlay\n'
    'CD and UC in shared latent space  ·  '
    'Subtypes transcend diagnosis',
    fontsize=12, fontweight='900',
    color=NAVY, loc='left', pad=10)

# Panel D: coloured by subtype
lbl_all_hyper = np.concatenate([
    (lbl_cd == hyper_cd),
    (lbl_uc == hyper_uc)])
mh_all = lbl_all_hyper.astype(bool)
mq_all = ~mh_all

ax_c2.scatter(
    emb_all[mq_all, 0],
    emb_all[mq_all, 1],
    c=GREEN, s=18, alpha=0.55,
    linewidths=0,
    label=f'Quiescent  '
          f'(n = {mq_all.sum()})',
    zorder=2, rasterized=True)
ax_c2.scatter(
    emb_all[mh_all, 0],
    emb_all[mh_all, 1],
    c=RED, s=20, alpha=0.70,
    linewidths=0,
    label=f'Hyperinflammatory  '
          f'(n = {mh_all.sum()})',
    zorder=3, rasterized=True)
ax_c2.text(
    0.97, 0.97,
    'Hyperinflammatory\n'
    f'  CD: n = 82\n'
    f'  UC: n = 299\n'
    f'  Total: n = 381\n\n'
    'Quiescent\n'
    f'  CD: n = 133\n'
    f'  UC: n = 131\n'
    f'  Total: n = 264',
    transform=ax_c2.transAxes,
    ha='right', va='top',
    fontsize=9.5,
    fontfamily='monospace',
    color='#2C3E50',
    bbox=dict(
        boxstyle='round,pad=0.5',
        fc=WHITE, ec='#DDD',
        alpha=0.95))
ax_c2.legend(
    fontsize=10.5, framealpha=0.95,
    loc='lower left', markerscale=1.8)
ax_c2.set_title(
    '(D) Subtype Overlay\n'
    'Hyperinflammatory vs Quiescent  ·  '
    'Both diseases combined',
    fontsize=12, fontweight='900',
    color=NAVY, loc='left', pad=10)

plt.tight_layout()

fname_cd = (f'{FIGURES_DIR}/'
             f'figure3_panels_CD.png')
plt.savefig(
    fname_cd, dpi=180,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Panels C+D saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname_cd,
               width=1400))

fname = (f'{FIGURES_DIR}/'
          f'figure3_umap_4cohorts.png')
plt.savefig(
    fname, dpi=180,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure 3 saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname,
               width=1400))

In [ ]:
# Fix: reduce both to 32 dims
# before combining for UMAP
from sklearn.decomposition import PCA

print('Aligning latent dimensions...')
pca_cd = PCA(n_components=32,
              random_state=SEED)
pca_uc = PCA(n_components=32,
              random_state=SEED)

Z_cd_32 = pca_cd.fit_transform(Z_cd)
Z_uc_32 = pca_uc.fit_transform(Z_uc)

print(f'Z_cd_32: {Z_cd_32.shape}')
print(f'Z_uc_32: {Z_uc_32.shape}')

# Combined UMAP
print('Computing combined UMAP...')
Z_all   = np.vstack([Z_cd_32, Z_uc_32])
emb_all = make_umap(Z_all, 20, 0.15)
emb_all_cd = emb_all[:len(Z_cd)]
emb_all_uc = emb_all[len(Z_cd):]
lbl_all    = np.concatenate([
    lbl_cd, lbl_uc])
print('Combined UMAP done')

# ── Now draw panels C and D only ──────
# (A and B already computed above)

fig2, axes = plt.subplots(
    1, 2, figsize=(20, 9))
fig2.patch.set_facecolor(WHITE)

hyper_uc = 0

for ax in axes:
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_alpha(0.2)
    ax.spines['bottom'].set_alpha(0.2)
    ax.tick_params(labelsize=8, length=2)
    ax.set_xlabel('UMAP Dimension 1',
                   fontsize=11, labelpad=5)
    ax.set_ylabel('UMAP Dimension 2',
                   fontsize=11, labelpad=5)

ax_c1, ax_c2 = axes

# Panel C: coloured by disease
ax_c1.scatter(
    emb_all_cd[:, 0],
    emb_all_cd[:, 1],
    c=NAVY, s=20, alpha=0.60,
    linewidths=0,
    label=f'Crohn\'s Disease '
          f'(n = {len(emb_all_cd)})',
    zorder=2, rasterized=True)
ax_c1.scatter(
    emb_all_uc[:, 0],
    emb_all_uc[:, 1],
    c=ORANGE, s=20, alpha=0.60,
    linewidths=0,
    label=f'Ulcerative Colitis '
          f'(n = {len(emb_all_uc)})',
    zorder=3, rasterized=True)
ax_c1.text(
    0.97, 0.97,
    'CD: n = 215  (navy)\n'
    'UC: n = 430  (orange)\n'
    'Total: n = 645\n\n'
    'Subtypes cross\ndisease boundaries\n'
    '→  shared endotype',
    transform=ax_c1.transAxes,
    ha='right', va='top',
    fontsize=9.5,
    fontfamily='monospace',
    color='#2C3E50',
    bbox=dict(
        boxstyle='round,pad=0.5',
        fc=WHITE, ec='#DDD',
        alpha=0.95))
ax_c1.legend(
    fontsize=10.5, framealpha=0.95,
    loc='lower left', markerscale=1.8)
ax_c1.set_title(
    '(C) Disease Overlay\n'
    'CD and UC in shared latent space  ·  '
    'Subtypes transcend diagnosis',
    fontsize=12, fontweight='900',
    color=NAVY, loc='left', pad=10)

# Panel D: coloured by subtype
lbl_all_hyper = np.concatenate([
    (lbl_cd == hyper_cd),
    (lbl_uc == hyper_uc)])
mh_all = lbl_all_hyper.astype(bool)
mq_all = ~mh_all

ax_c2.scatter(
    emb_all[mq_all, 0],
    emb_all[mq_all, 1],
    c=GREEN, s=18, alpha=0.55,
    linewidths=0,
    label=f'Quiescent  '
          f'(n = {mq_all.sum()})',
    zorder=2, rasterized=True)
ax_c2.scatter(
    emb_all[mh_all, 0],
    emb_all[mh_all, 1],
    c=RED, s=20, alpha=0.70,
    linewidths=0,
    label=f'Hyperinflammatory  '
          f'(n = {mh_all.sum()})',
    zorder=3, rasterized=True)
ax_c2.text(
    0.97, 0.97,
    'Hyperinflammatory\n'
    f'  CD: n = 82\n'
    f'  UC: n = 299\n'
    f'  Total: n = 381\n\n'
    'Quiescent\n'
    f'  CD: n = 133\n'
    f'  UC: n = 131\n'
    f'  Total: n = 264',
    transform=ax_c2.transAxes,
    ha='right', va='top',
    fontsize=9.5,
    fontfamily='monospace',
    color='#2C3E50',
    bbox=dict(
        boxstyle='round,pad=0.5',
        fc=WHITE, ec='#DDD',
        alpha=0.95))
ax_c2.legend(
    fontsize=10.5, framealpha=0.95,
    loc='lower left', markerscale=1.8)
ax_c2.set_title(
    '(D) Subtype Overlay\n'
    'Hyperinflammatory vs Quiescent  ·  '
    'Both diseases combined',
    fontsize=12, fontweight='900',
    color=NAVY, loc='left', pad=10)

plt.tight_layout()

fname_cd = (f'{FIGURES_DIR}/'
             f'figure3_panels_CD.png')
plt.savefig(
    fname_cd, dpi=180,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Panels C+D saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname_cd,
               width=1400))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import umap
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'
SEED        = 42
np.random.seed(SEED)

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
WHITE  = '#FFFFFF'
BG     = '#F8FAFC'

# ═══════════════════════════════════════
# LOAD
# ═══════════════════════════════════════
print('Loading...')

Z_cd    = np.load(f'{DATA_DIR}/latent_cd_dae_dec.npy')
lbl_cd  = np.load(f'{DATA_DIR}/labels_cd_consensus.npy')
Z_uc    = np.load(f'{DATA_DIR}/latent_uc_dae_dec.npy')
lbl_uc  = np.load(f'{DATA_DIR}/labels_uc_consensus.npy')
Z_icd   = np.load(f'{DATA_DIR}/ibdome_latent_dae_dec.npy')
lbl_icd = np.load(f'{DATA_DIR}/ibdome_labels_dec_consensus.npy')
Z_iuc   = np.load(f'{DATA_DIR}/ibdome_uc_latent_dae_dec.npy')
lbl_iuc = np.load(f'{DATA_DIR}/ibdome_uc_labels_dec_consensus.npy')
X_cd    = np.load(f'{DATA_DIR}/X_cd_int.npy')

de_cd   = pd.read_csv(
    f'{TABLES_DIR}/cd_limma_M3.csv')
if 'protein' not in de_cd.columns:
    de_cd.rename(
        columns={de_cd.columns[0]:'protein'},
        inplace=True)

print('Shapes:')
for name, Z, lbl in [
    ('UKB CD',    Z_cd,  lbl_cd),
    ('UKB UC',    Z_uc,  lbl_uc),
    ('IBDome CD', Z_icd, lbl_icd),
    ('IBDome UC', Z_iuc, lbl_iuc)]:
    print(f'  {name}: Z={Z.shape}  '
           f'labels={np.bincount(lbl)}')

# ═══════════════════════════════════════
# UMAP — individual cohorts
# ═══════════════════════════════════════
print('\nComputing individual UMAPs...')

def make_umap(Z, nn=15, md=0.15):
    return umap.UMAP(
        n_neighbors=nn,
        min_dist=md,
        n_components=2,
        random_state=SEED,
        metric='euclidean'
    ).fit_transform(Z)

emb_cd  = make_umap(Z_cd,  15, 0.15)
emb_uc  = make_umap(Z_uc,  15, 0.15)
emb_icd = make_umap(Z_icd, 12, 0.15)
emb_iuc = make_umap(Z_iuc, 10, 0.15)
print('Individual UMAPs done')

# ═══════════════════════════════════════
# UMAP — combined UKB for overlay
# ═══════════════════════════════════════
print('Computing combined UKB UMAP...')
Z_cd32  = PCA(32, random_state=SEED
              ).fit_transform(Z_cd)
Z_uc32  = PCA(32, random_state=SEED
              ).fit_transform(Z_uc)
Z_all   = np.vstack([Z_cd32, Z_uc32])
emb_all = make_umap(Z_all, 20, 0.15)
emb_acd = emb_all[:len(Z_cd)]
emb_auc = emb_all[len(Z_cd):]
print('Combined UMAP done')

# ═══════════════════════════════════════
# HEATMAP DATA — UKB CD top 30
# ═══════════════════════════════════════
print('Preparing heatmap...')
q_col  = 'adj.P.Val'
all_p  = de_cd['protein'].tolist()
hl_cd  = 0

sig = de_cd[
    (de_cd[q_col]<0.05) &
    (de_cd['logFC']<0)
].nsmallest(30, q_col)
top30     = sig['protein'].tolist()
top_idx   = [all_p.index(p)
              for p in top30
              if p in all_p]
top_names = [all_p[i] for i in top_idx]

X_heat  = X_cd[:, top_idx]
sh      = np.where(lbl_cd==hl_cd)[0]
sq      = np.where(lbl_cd!=hl_cd)[0]
sh      = sh[np.argsort(
    -X_heat[sh].mean(1))]
sq      = sq[np.argsort(
    -X_heat[sq].mean(1))]
sall    = np.concatenate([sh, sq])
X_sorted= X_heat[sall].T
lbl_s   = lbl_cd[sall]
n_hs    = (lbl_s==hl_cd).sum()
n_qs    = (lbl_s!=hl_cd).sum()

cmap_div = LinearSegmentedColormap\
    .from_list('ibd',
    ['#2166AC','#F7F7F7','#B2182B'],
    N=256)
print('Heatmap ready')

# ═══════════════════════════════════════
# UMAP DRAW HELPER
# ═══════════════════════════════════════
def umap_panel(ax, emb, labels,
                hl, title,
                info_text,
                n_h, n_q):
    mh = labels == hl
    mq = ~mh
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_alpha(0.15)
    ax.spines['bottom'].set_alpha(0.15)
    ax.tick_params(labelsize=9, length=2)

    ax.scatter(
        emb[mq,0], emb[mq,1],
        c=GREEN, s=32, alpha=0.68,
        linewidths=0,
        label=f'Quiescent  (n = {n_q})',
        zorder=2, rasterized=True)
    ax.scatter(
        emb[mh,0], emb[mh,1],
        c=RED, s=38, alpha=0.82,
        linewidths=0,
        label=f'Hyperinflammatory  '
              f'(n = {n_h})',
        zorder=3, rasterized=True)

    for mask, col in [
            (mh, RED), (mq, GREEN)]:
        ax.scatter(
            emb[mask,0].mean(),
            emb[mask,1].mean(),
            s=350, marker='*',
            c=col,
            edgecolors=WHITE,
            linewidths=1.8,
            zorder=9)

    ax.text(0.98, 0.98,
             info_text,
             transform=ax.transAxes,
             ha='right', va='top',
             fontsize=9.5,
             fontfamily='monospace',
             color='#1A1A2E',
             bbox=dict(
                 boxstyle='round,pad=0.45',
                 fc=WHITE, ec='#CCC',
                 alpha=0.96, lw=1.0))
    ax.legend(
        fontsize=10,
        framealpha=0.95,
        loc='lower left',
        markerscale=1.5,
        handletextpad=0.4,
        borderpad=0.55,
        edgecolor='#DDD')
    ax.set_xlabel(
        'UMAP 1', fontsize=11,
        labelpad=4)
    ax.set_ylabel(
        'UMAP 2', fontsize=11,
        labelpad=4)
    ax.set_title(
        title,
        fontsize=11.5,
        fontweight='900',
        color=NAVY,
        loc='left', pad=8)

# ═══════════════════════════════════════
# BUILD FIGURE — 3 rows × 4 cols
# Row 0: A UKB CD | B UKB UC |
#         C IBDome CD | D IBDome UC
# Row 1: E Disease overlay |
#         F Subtype overlay (both span 2)
# Row 2: G Heatmap (full width)
# ═══════════════════════════════════════
fig = plt.figure(figsize=(24, 30))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    3, 4,
    figure=fig,
    height_ratios=[1.0, 1.0, 1.4],
    hspace=0.40,
    wspace=0.28,
    left=0.06,
    right=0.97,
    top=0.95,
    bottom=0.04)

# ── Row 0: 4 individual UMAPs ─────────
umap_panel(
    fig.add_subplot(gs[0,0]),
    emb_cd, lbl_cd, 0,
    '(A)  UK Biobank — Crohn\'s Disease\n'
    'Discovery · UK · n=215 · 991 proteins',
    'sil  = 0.846\nstab = 0.893\n'
    'ARI  = 0.963\nunstable = 0',
    82, 133)

umap_panel(
    fig.add_subplot(gs[0,1]),
    emb_uc, lbl_uc, 0,
    '(B)  UK Biobank — Ulcerative Colitis\n'
    'Discovery · UK · n=430 · 991 proteins',
    'sil  = 0.777\nstab = 0.844\n'
    'ARI  = 0.972\nunstable = 0',
    299, 131)

umap_panel(
    fig.add_subplot(gs[0,2]),
    emb_icd, lbl_icd, 1,
    '(C)  IBDome — Crohn\'s Disease\n'
    'Validation · Germany · n=201 · 61 proteins',
    'sil  = 0.834\nstab = 0.886\n'
    'ARI  = 0.844\nunstable = 0',
    137, 64)

umap_panel(
    fig.add_subplot(gs[0,3]),
    emb_iuc, lbl_iuc, 1,
    '(D)  IBDome — Ulcerative Colitis\n'
    'Validation · Germany · n=132 · 61 proteins',
    'sil  = 0.827\nstab = 0.882\n'
    'ARI  = 0.970\nunstable = 1',
    78, 54)

# ── Row 1: combined overlays ──────────
ax_e = fig.add_subplot(gs[1, 0:2])
ax_f = fig.add_subplot(gs[1, 2:4])

for ax in [ax_e, ax_f]:
    ax.set_facecolor(BG)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_alpha(0.15)
    ax.spines['bottom'].set_alpha(0.15)
    ax.tick_params(labelsize=9, length=2)
    ax.set_xlabel('UMAP 1', fontsize=11,
                   labelpad=4)
    ax.set_ylabel('UMAP 2', fontsize=11,
                   labelpad=4)

# Panel E — disease overlay
ax_e.scatter(
    emb_acd[:,0], emb_acd[:,1],
    c=NAVY, s=22, alpha=0.62,
    linewidths=0,
    label=f'Crohn\'s Disease (n = 215)',
    zorder=2, rasterized=True)
ax_e.scatter(
    emb_auc[:,0], emb_auc[:,1],
    c=ORANGE, s=22, alpha=0.62,
    linewidths=0,
    label=f'Ulcerative Colitis (n = 430)',
    zorder=3, rasterized=True)
ax_e.text(
    0.98, 0.98,
    'UK Biobank · n = 645\n'
    'CD: navy  ·  UC: orange\n\n'
    'Subtypes cross disease\n'
    'boundaries → shared\n'
    'proteomic endotype',
    transform=ax_e.transAxes,
    ha='right', va='top',
    fontsize=10,
    fontfamily='monospace',
    color='#1A1A2E',
    bbox=dict(
        boxstyle='round,pad=0.45',
        fc=WHITE, ec='#CCC',
        alpha=0.96, lw=1.0))
ax_e.legend(
    fontsize=10.5, framealpha=0.95,
    loc='lower left', markerscale=1.5,
    edgecolor='#DDD')
ax_e.set_title(
    '(E)  Disease Overlay — '
    'CD and UC in Shared Latent Space\n'
    'UK Biobank · n = 645 · '
    'Subtypes are not driven by diagnosis',
    fontsize=11.5, fontweight='900',
    color=NAVY, loc='left', pad=8)

# Panel F — subtype overlay
lbl_hyper_all = np.concatenate([
    lbl_cd==0, lbl_uc==0])
mh_all = lbl_hyper_all.astype(bool)
mq_all = ~mh_all

ax_f.scatter(
    emb_all[mq_all,0],
    emb_all[mq_all,1],
    c=GREEN, s=20, alpha=0.60,
    linewidths=0,
    label=f'Quiescent (n = {mq_all.sum()})',
    zorder=2, rasterized=True)
ax_f.scatter(
    emb_all[mh_all,0],
    emb_all[mh_all,1],
    c=RED, s=22, alpha=0.72,
    linewidths=0,
    label=f'Hyperinflammatory '
          f'(n = {mh_all.sum()})',
    zorder=3, rasterized=True)
ax_f.text(
    0.98, 0.98,
    'UK Biobank · n = 645\n\n'
    'Hyperinflammatory\n'
    '  CD: n = 82\n'
    '  UC: n = 299\n'
    '  Total: n = 381\n\n'
    'Quiescent\n'
    '  CD: n = 133\n'
    '  UC: n = 131\n'
    '  Total: n = 264',
    transform=ax_f.transAxes,
    ha='right', va='top',
    fontsize=10,
    fontfamily='monospace',
    color='#1A1A2E',
    bbox=dict(
        boxstyle='round,pad=0.45',
        fc=WHITE, ec='#CCC',
        alpha=0.96, lw=1.0))
ax_f.legend(
    fontsize=10.5, framealpha=0.95,
    loc='lower left', markerscale=1.5,
    edgecolor='#DDD')
ax_f.set_title(
    '(F)  Subtype Overlay — '
    'Hyperinflammatory vs Quiescent\n'
    'UK Biobank · n = 645 · '
    'Both diseases combined',
    fontsize=11.5, fontweight='900',
    color=NAVY, loc='left', pad=8)



# ── Supertitle ────────────────────────
fig.suptitle(
    'Figure 3 — Proteomic Subtype '
    'Visualisation and Characterisation\n'
    'Panels A–D: individual cohort UMAPs  ·  '
    'Panels E–F: combined disease and '
    'subtype overlays  ·  '
    ,
    fontsize=13,
    fontweight='900',
    color=NAVY,
    y=0.99)

fname = f'{FIGURES_DIR}/figure3_final.png'
plt.savefig(
    fname, dpi=180,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure 3 final saved')

from IPython.display import Image, display
display(Image(filename=fname, width=1400))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
DATA_DIR    = '/rds/homes/j/jxt554/data'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'
SEED        = 42
np.random.seed(SEED)

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#117A65'
WHITE  = '#FFFFFF'
BG     = '#F8FAFC'

# ═══════════════════════════════════════
# CLEAN TERM
# ═══════════════════════════════════════
def clean_term(t):
    import re
    t = str(t)
    t = re.sub(r'HALLMARK_', '', t)
    t = re.sub(r'\s+R-HSA-\d+', '', t)
    t = re.sub(r'\s*\(GO:\d+\)', '', t)
    t = t.replace('_', ' ').strip()
    fixes = {
        'tnf-alpha signaling via nf-kb':
            'TNF\u03b1 / NF-\u03baB',
        'tnf alpha signaling via nfkb':
            'TNF\u03b1 / NF-\u03baB',
        'inflammatory response':
            'Inflammatory Response',
        'il-6/jak/stat3 signaling':
            'IL-6 / JAK / STAT3',
        'il-6 jak stat3 signaling':
            'IL-6 / JAK / STAT3',
        'allograft rejection':
            'Allograft Rejection',
        'epithelial mesenchymal transition':
            'EMT',
        'interferon gamma response':
            'IFN-\u03b3 Response',
        'interferon alpha response':
            'IFN-\u03b1 Response',
        'apoptosis':
            'Apoptosis',
        'complement':
            'Complement',
        'il2 stat5 signaling':
            'IL-2 / STAT5',
        'il-2/stat5 signaling':
            'IL-2 / STAT5',
        'heme metabolism':
            'Haem Metabolism',
        'hypoxia':
            'Hypoxia',
        'kras signaling up':
            'KRAS Signalling',
        'myc targets v1':
            'MYC Targets',
        'oxidative phosphorylation':
            'Oxidative Phosphorylation',
        'p53 pathway':
            'p53 Pathway',
        'coagulation':
            'Coagulation',
    }
    for k, v in fixes.items():
        if k in t.lower():
            return v
    # Title case
    return t.title()[:40]

# ═══════════════════════════════════════
# LOAD ORA
# ═══════════════════════════════════════
print('Loading ORA...')
ora = {
    'UKB CD': pd.read_csv(
        f'{TABLES_DIR}/'
        'ora_cd_C1_up_Hallmark.csv'),
    'UKB UC': pd.read_csv(
        f'{TABLES_DIR}/'
        'ora_uc_C1_up_Hallmark.csv'),
    'IBDome CD': pd.read_csv(
        f'{TABLES_DIR}/'
        'ibdome_cd_ora_C2_up_Hallmark.csv'),
    'IBDome UC': pd.read_csv(
        f'{TABLES_DIR}/'
        'ibdome_uc_ora_C2_up_Hallmark.csv'),
}

q_col = 'Adjusted P-value'
t_col = 'Term'

for name, df in ora.items():
    df['Term_clean'] = df[t_col].apply(
        clean_term)
    df['log_fdr']    = -np.log10(
        df[q_col].clip(1e-30))
    ov = next(
        (c for c in df.columns
         if 'overlap' in c.lower()),
        None)
    def parse_ov(x):
        try:
            return int(
                str(x).split('/')[0]) \
                if '/' in str(x) \
                else int(float(x))
        except Exception:
            return 5
    df['n_genes'] = (
        df[ov].apply(parse_ov)
        if ov else 5)
    print(f'  {name}: {len(df)} paths  '
           f'top={df["Term_clean"].iloc[0]}')

# Top 8 per cohort for display
top_n = 8
ora_top = {
    name: df.nsmallest(top_n, q_col)
    for name, df in ora.items()}

# ═══════════════════════════════════════
# LOAD DE DATA ALL 4 COHORTS
# ═══════════════════════════════════════
print('\nLoading DE data...')

de_ukb_cd = pd.read_csv(
    f'{TABLES_DIR}/cd_limma_M3.csv')
if 'protein' not in de_ukb_cd.columns:
    de_ukb_cd.rename(
        columns={de_ukb_cd.columns[0]:
                  'protein'},
        inplace=True)

de_ukb_uc = pd.read_csv(
    f'{TABLES_DIR}/uc_limma_M3.csv')
if 'protein' not in de_ukb_uc.columns:
    de_ukb_uc.rename(
        columns={de_ukb_uc.columns[0]:
                  'protein'},
        inplace=True)

de_ibd_cd = pd.read_csv(
    f'{TABLES_DIR}/ibdome_cd_de_v2.csv')
de_ibd_uc = pd.read_csv(
    f'{TABLES_DIR}/ibdome_uc_de_v2.csv')

# Standardise column names
# UKB: negative logFC = up hyper
# IBDome: positive logFC = up hyper
de_datasets = {
    'UKB CD': {
        'df'     : de_ukb_cd,
        'q_col'  : 'adj.P.Val',
        'fc_col' : 'logFC',
        'hyper_dir': 'neg',
        'n_sig'  : 615,
        'n_up_h' : 612,
        'n_up_q' : 3,
        'top_prot': ['DBI','NUDT5',
                      'HSPA1A',
                      'TRIAP1',
                      'SERPINB6'],
        'col'    : NAVY,
        'title'  : 'UK Biobank — CD',
        'tag'    : 'Discovery · UK · '
                   'n=215 · 991 proteins',
    },
    'UKB UC': {
        'df'     : de_ukb_uc,
        'q_col'  : 'adj.P.Val',
        'fc_col' : 'logFC',
        'hyper_dir': 'neg',
        'n_sig'  : 647,
        'n_up_h' : 633,
        'n_up_q' : 14,
        'top_prot': ['TBCB','SNAP23',
                      'MANF','CRKL',
                      'PPIB'],
        'col'    : ORANGE,
        'title'  : 'UK Biobank — UC',
        'tag'    : 'Discovery · UK · '
                   'n=430 · 991 proteins',
    },
    'IBDome CD': {
        'df'     : de_ibd_cd,
        'q_col'  : 'adj_p_value',
        'fc_col' : 'logFC',
        'hyper_dir': 'pos',
        'n_sig'  : 58,
        'n_up_h' : 58,
        'n_up_q' : 0,
        'top_prot': ['HGF','CXCL11',
                      'TNFRSF9',
                      'IL18','SLAMF1'],
        'col'    : PURPLE,
        'title'  : 'IBDome — CD',
        'tag'    : 'Validation · Germany · '
                   'n=201 · 61 proteins',
    },
    'IBDome UC': {
        'df'     : de_ibd_uc,
        'q_col'  : 'adj_p_value',
        'fc_col' : 'logFC',
        'hyper_dir': 'pos',
        'n_sig'  : 53,
        'n_up_h' : 53,
        'n_up_q' : 0,
        'top_prot': ['MMP-10','CD5',
                      'IL8','VEGFA',
                      'MCP-3'],
        'col'    : TEAL,
        'title'  : 'IBDome — UC',
        'tag'    : 'Validation · Germany · '
                   'n=132 · 61 proteins',
    },
}

# ═══════════════════════════════════════
# FIGURE — 2 rows
# Row 0: Panel A — ORA dot plot
# Row 1: Panel B — 2×2 volcano grid
# ═══════════════════════════════════════
fig = plt.figure(figsize=(22, 28))
fig.patch.set_facecolor(WHITE)

gs_main = gridspec.GridSpec(
    2, 1,
    figure=fig,
    height_ratios=[1.1, 1.2],
    hspace=0.38,
    left=0.05,
    right=0.97,
    top=0.95,
    bottom=0.04)

# ══════════════════════════════════════
# PANEL A — ORA dot plot
# ══════════════════════════════════════
ax_a = fig.add_subplot(gs_main[0])
ax_a.set_facecolor(WHITE)
ax_a.spines['top'].set_visible(False)
ax_a.spines['bottom'].set_visible(False)
ax_a.spines['right'].set_visible(False)
ax_a.spines['left'].set_visible(False)
ax_a.grid(
    axis='x', alpha=0.10,
    linestyle='--', lw=0.8, zorder=0)
ax_a.tick_params(length=0)

cohort_order = [
    ('UKB CD',    NAVY,   -0.45),
    ('UKB UC',    ORANGE, -0.15),
    ('IBDome CD', PURPLE,  0.15),
    ('IBDome UC', TEAL,    0.45),
]

# Collect all unique top pathways
all_terms = []
for name, df in ora_top.items():
    for t in df['Term_clean'].tolist():
        if t not in all_terms:
            all_terms.append(t)

# Reorder: put most significant first
term_scores = {}
for t in all_terms:
    best = 0
    for name, df in ora_top.items():
        m = df[df['Term_clean']==t]
        if len(m)>0:
            best = max(
                best,
                m['log_fdr'].values[0])
    term_scores[t] = best
all_terms = sorted(
    all_terms,
    key=lambda t: term_scores[t],
    reverse=False)

n_terms = len(all_terms)
y_spacing = 1.8

for ti, term in enumerate(all_terms):
    y_base = ti * y_spacing
    # Horizontal guide line
    ax_a.axhline(
        y_base,
        color='#F0F0F0',
        lw=0.8, zorder=0)

    for name, col, y_off \
            in cohort_order:
        df  = ora[name]
        m   = df[df['Term_clean']
                  == term]
        if len(m) == 0:
            m = df[
                df['Term_clean'].str
                .lower().str.contains(
                    term.lower()[:12]
                )]
        if len(m) > 0:
            row    = m.iloc[0]
            logfdr = row['log_fdr']
            ng     = row['n_genes']
            size   = 60 + min(
                ng * 20, 350)
            ax_a.scatter(
                logfdr,
                y_base + y_off,
                s=size,
                c=col,
                alpha=0.90,
                edgecolors=WHITE,
                linewidths=0.8,
                zorder=4)
            if logfdr > 2:
                ax_a.text(
                    logfdr + 0.5,
                    y_base + y_off,
                    f'{logfdr:.1f}',
                    va='center',
                    ha='left',
                    fontsize=8.5,
                    color=col,
                    fontweight='700',
                    zorder=5)
        else:
            ax_a.scatter(
                0.5,
                y_base + y_off,
                s=50,
                c='#CCCCCC',
                marker='x',
                linewidths=1.5,
                zorder=3)

# Y axis — pathway names
ax_a.set_yticks(
    [i*y_spacing
     for i in range(n_terms)])
ax_a.set_yticklabels(
    all_terms,
    fontsize=12.5,
    fontweight='600',
    color='#1A1A2E')

# FDR line
ax_a.axvline(
    -np.log10(0.05),
    color='#999',
    lw=1.5, linestyle='--',
    alpha=0.7, zorder=2)
ax_a.text(
    -np.log10(0.05)+0.3,
    -0.8,
    'FDR = 0.05',
    fontsize=9, color='#999',
    va='top')

ax_a.set_xlim(-1.5, 55)
ax_a.set_ylim(
    -y_spacing,
    (n_terms-1)*y_spacing + y_spacing)
ax_a.set_xlabel(
    '-log\u2081\u2080(FDR q-value)  '
    '\u2014  dot size proportional '
    'to number of proteins enriched',
    fontsize=12, labelpad=8)

# Legend
leg_h = [
    Line2D([0],[0],
            marker='o', color='w',
            markerfacecolor=col,
            markersize=12,
            label=f'{name}')
    for name, col, _ in cohort_order]
leg_h += [
    Line2D([0],[0],
            marker='x',
            color='#CCCCCC',
            markersize=10,
            linewidth=0,
            markeredgewidth=2.0,
            label='Not significant'),
]
ax_a.legend(
    handles=leg_h,
    fontsize=11,
    framealpha=0.95,
    loc='lower right',
    edgecolor='#DDD',
    ncol=3)

ax_a.set_title(
    '(A)  MSigDB Hallmark Pathway '
    'Enrichment — Hyperinflammatory '
    'Subtype Across All Four Cohorts\n'
    'Over-representation analysis  ·  '
    'Fisher exact test  ·  '
    'Benjamini-Hochberg FDR < 0.05  ·  '
    'Shared core inflammatory cascade '
    'confirmed in UK and Germany',
    fontsize=13, fontweight='900',
    color=NAVY, loc='left', pad=10)

# ══════════════════════════════════════
# PANEL B — 2×2 volcano grid
# ══════════════════════════════════════
gs_vol = gridspec.GridSpecFromSubplotSpec(
    2, 2,
    subplot_spec=gs_main[1],
    hspace=0.42,
    wspace=0.32)

vol_order = [
    ('UKB CD',    0, 0),
    ('UKB UC',    0, 1),
    ('IBDome CD', 1, 0),
    ('IBDome UC', 1, 1),
]

panel_labels = ['(B)', '(C)',
                 '(D)', '(E)']

for pi, (name, row, col_idx) \
        in enumerate(vol_order):
    ax_v = fig.add_subplot(
        gs_vol[row, col_idx])
    d    = de_datasets[name]
    df   = d['df'].copy()
    qc   = d['q_col']
    fc   = d['fc_col']
    hdir = d['hyper_dir']
    col  = d['col']

    df['log_q'] = -np.log10(
        df[qc].clip(1e-50))
    sig_mask  = df[qc] < 0.05

    if hdir == 'neg':
        up_h = sig_mask & (df[fc] < 0)
        up_q = sig_mask & (df[fc] > 0)
    else:
        up_h = sig_mask & (df[fc] > 0)
        up_q = sig_mask & (df[fc] < 0)
    ns = ~sig_mask

    ax_v.set_facecolor(BG)
    ax_v.spines['top'].set_visible(False)
    ax_v.spines['right'].set_visible(False)
    ax_v.spines['left'].set_alpha(0.15)
    ax_v.spines['bottom'].set_alpha(0.15)

    # Not significant
    ax_v.scatter(
        df.loc[ns, fc],
        df.loc[ns, 'log_q'],
        c='#CCCCCC', s=6,
        alpha=0.25, linewidths=0,
        zorder=1, rasterized=True)

    # Up in hyperinflam
    ax_v.scatter(
        df.loc[up_h, fc],
        df.loc[up_h, 'log_q'],
        c=RED, s=10,
        alpha=0.78, linewidths=0,
        zorder=3, rasterized=True,
        label=f'Hyperinflam  '
              f'(n\u2009=\u2009'
              f'{up_h.sum()})')

    # Up in quiescent
    if up_q.sum() > 0:
        ax_v.scatter(
            df.loc[up_q, fc],
            df.loc[up_q, 'log_q'],
            c=GREEN, s=10,
            alpha=0.78, linewidths=0,
            zorder=3, rasterized=True,
            label=f'Quiescent  '
                  f'(n\u2009=\u2009'
                  f'{up_q.sum()})')

    # Annotate top 5 proteins
    top5 = df[up_h].nsmallest(
        5, qc)
    for _, r in top5.iterrows():
        ax_v.annotate(
            r['protein'],
            (r[fc], r['log_q']),
            fontsize=8,
            style='italic',
            fontweight='600',
            color=RED,
            xytext=(5, 4),
            textcoords='offset points',
            arrowprops=dict(
                arrowstyle='-',
                color=RED,
                alpha=0.30, lw=0.7))

    # FDR line
    ax_v.axhline(
        -np.log10(0.05),
        color='#666', lw=1.2,
        linestyle='--', alpha=0.6)
    ax_v.axvline(
        0, color='#AAA',
        lw=0.8, alpha=0.5)

    # Stats box
    ax_v.text(
        0.03, 0.97,
        f'Tested: {len(df)}\n'
        f'Sig: {d["n_sig"]}\n'
        f'Up H: {d["n_up_h"]}\n'
        f'Up Q: {d["n_up_q"]}',
        transform=ax_v.transAxes,
        ha='left', va='top',
        fontsize=9,
        fontfamily='monospace',
        color='#1A1A2E',
        bbox=dict(
            boxstyle='round,pad=0.35',
            fc=WHITE, ec='#CCC',
            alpha=0.95, lw=0.8))

    ax_v.legend(
        fontsize=9,
        framealpha=0.95,
        loc='upper left',
        markerscale=1.8,
        edgecolor='#DDD')
    ax_v.tick_params(
        labelsize=8.5, length=2)
    ax_v.set_xlabel(
        'log\u2082 Fold Change',
        fontsize=10, labelpad=4)
    ax_v.set_ylabel(
        '-log\u2081\u2080(FDR)',
        fontsize=10, labelpad=4)
    ax_v.set_title(
        f'{panel_labels[pi]}  '
        f'{d["title"]}\n'
        f'{d["tag"]}',
        fontsize=11.5,
        fontweight='900',
        color=col,
        loc='left', pad=7)

# ── Supertitle ────────────────────────
fig.suptitle(
    'Figure 4 — Pathway Enrichment '
    'and Differential Abundance '
    'Across All Four Cohorts\n'
    'Panel A: Hallmark ORA dot plot  ·  '
    'Panels B\u2013E: Volcano plots  ·  '
    'Consistent hyperinflammatory '
    'signature confirmed in UK '
    'and Germany',
    fontsize=13,
    fontweight='900',
    color=NAVY,
    y=0.99)

fname = (f'{FIGURES_DIR}/'
          f'figure4_final.png')
plt.savefig(
    fname, dpi=180,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure 4 saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname,
               width=1400))

In [ ]:
import os
import pandas as pd

TABLES_DIR = '/rds/homes/j/jxt554/tables'

print('PPI files found:')
for f in sorted(os.listdir(TABLES_DIR)):
    if 'ppi' in f.lower() \
    or 'string' in f.lower() \
    or 'network' in f.lower():
        print(f'\n  {f}:')
        # Try different encodings
        for enc in ['latin1',
                     'cp1252',
                     'iso-8859-1']:
            try:
                df = pd.read_csv(
                    f'{TABLES_DIR}/{f}',
                    nrows=3,
                    encoding=enc)
                print(f'    encoding={enc} OK')
                print(f'    cols: '
                       f'{df.columns.tolist()}')
                print(f'    shape: '
                       f'{df.shape}')
                break
            except Exception as e:
                print(f'    {enc}: {e}')

In [ ]:
import pandas as pd
import os

TABLES_DIR = '/rds/homes/j/jxt554/tables'

ppi_files = [
    'ppi_CD_C1_top50.csv',
    'ppi_UC_C1_top50.csv',
    'ibdome_cd_ppi_C2_top30.csv',
    'ibdome_uc_ppi_C2_top30.csv',
]

for f in ppi_files:
    df = pd.read_csv(
        f'{TABLES_DIR}/{f}',
        encoding='latin1')
    print(f'\n{f}:')
    print(f'  Interactions : {len(df)}')
    print(f'  Score range  : '
           f'{df["score"].min():.3f} '
           f'– {df["score"].max():.3f}')
    # Unique proteins
    prots = set(
        df['preferredName_A'].tolist() +
        df['preferredName_B'].tolist())
    print(f'  Unique nodes : {len(prots)}')
    # Top hubs by degree
    from collections import Counter
    all_names = (
        df['preferredName_A'].tolist() +
        df['preferredName_B'].tolist())
    top5 = Counter(
        all_names).most_common(5)
    print(f'  Top hubs:')
    for p, d in top5:
        print(f'    {p}: degree={d}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = '/rds/homes/j/jxt554/figures'
TABLES_DIR  = '/rds/homes/j/jxt554/tables'
SEED        = 42
np.random.seed(SEED)

NAVY   = '#1B3A6B'
RED    = '#E24B4A'
GREEN  = '#2E7D5B'
ORANGE = '#C96A1F'
PURPLE = '#6C3483'
TEAL   = '#117A65'
WHITE  = '#FFFFFF'
BG     = '#F8FAFC'

# ═══════════════════════════════════════
# LOAD
# ═══════════════════════════════════════
print('Loading PPI data...')

configs = [
    {
        'name'      : 'UK Biobank — '
                      'Ulcerative Colitis',
        'file'      : 'ppi_UC_C1_top50.csv',
        'panel'     : '(A)',
        'tag'       : 'Discovery · UK · '
                      'n = 430 · 991 proteins',
        'accent'    : ORANGE,
        'hub_col'   : '#922B21',
        'key_nodes' : {'SRC','FYB1',
                        'CRKL','STAT5B'},
        'hub_note'  : 'SRC kinase\n'
                      'degree = 17\n'
                      'BC = 0.605\n'
                      '→ drug target',
    },
    {
        'name'      : 'IBDome — '
                      'Crohn\'s Disease',
        'file'      : 'ibdome_cd_ppi_C2_top30.csv',
        'panel'     : '(B)',
        'tag'       : 'Validation · Germany · '
                      'n = 201 · 61 proteins',
        'accent'    : PURPLE,
        'hub_col'   : '#4A235A',
        'key_nodes' : {'IL6','IL10',
                        'CXCL10','CXCL8',
                        'CCL2','HGF'},
        'hub_note'  : 'IL-6 / IL-10\n'
                      'degree = 26\n'
                      'Dual cytokine hubs\n'
                      '→ JAK inhibitor target',
    },
    {
        'name'      : 'IBDome — '
                      'Ulcerative Colitis',
        'file'      : 'ibdome_uc_ppi_C2_top30.csv',
        'panel'     : '(C)',
        'tag'       : 'Validation · Germany · '
                      'n = 132 · 61 proteins',
        'accent'    : TEAL,
        'hub_col'   : '#0E6655',
        'key_nodes' : {'IL6','IFNG',
                        'IL10','CXCL8',
                        'CXCL5','CCL2'},
        'hub_note'  : 'IL-6 / IFN-γ\n'
                      'degree = 27 / 25\n'
                      'Shared with IBDome CD\n'
                      '→ consistent biology',
    },
]

nets = []
for cfg in configs:
    df = pd.read_csv(
        f'{TABLES_DIR}/{cfg["file"]}',
        encoding='latin1')
    df = df[df['score'] >= 0.40]
    G  = nx.Graph()
    for _, row in df.iterrows():
        a = row['preferredName_A']
        b = row['preferredName_B']
        w = float(row['score'])
        if a and b and a != b:
            G.add_edge(a, b, weight=w)
    deg = dict(G.degree())
    bc  = nx.betweenness_centrality(G)
    cfg['G']   = G
    cfg['deg'] = deg
    cfg['bc']  = bc
    nets.append(cfg)
    print(f'  {cfg["name"]}: '
           f'{G.number_of_nodes()} nodes '
           f'{G.number_of_edges()} edges')

# ═══════════════════════════════════════
# FIGURE — 1 row × 3 cols
# Each panel large and uncluttered
# ═══════════════════════════════════════
fig = plt.figure(figsize=(26, 12))
fig.patch.set_facecolor(WHITE)

gs = gridspec.GridSpec(
    1, 3,
    figure=fig,
    wspace=0.08,
    left=0.02,
    right=0.98,
    top=0.84,
    bottom=0.14)

for ci, cfg in enumerate(nets):
    ax  = fig.add_subplot(gs[0, ci])
    G   = cfg['G']
    deg = cfg['deg']
    bc  = cfg['bc']

    ax.set_facecolor(BG)
    ax.axis('off')

    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, 'No data',
                 ha='center', va='center',
                 transform=ax.transAxes)
        continue

    # ── Layout ────────────────────────
    # Use kamada-kawai for cleaner
    # spacing when possible
    try:
        pos = nx.kamada_kawai_layout(G)
    except Exception:
        pos = nx.spring_layout(
            G,
            k=3.0/np.sqrt(
                G.number_of_nodes()),
            iterations=120,
            seed=SEED)

    # ── Node metrics ──────────────────
    max_deg = max(deg.values())
    max_bc  = max(bc.values()) \
        if bc else 1.0

    # Sizes: 300 to 2500
    sizes = {
        n: 300 + (deg[n]/max_deg)*2200
        for n in G.nodes()}

    # Colours by BC
    node_cols = {}
    for n in G.nodes():
        bc_n = bc.get(n, 0)/max_bc
        if n in cfg['key_nodes']:
            node_cols[n] = cfg['hub_col']
        elif bc_n > 0.15:
            node_cols[n] = cfg['accent']
        else:
            node_cols[n] = '#AAB7B8'

    # ── Edges ─────────────────────────
    edges = list(G.edges(data=True))
    if edges:
        max_w = max(
            d.get('weight', 0.4)
            for _, _, d in edges)
        edge_ws = [
            0.4 +
            (d.get('weight',0.4)/max_w)*3.5
            for _, _, d in edges]
        edge_cs = [
            '#BDC3C7'
            if d.get('weight',0.4) < 0.7
            else '#7F8C8D'
            for _, _, d in edges]
    else:
        edge_ws = []
        edge_cs = []

    # ── Draw edges ────────────────────
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edge_color=edge_cs,
        width=edge_ws,
        alpha=0.50)

    # ── Draw nodes ────────────────────
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=[sizes[n]
                    for n in G.nodes()],
        node_color=[node_cols[n]
                     for n in G.nodes()],
        edgecolors=WHITE,
        linewidths=2.0,
        alpha=0.92)

    # ── Labels — key nodes only ───────
    # Always label top 10 by degree
    top10 = {
        n for n, _ in sorted(
            deg.items(),
            key=lambda x: x[1],
            reverse=True)[:10]}
    label_nodes = (
        top10 | cfg['key_nodes'])

    for n in G.nodes():
        if n not in label_nodes:
            continue
        x, y   = pos[n]
        is_hub = n in cfg['key_nodes']
        fs     = 11 if is_hub else 9
        fw     = '900' if is_hub else '700'
        # Text colour: white on dark,
        # dark on light
        bc_n = bc.get(n, 0)/max_bc
        if n in cfg['key_nodes'] or \
                bc_n > 0.15:
            tc = WHITE
        else:
            tc = '#1A1A2E'
        ax.text(
            x, y, n,
            ha='center', va='center',
            fontsize=fs,
            fontweight=fw,
            color=tc,
            style='italic',
            zorder=7,
            bbox=dict(
                boxstyle='round,pad=0.15',
                fc='none',
                ec='none'))

    # ── Hub annotation box ────────────
    ax.text(
        0.02, 0.02,
        f'Nodes: '
        f'{G.number_of_nodes()}\n'
        f'Edges: '
        f'{G.number_of_edges()}\n\n'
        f'{cfg["hub_note"]}',
        transform=ax.transAxes,
        ha='left', va='bottom',
        fontsize=10,
        fontfamily='monospace',
        color='#1A1A2E',
        bbox=dict(
            boxstyle='round,pad=0.50',
            fc=WHITE,
            ec=cfg['accent'],
            alpha=0.97,
            lw=1.5),
        zorder=8)

    # ── Panel label top left ──────────
    ax.text(
        0.01, 0.99,
        cfg['panel'],
        transform=ax.transAxes,
        ha='left', va='top',
        fontsize=22,
        fontweight='900',
        color=NAVY, zorder=9)

    # ── Title below panel label ───────
    ax.text(
        0.5, 1.06,
        cfg['name'],
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=13,
        fontweight='900',
        color=cfg['accent'])
    ax.text(
        0.5, 1.01,
        cfg['tag'],
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=10,
        color='#555',
        style='italic')

# ── Shared legend ─────────────────────
legend_h = [
    mpatches.Patch(
        color='#922B21',
        label='Key hub protein  '
              '(high BC + degree)'),
    mpatches.Patch(
        color='#7F8C8D',
        label='Intermediate node  '
              '(BC > 0.15)'),
    mpatches.Patch(
        color='#AAB7B8',
        label='Peripheral node'),
    Line2D([0],[0],
            color='#7F8C8D',
            lw=3.0,
            label='High confidence  '
                  '(score \u2265 0.70)'),
    Line2D([0],[0],
            color='#BDC3C7',
            lw=1.2,
            label='Medium confidence  '
                  '(score 0.40\u20130.70)'),
]
fig.legend(
    handles=legend_h,
    fontsize=11,
    loc='lower center',
    ncol=5,
    bbox_to_anchor=(0.5, 0.01),
    framealpha=0.95,
    edgecolor='#DDD',
    handlelength=1.8)

# ── Supertitle ────────────────────────
fig.text(
    0.5, 0.97,
    'Figure 5 — Protein-Protein '
    'Interaction Networks of the '
    'Hyperinflammatory Subtype',
    ha='center', va='top',
    fontsize=15,
    fontweight='900',
    color=NAVY)
fig.text(
    0.5, 0.93,
    'STRING database  ·  '
    'Confidence score \u2265 0.40  ·  '
    'Node size = degree centrality  ·  '
    'Node colour = betweenness '
    'centrality  ·  '
    'Key hubs = candidate '
    'therapeutic targets',
    ha='center', va='top',
    fontsize=11,
    color='#555',
    style='italic')

fname = (f'{FIGURES_DIR}/'
          f'figure5_ppi_clean.png')
plt.savefig(
    fname, dpi=180,
    bbox_inches='tight',
    facecolor=WHITE)
plt.close()
print('Figure 5 saved')

from IPython.display import (
    Image, display)
display(Image(filename=fname,
               width=1400))

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import mannwhitneyu

DATA_DIR = '/rds/homes/j/jxt554/data'

# ═══════════════════════════════════════
# UKB BIOBANK
# ═══════════════════════════════════════
print('UK BIOBANK CLINICAL TABLE')
print('='*60)

meta_cd = pd.read_csv(
    f'{DATA_DIR}/meta_cd_final.csv')
meta_uc = pd.read_csv(
    f'{DATA_DIR}/meta_uc_final.csv')

print(f'CD n={len(meta_cd)}')
print(f'UC n={len(meta_uc)}')
print(f'\nCD columns:')
print(meta_cd.columns.tolist())
print(f'\nUC columns:')
print(meta_uc.columns.tolist())

# Add disease label
meta_cd['disease'] = 'CD'
meta_uc['disease'] = 'UC'

# Find common columns
common = [
    c for c in meta_cd.columns
    if c in meta_uc.columns]
print(f'\nCommon columns: {common}')

# Combine
ukb = pd.concat(
    [meta_cd, meta_uc],
    ignore_index=True)

print(f'\nCombined: {ukb.shape}')
print(f'\nValue counts per column:')
for col in ukb.columns:
    if col == 'disease':
        continue
    n_miss   = ukb[col].isna().sum()
    miss_pct = n_miss/len(ukb)*100
    n_unique = ukb[col].nunique()
    if n_unique < 12:
        print(f'\n  {col} '
               f'(missing={miss_pct:.1f}%):')
        print(f'    CD: {meta_cd[col].value_counts().to_dict() if col in meta_cd.columns else "N/A"}')
        print(f'    UC: {meta_uc[col].value_counts().to_dict() if col in meta_uc.columns else "N/A"}')
    else:
        if col in meta_cd.columns and \
           col in meta_uc.columns:
            cd_vals = pd.to_numeric(
                meta_cd[col],
                errors='coerce').dropna()
            uc_vals = pd.to_numeric(
                meta_uc[col],
                errors='coerce').dropna()
            if len(cd_vals) > 5:
                print(f'\n  {col} '
                       f'(missing={miss_pct:.1f}%):')
                print(f'    CD: '
                       f'{cd_vals.mean():.2f}'
                       f' ± {cd_vals.std():.2f}')
                print(f'    UC: '
                       f'{uc_vals.mean():.2f}'
                       f' ± {uc_vals.std():.2f}')
                _, p = mannwhitneyu(
                    cd_vals, uc_vals,
                    alternative='two-sided')
                print(f'    p={p:.4f}')

# ═══════════════════════════════════════
# IBDOME
# ═══════════════════════════════════════
print('\n\nIBDOME CLINICAL TABLE')
print('='*60)

# Load IBDome metadata
raw_meta = pd.read_csv(
    '/rds/homes/j/jxt554/'
    'IBDome_olink_proteomics'
    '_metadata_v1.0.3.csv')

# Filter CD and UC baseline
raw_meta['date'] = pd.to_datetime(
    raw_meta['date'], errors='coerce')

ibd_cd = raw_meta[
    raw_meta['disease'].str.contains(
        'Crohn', na=False)
].sort_values('date').groupby(
    'subject_id').first().reset_index()

ibd_uc = raw_meta[
    raw_meta['disease'].str.contains(
        'Ulcerative', na=False)
].sort_values('date').groupby(
    'subject_id').first().reset_index()

print(f'IBDome CD n={len(ibd_cd)}')
print(f'IBDome UC n={len(ibd_uc)}')
print(f'\nColumns:')
print(raw_meta.columns.tolist())

print(f'\nValue counts per column:')
for col in raw_meta.columns:
    if col in ['sample_id','subject_id',
                'date','sample_type',
                'sampling_procedure',
                'dataset','disease']:
        continue
    n_miss_cd = ibd_cd[col].isna(
        ).sum() if col in ibd_cd.columns \
        else 0
    n_miss_uc = ibd_uc[col].isna(
        ).sum() if col in ibd_uc.columns \
        else 0
    n_total   = len(ibd_cd)+len(ibd_uc)
    miss_pct  = (n_miss_cd+n_miss_uc
                  )/n_total*100
    n_unique  = raw_meta[col].nunique()

    if n_unique < 12:
        print(f'\n  {col} '
               f'(missing={miss_pct:.1f}%):')
        if col in ibd_cd.columns:
            print(f'    CD: '
                   f'{ibd_cd[col].value_counts().to_dict()}')
        if col in ibd_uc.columns:
            print(f'    UC: '
                   f'{ibd_uc[col].value_counts().to_dict()}')
        # Test
        if col in ibd_cd.columns and \
           col in ibd_uc.columns:
            try:
                ct = pd.crosstab(
                    pd.concat([
                        ibd_cd[col].map(
                            lambda x:
                            'CD'),
                        ibd_uc[col].map(
                            lambda x:
                            'UC')]),
                    pd.concat([
                        ibd_cd[col],
                        ibd_uc[col]]))
                _, p, _, _ = \
                    stats.chi2_contingency(ct)
                print(f'    p={p:.4f}  '
                       f'Chi-squared')
            except Exception:
                pass
    else:
        if col in ibd_cd.columns and \
           col in ibd_uc.columns:
            cd_v = pd.to_numeric(
                ibd_cd[col],
                errors='coerce').dropna()
            uc_v = pd.to_numeric(
                ibd_uc[col],
                errors='coerce').dropna()
            if len(cd_v) > 5:
                print(f'\n  {col} '
                       f'(missing={miss_pct:.1f}%):')
                print(f'    CD: '
                       f'{cd_v.mean():.2f}'
                       f' ± {cd_v.std():.2f}')
                print(f'    UC: '
                       f'{uc_v.mean():.2f}'
                       f' ± {uc_v.std():.2f}')
                _, p = mannwhitneyu(
                    cd_v, uc_v,
                    alternative='two-sided')
                print(f'    p={p:.4f}  '
                       f'Mann-Whitney U')